In [ ]:
# ============================================================
# BART MULTIMODAL MASTER PIPELINE (Pump-level) — ~55+ columns
# Input: *_beh.xlsx + matching *.xdf/*.xdfz (LabRecorder)
# Output: outputs/BART_master_pump_level.{csv,xlsx}
#
# Key fixes:
#  - FIXED indentation + scoping issues in beh_to_pump_table()
#  - Post baseline extraction works (posttask_EOB/ECB/CB avgTheta)
#  - Pump theta + trial theta no longer blank due to clean merges
#  - Missing XDF allowed (BEH-only rows kept)
#  - Robust marker stream selection
#  - Alignment uses marker_stream["time_stamps"] (lsltime)
#  - PGSI score merged from manifest.xlsx
#  - Survey responses merged from session 5 BEH survey sheet
#  - Survey columns placed beside pID
# ============================================================

!pip -q install pyxdf mne openpyxl pandas numpy

import re, json
from pathlib import Path
import numpy as np
import pandas as pd

import pyxdf
import mne

# ----------------------------
# CONFIG
# ----------------------------
ROOT = Path("/content") if Path("/content").exists() else Path(".")
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)

TARGET_SFREQ = 512.0

# ERP epoching (explode)
ERP_TMIN, ERP_TMAX = -0.2, 0.8
ERP_BASELINE = (ERP_TMIN, 0.0)
P3_WIN = (0.300, 0.450)  # seconds

# Theta windows
WIN200 = 0.200
THETA_BAND = (4.0, 7.0)

# Your ActiCAP channel order (1..16)
EEG_ORDER = ["Fz","Cz","Pz","POz","Fp1","Fp2","F3","F4","FCz","C3","C4","CPz","P3","P4","O1","O2"]

FRONTAL_THETA_CHS = ["Fp1","Fp2","Fz","F3","F4","FCz"]
P300_CHS = ["Fz","FCz","Cz","Pz"]

# ----------------------------
# Optional manifest (PGSI etc.) - robust loader
# ----------------------------
manifest_path = next(iter(ROOT.rglob("manifest.xlsx")), None)
manifest = None

def normalize_pid_series(s):
    return (
        s.astype(str)
         .str.replace("sub-", "", regex=False)
         .str.replace("SUB-", "", regex=False)
         .str.strip()
         .str.upper()
    )

if manifest_path is not None and Path(manifest_path).exists():
    xls = pd.ExcelFile(manifest_path)

    manifest_candidates = []
    for sh in xls.sheet_names:
        df = pd.read_excel(manifest_path, sheet_name=sh)
        df.columns = [str(c).strip() for c in df.columns]

        cols_l = {str(c).strip().lower(): c for c in df.columns}

        pid_col = None
        pgsi_col = None
        cond_col = None
        risk_col = None

        for k in ["pid", "p_id", "participant_id", "participant", "subject", "sub", "id"]:
            if k in cols_l:
                pid_col = cols_l[k]
                break

        for k in ["pgsi score", "pgsi", "pgsi total", "pgsi_total", "pgsi score"]:
            if k in cols_l:
                pgsi_col = cols_l[k]
                break

        for k in ["cond", "condition"]:
            if k in cols_l:
                cond_col = cols_l[k]
                break

        for k in ["risk_group", "risk", "group"]:
            if k in cols_l:
                risk_col = cols_l[k]
                break

        if pid_col is not None:
            keep = [pid_col]
            rename_map = {pid_col: "pID"}

            if pgsi_col is not None:
                keep.append(pgsi_col)
                rename_map[pgsi_col] = "pgsi_score"
            if cond_col is not None:
                keep.append(cond_col)
                rename_map[cond_col] = "cond_manifest"
            if risk_col is not None:
                keep.append(risk_col)
                rename_map[risk_col] = "risk_group"

            tmp = df[keep].copy().rename(columns=rename_map)
            tmp["pID"] = normalize_pid_series(tmp["pID"])
            if "pgsi_score" in tmp.columns:
                tmp["pgsi_score"] = pd.to_numeric(tmp["pgsi_score"], errors="coerce")

            manifest_candidates.append((sh, tmp))

    # choose the best sheet = one that has both pID and pgsi_score, otherwise first with pID
    chosen = None
    for sh, df in manifest_candidates:
        if "pgsi_score" in df.columns:
            chosen = (sh, df)
            break
    if chosen is None and len(manifest_candidates):
        chosen = manifest_candidates[0]

    if chosen is not None:
        sh, manifest = chosen
        manifest = manifest.drop_duplicates(subset=["pID"], keep="first")
        print("Loaded manifest:", manifest_path)
        print("Using manifest sheet:", sh)
        print("Manifest columns:", manifest.columns.tolist())
        if "pgsi_score" in manifest.columns:
            print("Manifest non-missing PGSI rows:", int(manifest["pgsi_score"].notna().sum()))
            print("Manifest sample pID/PGSI:")
            print(manifest[["pID", "pgsi_score"]].dropna().head(10))
    else:
        print("Manifest found, but no usable participant ID column detected.")
else:
    print("No manifest.xlsx found.")
# ----------------------------
# Utilities
# ----------------------------
def parse_bids_bits(name: str):
    base = Path(name).name
    sub = re.search(r"sub-([A-Za-z0-9]+)", base)
    ses = re.search(r"ses-([A-Za-z0-9]+)", base)
    run = re.search(r"run-([0-9]+)", base)
    task = re.search(r"task-([A-Za-z0-9]+)", base)
    cond = re.search(r"cond-([A-Za-z0-9_]+)", base)

    out = {
        "sub": sub.group(1) if sub else None,
        "ses": ses.group(1) if ses else None,
        "run": run.group(1) if run else None,
        "task": task.group(1) if task else None,
        "condition": cond.group(1) if cond else None,
    }
    if out["ses"] is not None:
        m = re.search(r"(\d+)", out["ses"])
        out["ses_num"] = int(m.group(1)) if m else np.nan
    else:
        out["ses_num"] = np.nan
    return out

def read_beh_sheets(fp: Path):
    xls = pd.ExcelFile(fp)
    sheets = {}
    for sh in xls.sheet_names:
        df = pd.read_excel(fp, sheet_name=sh)
        df.columns = [str(c).strip() for c in df.columns]
        sheets[sh.lower()] = df
    return sheets

def get_sheet(sheets: dict, candidates):
    for c in candidates:
        if c in sheets:
            return sheets[c]
    return None

def safe_json_list(x):
    try:
        return json.loads(x) if isinstance(x, str) else (x if isinstance(x, list) else None)
    except Exception:
        return None

def extract_survey_row(survey_df, participant_id):
    """
    Convert a survey sheet into one flat row for one participant.
    Keeps all questions as columns.
    Handles:
    - single-row wide format
    - question/response long format
    - generic multi-row fallback
    """
    if survey_df is None or len(survey_df) == 0:
        return None

    df = survey_df.copy()
    df.columns = [str(c).strip() for c in df.columns]

    participant_id = str(participant_id).replace("sub-", "").strip().upper()

    # Case 1: one-row wide format
    if len(df) == 1:
        row = df.iloc[0].to_dict()
        row["pID"] = participant_id
        return row

    # Case 2: question/response long format
    cols_l = {str(c).strip().lower(): c for c in df.columns}
    q_col = None
    r_col = None

    for k in ["question", "item", "prompt"]:
        if k in cols_l:
            q_col = cols_l[k]
            break
    for k in ["response", "answer", "value", "score"]:
        if k in cols_l:
            r_col = cols_l[k]
            break

    if q_col is not None and r_col is not None:
        out = {"pID": participant_id}
        for _, r in df.iterrows():
            q = str(r[q_col]).strip()
            q_clean = re.sub(r"[^A-Za-z0-9]+", "_", q).strip("_").lower()
            out[f"survey_{q_clean}"] = r[r_col]
        return out

    # Case 3: generic multi-row fallback
    out = {"pID": participant_id}
    for i, (_, r) in enumerate(df.iterrows(), start=1):
        for c in df.columns:
            cname = re.sub(r"[^A-Za-z0-9]+", "_", str(c)).strip("_").lower()
            out[f"survey_r{i}_{cname}"] = r[c]
    return out

# ----------------------------
# XDF helpers
# ----------------------------
def parse_marker_message(msg):
    s = str(msg)
    parts = s.split(";")
    out = {"code": parts[0].strip()}
    for p in parts[1:]:
        if "=" in p:
            k, v = p.split("=", 1)
            out[k.strip()] = v.strip()
    return out

def _stream_name_lower(s):
    info = s.get("info", {})
    nm = info.get("name", "")
    if isinstance(nm, (list, tuple, np.ndarray)):
        nm = nm[0] if len(nm) else ""
    return str(nm).strip().lower()

def pick_streams(streams):
    """
    EEG: stream with largest channel count
    Markers: prefer name containing 'bart' or 'marker' (1 channel)
    """
    eeg = None
    best = -1

    for s in streams:
        ts = s.get("time_series", None)
        if ts is None:
            continue
        arr = np.asarray(ts)
        ch = 1 if arr.ndim == 1 else arr.shape[1]
        if ch > best:
            eeg = s
            best = ch

    markers = None
    for s in streams:
        ts = s.get("time_series", None)
        if ts is None:
            continue
        arr = np.asarray(ts)
        ch = 1 if arr.ndim == 1 else arr.shape[1]
        if ch != 1:
            continue
        nm = _stream_name_lower(s)
        if ("bart" in nm) or ("marker" in nm) or ("markers" in nm) or ("event" in nm):
            markers = s
            break

    if markers is None:
        for s in streams:
            ts = s.get("time_series", None)
            if ts is None:
                continue
            arr = np.asarray(ts)
            ch = 1 if arr.ndim == 1 else arr.shape[1]
            if ch == 1:
                markers = s
                break

    return eeg, markers

def markers_to_df(marker_stream):
    m_ts = np.asarray(marker_stream["time_stamps"], dtype=float)
    m_val = np.asarray(marker_stream["time_series"], dtype=object)

    rows = []
    for t, v in zip(m_ts, m_val):
        msg = v[0] if isinstance(v, (list, tuple, np.ndarray)) else v
        d = parse_marker_message(msg)
        d["lsltime"] = float(t)   # alignment reference
        rows.append(d)

    df = pd.DataFrame(rows)
    for k in ["trial", "pump", "loss", "earnings", "total"]:
        if k in df.columns:
            df[k] = pd.to_numeric(df[k], errors="coerce")
    return df

def build_raw_from_xdf_eeg(eeg_stream):
    data = np.asarray(eeg_stream["time_series"])
    t = np.asarray(eeg_stream["time_stamps"], dtype=float)
    if data.ndim == 1:
        data = data[:, None]
    n_samp, n_ch = data.shape

    sfreq_est = 1.0 / np.median(np.diff(t))
    ch_names = EEG_ORDER[:n_ch] if n_ch <= len(EEG_ORDER) else [f"EEG{ii+1:02d}" for ii in range(n_ch)]
    arr = data.T.astype(float)

    peak = float(np.nanmax(np.abs(arr)))
    if peak > 1e3:
        arr *= 1e-6
    elif peak > 5.0:
        arr *= 1e-3

    info = mne.create_info(ch_names=ch_names, sfreq=sfreq_est, ch_types=["eeg"] * n_ch)
    raw = mne.io.RawArray(arr, info, verbose=False)
    return raw, t

def preprocess_raw(raw):
    raw = raw.copy()
    raw.filter(0.1, 30.0, picks="eeg", verbose=False)
    try:
        raw.notch_filter([60], picks="eeg", verbose=False)
    except Exception:
        pass
    try:
        raw.set_eeg_reference("average", verbose=False)
    except Exception:
        pass
    return raw

def lsltime_to_sample(lsltime, eeg_t0, sfreq):
    return int(np.round((lsltime - eeg_t0) * sfreq))

# ----------------------------
# EEG feature extraction
# ----------------------------
def compute_theta_power_window(raw_theta, sample, tmin_s, tmax_s, picks):
    sf = raw_theta.info["sfreq"]
    start = int(sample + np.round(tmin_s * sf))
    stop  = int(sample + np.round(tmax_s * sf))
    start = max(start, 0)
    stop  = min(stop, raw_theta.n_times)
    if stop <= start + 2:
        return np.nan
    seg = raw_theta.get_data(picks=picks, start=start, stop=stop)
    return float(np.mean(seg**2))

def extract_session_theta_metrics(raw):
    rtheta = raw.copy().filter(THETA_BAND[0], THETA_BAND[1], picks="eeg", verbose=False)
    picks = [raw.ch_names.index(ch) for ch in FRONTAL_THETA_CHS if ch in raw.ch_names]
    if len(picks) == 0:
        return np.nan, np.nan
    X = rtheta.get_data(picks=picks)
    theta_ts = np.mean(X**2, axis=0)
    return float(np.mean(theta_ts)), float(np.std(theta_ts, ddof=1))

def extract_p300_metrics(raw, md, eeg_t0):
    keys = ["ses_avgP300_All","ses_avgP300_Fz","ses_avgP300_FCz","ses_avgP300_Cz","ses_avgP300_Pz"]
    if "code" not in md.columns:
        return {k: np.nan for k in keys}

    exp = md[md["code"].astype(str).str.upper().eq("BART_EXPLODE")].copy()
    if len(exp) == 0:
        return {k: np.nan for k in keys}

    sf = raw.info["sfreq"]
    exp = exp.dropna(subset=["lsltime"]).copy()
    exp["sample"] = exp["lsltime"].apply(lambda t: lsltime_to_sample(float(t), eeg_t0, sf)).astype(int)

    min_samp = int(np.ceil((-ERP_TMIN) * sf))
    exp = exp[(exp["sample"] >= min_samp) & (exp["sample"] < raw.n_times)].copy()
    if len(exp) == 0:
        return {k: np.nan for k in keys}

    events = np.c_[exp["sample"].to_numpy(), np.zeros(len(exp), int), np.ones(len(exp), int)]
    epochs = mne.Epochs(
        raw, events, event_id={"EXPLODE": 1},
        tmin=ERP_TMIN, tmax=ERP_TMAX, baseline=ERP_BASELINE,
        preload=True, reject_by_annotation=True, verbose=False
    )
    if len(epochs) == 0:
        return {k: np.nan for k in keys}

    ev = epochs.average()
    times = ev.times
    m1 = np.argmin(np.abs(times - P3_WIN[0]))
    m2 = np.argmin(np.abs(times - P3_WIN[1]))

    out = {}
    vals = []
    for ch in ["Fz","FCz","Cz","Pz"]:
        k = f"ses_avgP300_{ch}"
        if ch in ev.ch_names:
            idx = ev.ch_names.index(ch)
            erp_uV = ev.data[idx] * 1e6
            p3 = float(np.max(erp_uV[m1:m2]))
            out[k] = p3
            vals.append(p3)
        else:
            out[k] = np.nan
    out["ses_avgP300_All"] = float(np.nanmean(vals)) if len(vals) else np.nan
    return out

def extract_event_theta_windows(raw, md, eeg_t0):
    rtheta = raw.copy().filter(THETA_BAND[0], THETA_BAND[1], picks="eeg", verbose=False)
    sf = raw.info["sfreq"]
    picks = [raw.ch_names.index(ch) for ch in FRONTAL_THETA_CHS if ch in raw.ch_names]
    if len(picks) == 0:
        return {
            "pre200_pop_Theta": np.nan, "post200_pop_Theta": np.nan,
            "pre200_collect_Theta": np.nan, "post200_collect_Theta": np.nan,
            "pre200_pmp_Theta": np.nan,
        }

    def _mean_prepost(code, pre=True):
        d = md[md["code"].astype(str).str.upper().eq(code)].dropna(subset=["lsltime"]).copy()
        if len(d) == 0:
            return np.nan
        d["sample"] = d["lsltime"].apply(lambda t: lsltime_to_sample(float(t), eeg_t0, sf)).astype(int)
        vals = []
        for s in d["sample"].tolist():
            v = compute_theta_power_window(rtheta, s, -WIN200, 0.0, picks) if pre else compute_theta_power_window(rtheta, s, 0.0, WIN200, picks)
            vals.append(v)
        return float(np.nanmean(vals)) if len(vals) else np.nan

    return {
        "pre200_pop_Theta": _mean_prepost("BART_EXPLODE", pre=True),
        "post200_pop_Theta": _mean_prepost("BART_EXPLODE", pre=False),
        "pre200_collect_Theta": _mean_prepost("BART_COLLECT", pre=True),
        "post200_collect_Theta": _mean_prepost("BART_COLLECT", pre=False),
        "pre200_pmp_Theta": _mean_prepost("BART_PUMP", pre=True),
    }

def theta_before_each_pump(raw, md, eeg_t0):
    pumps = md[md["code"].astype(str).str.upper().eq("BART_PUMP")].copy()
    if len(pumps) == 0 or ("trial" not in pumps.columns) or ("pump" not in pumps.columns):
        return pd.DataFrame(columns=["trial","pump_number","theta_before_pump","theta_before_pump_Fp1","theta_before_pump_Fp2","frontal_asymmetry"])

    rtheta = raw.copy().filter(THETA_BAND[0], THETA_BAND[1], picks="eeg", verbose=False)
    sf = raw.info["sfreq"]

    pumps["trial"] = pd.to_numeric(pumps["trial"], errors="coerce")
    pumps["pump_number"] = pd.to_numeric(pumps["pump"], errors="coerce")
    pumps = pumps.dropna(subset=["trial","pump_number","lsltime"]).copy()

    pumps["trial"] = pumps["trial"].astype(int)
    pumps["pump_number"] = pumps["pump_number"].astype(int)
    pumps["sample"] = pumps["lsltime"].apply(lambda t: lsltime_to_sample(float(t), eeg_t0, sf)).astype(int)

    min_samp = int(np.ceil(WIN200 * sf))
    pumps = pumps[(pumps["sample"] >= min_samp) & (pumps["sample"] < raw.n_times)].copy()
    if len(pumps) == 0:
        return pd.DataFrame(columns=["trial","pump_number","theta_before_pump","theta_before_pump_Fp1","theta_before_pump_Fp2","frontal_asymmetry"])

    frontal_picks = [raw.ch_names.index(ch) for ch in FRONTAL_THETA_CHS if ch in raw.ch_names]
    fp1_pick = [raw.ch_names.index("Fp1")] if "Fp1" in raw.ch_names else []
    fp2_pick = [raw.ch_names.index("Fp2")] if "Fp2" in raw.ch_names else []

    rows = []
    for _, r in pumps.iterrows():
        s = int(r["sample"])
        pre_frontal = compute_theta_power_window(rtheta, s, -WIN200, 0.0, frontal_picks) if frontal_picks else np.nan
        pre_fp1 = compute_theta_power_window(rtheta, s, -WIN200, 0.0, fp1_pick) if fp1_pick else np.nan
        pre_fp2 = compute_theta_power_window(rtheta, s, -WIN200, 0.0, fp2_pick) if fp2_pick else np.nan
        fa = (np.log(pre_fp2 + 1e-12) - np.log(pre_fp1 + 1e-12)) if (np.isfinite(pre_fp1) and np.isfinite(pre_fp2)) else np.nan

        rows.append({
            "trial": int(r["trial"]),
            "pump_number": int(r["pump_number"]),
            "theta_before_pump": pre_frontal,
            "theta_before_pump_Fp1": pre_fp1,
            "theta_before_pump_Fp2": pre_fp2,
            "frontal_asymmetry": fa
        })

    return pd.DataFrame(rows)

def trial_theta_from_markers(raw, md, eeg_t0):
    sf = raw.info["sfreq"]
    rtheta = raw.copy().filter(THETA_BAND[0], THETA_BAND[1], picks="eeg", verbose=False)
    picks = [raw.ch_names.index(ch) for ch in FRONTAL_THETA_CHS if ch in raw.ch_names]
    if len(picks) == 0:
        return pd.DataFrame(columns=["trial","trial_mTheta","trial_SD_mThetaB"])

    starts = md[md["code"].astype(str).str.upper().eq("BART_TRIAL_START")].copy()
    ends = md[md["code"].astype(str).str.upper().isin(["BART_COLLECT","BART_EXPLODE"])].copy()
    if "trial" not in starts.columns or "trial" not in ends.columns:
        return pd.DataFrame(columns=["trial","trial_mTheta","trial_SD_mThetaB"])

    starts["trial"] = pd.to_numeric(starts["trial"], errors="coerce")
    ends["trial"] = pd.to_numeric(ends["trial"], errors="coerce")
    starts = starts.dropna(subset=["trial","lsltime"]).copy()
    ends = ends.dropna(subset=["trial","lsltime"]).copy()
    if len(starts) == 0 or len(ends) == 0:
        return pd.DataFrame(columns=["trial","trial_mTheta","trial_SD_mThetaB"])

    out = []
    for tnum, g0 in starts.groupby(starts["trial"].astype(int)):
        t_start = float(g0["lsltime"].min())
        g1 = ends[ends["trial"].astype(int) == int(tnum)]
        if len(g1) == 0:
            continue
        t_end = float(g1["lsltime"].min())
        if t_end <= t_start:
            continue

        s0 = lsltime_to_sample(t_start, eeg_t0, sf)
        s1 = lsltime_to_sample(t_end, eeg_t0, sf)
        s0 = max(s0, 0)
        s1 = min(s1, raw.n_times)
        if s1 <= s0 + int(sf * 0.2):
            continue

        seg = rtheta.get_data(picks=picks, start=s0, stop=s1)
        theta_ts = np.mean(seg**2, axis=0)

        out.append({
            "trial": int(tnum),
            "trial_mTheta": float(np.mean(theta_ts)),
            "trial_SD_mThetaB": float(np.std(theta_ts, ddof=1)),
        })

    return pd.DataFrame(out)

# ----------------------------
# BEH -> pump table
# ----------------------------
def beh_to_pump_table(beh_fp: Path):
    sheets = read_beh_sheets(beh_fp)
    trials = get_sheet(sheets, ["trials", "trial"])
    summary = get_sheet(sheets, ["summary"])

    if trials is None or len(trials) == 0:
        return None, None, None

    trials = trials.copy()
    bits = parse_bids_bits(beh_fp.name)

    # explode/collect flags
    if "exploded" in trials.columns:
        trials["pop"] = trials["exploded"].astype(str).str.upper().isin(["TRUE","1","YES"]).astype(int)
    else:
        trials["pop"] = 0

    if "collected" in trials.columns:
        trials["collect"] = pd.to_numeric(trials["collected"], errors="coerce").fillna(0).astype(int)
    else:
        trials["collect"] = trials["events"].astype(str).str.contains("collect", case=False).astype(int) if "events" in trials.columns else 0

    # numeric conversions
    for c in [
        "trial","pump_count","trial_earnings","loss_if_pop","trial_value","total_earnings",
        "pump_latency_mean","pump_latency_median","pump_latency_first",
        "collect_latency_from_ready","collect_latency_from_trial_start",
        "nf_high_frac","nf_green_success",
        "risk_after_explosion","risk_after_collect","pump_speed_change",
        "nf_mean_normalized"
    ]:
        if c in trials.columns:
            trials[c] = pd.to_numeric(trials[c], errors="coerce")

    # nf vars
    if "nf_high_frac" in trials.columns:
        trials["nf_high_fraction"] = pd.to_numeric(trials["nf_high_frac"], errors="coerce")
    elif "nf_high_fraction" not in trials.columns:
        trials["nf_high_fraction"] = np.nan

    if "nf_green_success" in trials.columns and trials["nf_green_success"].notna().any():
        trials["nf_green_success"] = pd.to_numeric(trials["nf_green_success"], errors="coerce")
    else:
        trials["nf_green_success"] = np.where(trials["nf_high_fraction"] > 0.75, 1, 0).astype(int) if "nf_high_fraction" in trials.columns else np.nan

    # bonus awarded: 3 greens in a row (within block)
    block_col = "block" if "block" in trials.columns else None
    if block_col is not None:
        trials = trials.sort_values([block_col, "trial"], kind="mergesort") if "trial" in trials.columns else trials
        g = trials.groupby(block_col)["nf_green_success"]
        trials["nf_bonus_awarded_trial"] = ((g.shift(0)==1) & (g.shift(1)==1) & (g.shift(2)==1)).astype(int)
    else:
        trials["nf_bonus_awarded_trial"] = 0

    # derived trial metrics
    if "pump_count" in trials.columns and block_col is not None and "trial" in trials.columns:
        trials["prev_pumps"] = trials.groupby(block_col)["pump_count"].shift(1)
        prev_pop = trials.groupby(block_col)["pop"].shift(1)
        prev_collect = trials.groupby(block_col)["collect"].shift(1)

        trials["risk_after_explosion"] = np.where(prev_pop == 1, trials["pump_count"] - trials["prev_pumps"], np.nan)
        trials["risk_after_collect"] = np.where(prev_collect == 1, trials["pump_count"] - trials["prev_pumps"], np.nan)

        if "pump_latency_mean" in trials.columns:
            trials["pump_speed_change"] = trials["pump_latency_mean"] - trials.groupby(block_col)["pump_latency_mean"].shift(1)
        else:
            trials["pump_speed_change"] = np.nan
    else:
        if "risk_after_explosion" not in trials.columns:
            trials["risk_after_explosion"] = np.nan
        if "risk_after_collect" not in trials.columns:
            trials["risk_after_collect"] = np.nan
        if "pump_speed_change" not in trials.columns:
            trials["pump_speed_change"] = np.nan

    # session-level behavioral aggregates
    sesPoints = trials["total_earnings"].dropna().max() if "total_earnings" in trials.columns else np.nan
    exp_N = int(trials["pop"].sum())
    collect_N = int(trials["collect"].sum())
    tN = int(len(trials))
    exp_rate = exp_N / tN if tN else np.nan
    collect_rate = collect_N / tN if tN else np.nan
    bonus_N = int(trials["nf_bonus_awarded_trial"].sum()) if "nf_bonus_awarded_trial" in trials.columns else 0

    smPmp = float(trials["pump_count"].mean()) if "pump_count" in trials.columns else np.nan
    SD_smPmp = float(trials["pump_count"].std(ddof=1)) if "pump_count" in trials.columns else np.nan

    trials["adjusted_pumps_trial"] = trials["pump_count"] if "pump_count" in trials.columns else np.nan
    smADJ_pmp = float(trials["adjusted_pumps_trial"].mean()) if "adjusted_pumps_trial" in trials.columns else np.nan
    SD_smADJ_pmp = float(trials["adjusted_pumps_trial"].std(ddof=1)) if "adjusted_pumps_trial" in trials.columns else np.nan

    mpmpL = float(trials["pump_latency_mean"].mean()) if "pump_latency_mean" in trials.columns else np.nan
    SD_mpmpL = float(trials["pump_latency_mean"].std(ddof=1)) if "pump_latency_mean" in trials.columns else np.nan
    medpmpL = float(trials["pump_latency_median"].mean()) if "pump_latency_median" in trials.columns else np.nan
    SD_medpmpL = float(trials["pump_latency_median"].std(ddof=1)) if "pump_latency_median" in trials.columns else np.nan

    # baselines from Trials OR Summary
    def _get_scalar_from_sources(keys, sources):
        for df in sources:
            if df is None or len(df) == 0:
                continue
            cols_lut = {str(c).strip().lower(): c for c in df.columns}
            for k in keys:
                kk = str(k).strip().lower()
                if kk in cols_lut:
                    col = cols_lut[kk]
                    ser = pd.to_numeric(df[col], errors="coerce")
                    if ser.notna().any():
                        return float(ser.dropna().iloc[0])
        return np.nan

    sources = [trials, summary]

    baseline_cols = {
        "baseline_eo_mu": _get_scalar_from_sources(
            ["baseline_eo_mu", "baseline_pre_eo_mu", "pretask_eob_avgtheta", "pretask_eo_avgtheta"], sources),
        "baseline_ec_mu": _get_scalar_from_sources(
            ["baseline_ec_mu", "baseline_pre_ec_mu", "pretask_ecb_avgtheta", "pretask_ec_avgtheta"], sources),
        "baseline_conc_mu": _get_scalar_from_sources(
            ["baseline_conc_mu", "baseline_pre_conc_mu", "pretask_cb_avgtheta", "pretask_conc_avgtheta"], sources),

        "baseline_post_eo_mu": _get_scalar_from_sources(
            ["baseline_post_eo_mu", "posttask_eob_avgtheta", "post_eob_avgtheta", "posttask_eo_avgtheta"], sources),
        "baseline_post_ec_mu": _get_scalar_from_sources(
            ["baseline_post_ec_mu", "posttask_ecb_avgtheta", "post_ecb_avgtheta", "posttask_ec_avgtheta"], sources),
        "baseline_post_conc_mu": _get_scalar_from_sources(
            ["baseline_post_conc_mu", "posttask_cb_avgtheta", "post_cb_avgtheta", "posttask_conc_avgtheta"], sources),
    }

    # expand to pump-level rows
    pump_rows = []
    for _, tr in trials.iterrows():
        n = int(tr["pump_count"]) if pd.notna(tr.get("pump_count", np.nan)) else 0
        if n <= 0:
            continue

        lat_list = safe_json_list(tr.get("pump_latencies_json", None))

        for p in range(1, n + 1):
            row = {}

            row["pID"] = str(bits["sub"]).replace("sub-", "").strip().upper()
            row["cond"] = bits["condition"]
            row["sID"] = bits["ses"]
            row["ses_num"] = bits["ses_num"]

            row["pPoints"] = np.nan
            row["pgsi_score"] = np.nan

            row["sesPoints"] = sesPoints
            row["exp_N"] = exp_N
            row["collect_N"] = collect_N
            row["bonus_N"] = bonus_N
            row["exp_rate"] = exp_rate
            row["collect_rate"] = collect_rate
            row["smPmp"] = smPmp
            row["SD_smPmp"] = SD_smPmp
            row["smADJ_pmp"] = smADJ_pmp
            row["SD_smADJ_pmp"] = SD_smADJ_pmp
            row["mpmpL"] = mpmpL
            row["SD_mpmpL"] = SD_mpmpL
            row["medpmpL"] = medpmpL
            row["SD_medpmpL"] = SD_medpmpL

            row["pretask_EOB_avgTheta"]  = baseline_cols.get("baseline_eo_mu", np.nan)
            row["posttask_EOB_avgTheta"] = baseline_cols.get("baseline_post_eo_mu", np.nan)
            row["pretask_ECB_avgTheta"]  = baseline_cols.get("baseline_ec_mu", np.nan)
            row["posttask_ECB_avgTheta"] = baseline_cols.get("baseline_post_ec_mu", np.nan)
            row["pretask_CB_avgTheta"]   = baseline_cols.get("baseline_conc_mu", np.nan)
            row["posttask_CB_avgTheta"]  = baseline_cols.get("baseline_post_conc_mu", np.nan)

            row["tID"] = int(tr["trial"]) if "trial" in tr and pd.notna(tr["trial"]) else np.nan
            row["pop"] = int(tr["pop"])
            row["collect"] = int(tr["collect"])
            row["pmp_count"] = int(tr["pump_count"]) if pd.notna(tr.get("pump_count", np.nan)) else np.nan
            row["t_earned"] = float(tr["trial_earnings"]) if "trial_earnings" in tr and pd.notna(tr["trial_earnings"]) else np.nan
            row["t_loss"] = float(tr["loss_if_pop"]) if "loss_if_pop" in tr and pd.notna(tr["loss_if_pop"]) else np.nan

            row["collectL_trial"] = float(tr["collect_latency_from_trial_start"]) if "collect_latency_from_trial_start" in tr and pd.notna(tr["collect_latency_from_trial_start"]) else np.nan
            row["collectL_ready"] = float(tr["collect_latency_from_ready"]) if "collect_latency_from_ready" in tr and pd.notna(tr["collect_latency_from_ready"]) else np.nan

            row["nf_high_fraction"] = float(tr["nf_high_fraction"]) if "nf_high_fraction" in tr and pd.notna(tr["nf_high_fraction"]) else np.nan
            row["nf_green_success"] = int(tr["nf_green_success"]) if "nf_green_success" in tr and pd.notna(tr["nf_green_success"]) else np.nan

            row["risk_after_explosion"] = float(tr.get("risk_after_explosion", np.nan)) if pd.notna(tr.get("risk_after_explosion", np.nan)) else np.nan
            row["risk_after_collect"] = float(tr.get("risk_after_collect", np.nan)) if pd.notna(tr.get("risk_after_collect", np.nan)) else np.nan
            row["pump_speed_change"] = float(tr.get("pump_speed_change", np.nan)) if pd.notna(tr.get("pump_speed_change", np.nan)) else np.nan

            row["pump_number"] = int(p)
            row["pmpID"] = f"{bits['sub']}_{bits['ses']}_T{int(row['tID'])}_P{p}" if pd.notna(row["tID"]) else f"{bits['sub']}_{bits['ses']}_P{p}"
            row["pmpL"] = float(lat_list[p-1]) if isinstance(lat_list, list) and len(lat_list) >= p else np.nan

            row["ses_mTheta"] = np.nan
            row["ses_SD_mThetaB"] = np.nan
            row["ses_avgP300_All"] = np.nan
            row["ses_avgP300_Fz"] = np.nan
            row["ses_avgP300_FCz"] = np.nan
            row["ses_avgP300_Cz"] = np.nan
            row["ses_avgP300_Pz"] = np.nan

            row["pre200_pop_Theta"] = np.nan
            row["post200_pop_Theta"] = np.nan
            row["pre200_collect_Theta"] = np.nan
            row["post200_collect_Theta"] = np.nan
            row["pre200_pmp_Theta"] = np.nan

            row["theta_before_pump"] = np.nan
            row["theta_before_pump_Fp1"] = np.nan
            row["theta_before_pump_Fp2"] = np.nan
            row["frontal_asymmetry"] = np.nan

            row["trial_mTheta"] = np.nan
            row["trial_SD_mThetaB"] = np.nan

            pump_rows.append(row)

    pump_df = pd.DataFrame(pump_rows)
    return pump_df, trials, summary

# ----------------------------
# Pair processing
# ----------------------------
def process_pair(beh_fp: Path, xdf_fp: Path | None):
    pump_df, trials_df, summary_df = beh_to_pump_table(beh_fp)
    if pump_df is None or len(pump_df) == 0:
        return None

    if xdf_fp is None:
        return pump_df

    try:
        streams, _ = pyxdf.load_xdf(str(xdf_fp))
        eeg_stream, marker_stream = pick_streams(streams)
        if eeg_stream is None or marker_stream is None:
            return pump_df

        raw, eeg_t = build_raw_from_xdf_eeg(eeg_stream)
        raw = preprocess_raw(raw)
        raw.resample(TARGET_SFREQ, verbose=False)
        eeg_t0 = float(eeg_t[0])

        md = markers_to_df(marker_stream)

        ses_mTheta, ses_SD_mThetaB = extract_session_theta_metrics(raw)
        p300 = extract_p300_metrics(raw, md, eeg_t0)
        ev_theta = extract_event_theta_windows(raw, md, eeg_t0)

        pump_theta = theta_before_each_pump(raw, md, eeg_t0)
        trial_theta = trial_theta_from_markers(raw, md, eeg_t0)

        pump_df["ses_mTheta"] = ses_mTheta
        pump_df["ses_SD_mThetaB"] = ses_SD_mThetaB
        for k, v in p300.items():
            pump_df[k] = v
        for k, v in ev_theta.items():
            pump_df[k] = v

        pump_df["tID"] = pd.to_numeric(pump_df["tID"], errors="coerce").astype("Int64")
        pump_df["pump_number"] = pd.to_numeric(pump_df["pump_number"], errors="coerce").astype("Int64")

        if trial_theta is not None and len(trial_theta) > 0:
            tt = trial_theta.copy()
            tt["tID"] = pd.to_numeric(tt["trial"], errors="coerce").astype("Int64")
            tt = tt.drop(columns=["trial"], errors="ignore")

            for c in ["trial_mTheta", "trial_SD_mThetaB"]:
                if c in pump_df.columns:
                    pump_df = pump_df.drop(columns=[c])

            pump_df = pump_df.merge(tt, on="tID", how="left")

        if pump_theta is not None and len(pump_theta) > 0:
            pt = pump_theta.copy()
            pt["tID"] = pd.to_numeric(pt["trial"], errors="coerce").astype("Int64")
            pt["pump_number"] = pd.to_numeric(pt["pump_number"], errors="coerce").astype("Int64")
            pt = pt.drop(columns=["trial"], errors="ignore")

            for c in ["theta_before_pump", "theta_before_pump_Fp1", "theta_before_pump_Fp2", "frontal_asymmetry"]:
                if c in pump_df.columns:
                    pump_df = pump_df.drop(columns=[c])

            pump_df = pump_df.merge(pt, on=["tID", "pump_number"], how="left")

        for c in ["trial_mTheta","trial_SD_mThetaB","theta_before_pump","theta_before_pump_Fp1","theta_before_pump_Fp2","frontal_asymmetry"]:
            if c not in pump_df.columns:
                pump_df[c] = np.nan

        return pump_df

    except Exception as e:
        print("⚠️ XDF failed:", xdf_fp.name, "->", e)
        return pump_df

# ----------------------------
# Main: find files + build master
# ----------------------------
beh_files = sorted(ROOT.rglob("*_beh.xlsx"))
xdf_files = sorted(ROOT.rglob("*.xdf")) + sorted(ROOT.rglob("*.xdfz"))

print("Found BEH files:", len(beh_files))
print("Found XDF files:", len(xdf_files))

xdf_map = {}
for xf in xdf_files:
    key = xf.name.split("_task")[0]
    xdf_map.setdefault(key, []).append(xf)

all_rows = []
missing_xdf = 0
processed = 0

for bf in beh_files:
    key = bf.name.split("_task")[0]
    xmatch = xdf_map.get(key, [])
    xdf_fp = xmatch[0] if len(xmatch) else None
    if xdf_fp is None:
        missing_xdf += 1

    print("\n---")
    print("BEH:", bf.name)
    print("XDF:", xdf_fp.name if xdf_fp else "MISSING (BEH-only)")

    out = process_pair(bf, xdf_fp)
    if out is not None and len(out) > 0:
        all_rows.append(out)
        processed += 1
        print("Rows:", out.shape)

if len(all_rows) == 0:
    raise RuntimeError("No output rows produced. Check BEH files contain Trials + pump_count > 0.")

master = pd.concat(all_rows, ignore_index=True)
print("\n✅ sessions processed:", processed)
print("⚠️ missing XDF sessions:", missing_xdf)
print("✅ master shape:", master.shape)

# ----------------------------
# Participant total points over all sessions (pPoints)
# ----------------------------
if "pID" in master.columns and "sesPoints" in master.columns and "sID" in master.columns:
    master["pID"] = (
        master["pID"]
        .astype(str)
        .str.replace("sub-", "", regex=False)
        .str.strip()
        .str.upper()
    )
    tmp = master[["pID","sID","sesPoints"]].drop_duplicates()
    totals = tmp.groupby("pID")["sesPoints"].sum()
    master["pPoints"] = master["pID"].map(totals)

# ----------------------------
# Build participant-level survey table from session 5 BEH files
# ----------------------------
survey_rows = []

for bf in beh_files:
    bits = parse_bids_bits(bf.name)
    if pd.isna(bits["ses_num"]) or int(bits["ses_num"]) != 5:
        continue

    sheets = read_beh_sheets(bf)
    survey_df = get_sheet(sheets, ["survey", "surveys"])
    if survey_df is None or len(survey_df) == 0:
        continue

    participant_id = str(bits["sub"]).replace("sub-", "").strip().upper()
    survey_row = extract_survey_row(survey_df, participant_id)
    if survey_row is not None:
        survey_rows.append(survey_row)

survey_master = pd.DataFrame(survey_rows) if len(survey_rows) else pd.DataFrame(columns=["pID"])

if len(survey_master):
    survey_master["pID"] = (
        survey_master["pID"]
        .astype(str)
        .str.replace("sub-", "", regex=False)
        .str.strip()
        .str.upper()
    )
    survey_master = survey_master.drop_duplicates(subset=["pID"], keep="first")
    print("Survey rows loaded:", survey_master.shape)
else:
    print("No survey rows found.")

# ----------------------------
# Merge PGSI + manifest fields
# ----------------------------
if manifest is not None and "pID" in manifest.columns:
    master["pID"] = normalize_pid_series(master["pID"])

    keep_manifest_cols = [c for c in manifest.columns if c in ["pID", "pgsi_score", "risk_group", "cond_manifest"]]
    manifest_small = manifest[keep_manifest_cols].drop_duplicates(subset=["pID"], keep="first")

    # debug before merge
    master_ids = set(master["pID"].dropna().unique())
    manifest_ids = set(manifest_small["pID"].dropna().unique())
    overlap = sorted(master_ids.intersection(manifest_ids))

    print("Master unique pIDs:", len(master_ids))
    print("Manifest unique pIDs:", len(manifest_ids))
    print("Overlapping pIDs:", len(overlap))
    print("First overlapping IDs:", overlap[:10])

    master = master.merge(manifest_small, on="pID", how="left")

    if "pgsi_score" in master.columns:
        print("Non-missing pgsi_score rows after merge:", int(master["pgsi_score"].notna().sum()))
        print(master[["pID", "pgsi_score"]].dropna().drop_duplicates().head(10))
# ----------------------------
# Merge survey responses from session 5
# ----------------------------
if len(survey_master):
    master = master.merge(survey_master, on="pID", how="left")

# ----------------------------
# Ensure final schema columns exist
# ----------------------------
schema_cols = [
    "pID","cond","pPoints","pgsi_score","sID","ses_num",
    "sesPoints","ses_mTheta","ses_SD_mThetaB",
    "ses_avgP300_All","ses_avgP300_Fz","ses_avgP300_FCz","ses_avgP300_Cz","ses_avgP300_Pz",
    "smPmp","SD_smPmp","smADJ_pmp","SD_smADJ_pmp",
    "exp_rate","exp_N","bonus_N","collect_N","collect_rate",
    "pretask_EOB_avgTheta","posttask_EOB_avgTheta",
    "pretask_ECB_avgTheta","posttask_ECB_avgTheta",
    "pretask_CB_avgTheta","posttask_CB_avgTheta",
    "tID","trial_mTheta","trial_SD_mThetaB",
    "pop","collect","nf_high_fraction","nf_green_success","t_earned","t_loss",
    "collectL_trial","collectL_ready",
    "pmp_count",
    "pre200_pop_Theta","post200_pop_Theta","pre200_collect_Theta","post200_collect_Theta","pre200_pmp_Theta",
    "risk_after_explosion","risk_after_collect","pump_speed_change",
    "pump_number","pmpID","pmpL",
    "mpmpL","SD_mpmpL","medpmpL","SD_medpmpL",
    "theta_before_pump","theta_before_pump_Fp1","theta_before_pump_Fp2","frontal_asymmetry",
]

for c in schema_cols:
    if c not in master.columns:
        master[c] = np.nan

# ----------------------------
# Reorder: pID, then survey columns, then schema, then everything else
# ----------------------------
survey_cols = [c for c in master.columns if str(c).lower().startswith("survey_")]

base_order = [c for c in schema_cols if c in master.columns and c != "pID"]
remaining = [c for c in master.columns if c not in (["pID"] + survey_cols + base_order)]

ordered = ["pID"] + survey_cols + base_order + remaining
master = master[ordered]

# ----------------------------
# Quick checks
# ----------------------------
print("Columns containing 'pgsi':", [c for c in master.columns if "pgsi" in str(c).lower()])
print("Survey columns found:", len([c for c in master.columns if str(c).lower().startswith("survey_")]))
if "pgsi_score" in master.columns:
    vals = master["pgsi_score"].dropna().unique().tolist()
    try:
        vals = sorted(vals)[:20]
    except Exception:
        vals = vals[:20]
    print("Unique non-missing PGSI values:", vals)

# ----------------------------
# Save
# ----------------------------
csv_path = OUT / "BART_master_pump_level.csv"
xlsx_path = OUT / "BART_master_pump_level.xlsx"
master.to_csv(csv_path, index=False)
master.to_excel(xlsx_path, index=False)

print("\n✅ Saved CSV:", csv_path)
print("✅ Saved XLSX:", xlsx_path)

# Preview
master.head(10)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 20.6 MB/s eta 0:00:00
No manifest.xlsx found.
Found BEH files: 90
Found XDF files: 84

---
BEH: sub-P001_ses-S001_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P001_ses-S001_task-Default_run-001_eeg.xdf
Rows: (536, 60)

---
BEH: sub-P001_ses-S002_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P001_ses-S002_task-Default_run-001_eeg.xdf
Rows: (691, 60)

---
BEH: sub-P001_ses-S003_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P001_ses-S003_task-Default_run-001_eeg.xdf
Rows: (688, 60)

---
BEH: sub-P001_ses-S004_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: MISSING (BEH-only)
Rows: (562, 60)

---
BEH: sub-P001_ses-S005_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P001_ses-S005_task-Default_run-001_eeg.xdf
Rows: (678, 60)

---
BEH: sub-P003_ses-S001_task-BART_cond-nf_sham_run-001_beh.xlsx
XDF: sub-P003_ses-S001_task-Default_run-001_eeg.xdf
Rows: (305, 60)

---
BEH: sub-P003_ses-S002_task-BART_cond-nf_sham_run-001_beh.xlsx
XDF

Rows: (786, 60)

---
BEH: sub-P005_ses-S001_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P005_ses-S001_task-Default_run-001_eeg.xdf
Rows: (633, 60)

---
BEH: sub-P005_ses-S002_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P005_ses-S002_task-Default_run-001_eeg.xdf
Rows: (580, 60)

---
BEH: sub-P005_ses-S003_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P005_ses-S003_task-Default_run-001_eeg.xdf
Rows: (618, 60)

---
BEH: sub-P005_ses-S004_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P005_ses-S004_task-Default_run-001_eeg.xdf
Rows: (621, 60)

---
BEH: sub-P005_ses-S005_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P005_ses-S005_task-Default_run-001_eeg.xdf
Rows: (574, 60)

---
BEH: sub-P006_ses-S001_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P006_ses-S001_task-Default_run-001_eeg.xdf
Rows: (507, 60)

---
BEH: sub-P006_ses-S002_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P006_ses-S002_task-Default_run-001_eeg.xdf
Rows: (527, 60)

---
BEH: sub-P006_ses-S003_task

Rows: (760, 60)

---
BEH: sub-P018_ses-S004_task-BART_cond-nf_sham_run-001_beh.xlsx
XDF: sub-P018_ses-S004_task-Default_run-001_eeg.xdf
Rows: (646, 60)

---
BEH: sub-P018_ses-S005_task-BART_cond-nf_sham_run-001_beh.xlsx
XDF: sub-P018_ses-S005_task-Default_run-001_eeg.xdf
Rows: (807, 60)

---
BEH: sub-P019_ses-S001_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P019_ses-S001_task-Default_run-001_eeg.xdf
Rows: (618, 60)

---
BEH: sub-P019_ses-S002_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P019_ses-S002_task-Default_run-001_eeg.xdf
Rows: (585, 60)

---
BEH: sub-P019_ses-S003_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P019_ses-S003_task-Default_run-001_eeg.xdf
Rows: (515, 60)

---
BEH: sub-P019_ses-S004_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P019_ses-S004_task-Default_run-001_eeg.xdf
Rows: (534, 60)

---
BEH: sub-P019_ses-S005_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P019_ses-S005_task-Default_run-001_eeg.xdf
Rows: (552, 60)

✅ sessions processed: 90
⚠️ mis

,pID,survey_r1_sub,survey_r1_ses,survey_r1_run,survey_r1_task,survey_r1_condition,survey_r1_qid,survey_r1_section,survey_r1_question,survey_r1_response_1to5,...,pmpID,pmpL,mpmpL,SD_mpmpL,medpmpL,SD_medpmpL,theta_before_pump,theta_before_pump_Fp1,theta_before_pump_Fp2,frontal_asymmetry
0,P001,P001,S005,1,BART,nf_real,S1_Q1,Stress,"Overall, how stressful did you find the tasks ...",1,...,P001_S001_T1_P1,0.0234,0.270922,0.08398,0.237424,0.09864,4.527025e-11,2.317708e-11,3.801747e-11,0.478605
1,P001,P001,S005,1,BART,nf_real,S1_Q1,Stress,"Overall, how stressful did you find the tasks ...",1,...,P001_S001_T1_P1,0.0234,0.270922,0.08398,0.237424,0.09864,2.708999e-11,3.875262e-11,4.269164e-11,0.094481
2,P001,P001,S005,1,BART,nf_real,S1_Q1,Stress,"Overall, how stressful did you find the tasks ...",1,...,P001_S001_T1_P2,1.3350,0.270922,0.08398,0.237424,0.09864,4.678572e-11,2.006399e-11,5.438994e-11,0.966833
3,P001,P001,S005,1,BART,nf_real,S1_Q1,Stress,"Overall, how stressful did you find the tasks ...",1,...,P001_S001_T1_P2,1.3350,0.270922,0.08398,0.237424,0.09864,6.140069e-11,9.258605e-11,1.432722e-10,0.432821
4,P001,P001,S005,1,BART,nf_real,S1_Q1,Stress,"Overall, how stressful did you find the tasks ...",1,...,P001_S001_T1_P3,0.3838,0.270922,0.08398,0.237424,0.09864,2.324976e-11,1.047581e-11,1.462728e-11,0.308777
5,P001,P001,S005,1,BART,nf_real,S1_Q1,Stress,"Overall, how stressful did you find the tasks ...",1,...,P001_S001_T1_P3,0.3838,0.270922,0.08398,0.237424,0.09864,4.378374e-11,4.157826e-11,4.647099e-11,0.108775
6,P001,P001,S005,1,BART,nf_real,S1_Q1,Stress,"Overall, how stressful did you find the tasks ...",1,...,P001_S001_T1_P4,0.3838,0.270922,0.08398,0.237424,0.09864,6.958198e-11,1.502712e-10,1.646237e-10,0.090644
7,P001,P001,S005,1,BART,nf_real,S1_Q1,Stress,"Overall, how stressful did you find the tasks ...",1,...,P001_S001_T1_P4,0.3838,0.270922,0.08398,0.237424,0.09864,1.505707e-11,1.371341e-11,6.212980e-12,-0.712877
8,P001,P001,S005,1,BART,nf_real,S1_Q1,Stress,"Overall, how stressful did you find the tasks ...",1,...,P001_S001_T1_P5,0.4171,0.270922,0.08398,0.237424,0.09864,4.014135e-11,4.098451e-11,5.847166e-11,0.348199
9,P001,P001,S005,1,BART,nf_real,S1_Q1,Stress,"Overall, how stressful did you find the tasks ...",1,...,P001_S001_T1_P5,0.4171,0.270922,0.08398,0.237424,0.09864,3.838771e-11,2.838379e-11,3.433010e-11,0.184292


In [ ]:
# ============================================================
# BART MULTIMODAL MASTER PIPELINE (Pump-level) — ~55+ columns
# Input: *_beh.xlsx + matching *.xdf/*.xdfz (LabRecorder)
# Output: outputs/BART_master_pump_level.{csv,xlsx}
#
# Includes:
#  - Behavioral pump/trial/session metrics
#  - EEG theta + P300 metrics from XDF
#  - PGSI / risk_group / gender from manifest.xlsx
#  - Survey responses from session 5 BEH survey sheet
# ============================================================

!pip -q install pyxdf mne openpyxl pandas numpy

import re, json
from pathlib import Path
import numpy as np
import pandas as pd

import pyxdf
import mne

# ----------------------------
# CONFIG
# ----------------------------
ROOT = Path("/content") if Path("/content").exists() else Path(".")
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)

TARGET_SFREQ = 512.0

# ERP epoching (explode)
ERP_TMIN, ERP_TMAX = -0.2, 0.8
ERP_BASELINE = (ERP_TMIN, 0.0)
P3_WIN = (0.300, 0.450)

# Theta windows
WIN200 = 0.200
THETA_BAND = (4.0, 7.0)

# ActiCAP channel order
EEG_ORDER = ["Fz","Cz","Pz","POz","Fp1","Fp2","F3","F4","FCz","C3","C4","CPz","P3","P4","O1","O2"]

FRONTAL_THETA_CHS = ["Fp1","Fp2","Fz","F3","F4","FCz"]

# ----------------------------
# ID normalization
# ----------------------------
def normalize_pid_series(s):
    s = s.astype(str).str.strip().str.upper()

    s = s.str.replace("SUB-", "", regex=False)
    s = s.str.replace("SUB_", "", regex=False)
    s = s.str.replace("PARTICIPANT", "", regex=False)
    s = s.str.replace("PID", "", regex=False)
    s = s.str.replace("P_ID", "", regex=False)
    s = s.str.replace("SUBJECT_ID", "", regex=False)

    nums = s.str.extract(r"(\d+)")[0]
    out = nums.fillna("").apply(lambda x: f"P{int(x):03d}" if x != "" else np.nan)
    out = out.where(out.notna(), s)
    return out

# ----------------------------
# Optional manifest
# ----------------------------
manifest_path = next(iter(ROOT.rglob("manifest.xlsx")), None)
manifest = None

if manifest_path is not None and Path(manifest_path).exists():
    xls = pd.ExcelFile(manifest_path)
    manifest_candidates = []

    for sh in xls.sheet_names:
        df = pd.read_excel(manifest_path, sheet_name=sh)
        df.columns = [str(c).strip() for c in df.columns]

        cols_l = {str(c).strip().lower(): c for c in df.columns}

        pid_col = None
        pgsi_col = None
        cond_col = None
        risk_col = None
        gender_col = None

        for k in ["subject_id", "pid", "p_id", "participant_id", "participant", "subject", "sub", "id"]:
            if k in cols_l:
                pid_col = cols_l[k]
                break

        for k in ["pgsi score", "pgsi_score", "pgsi", "pgsi total", "pgsi_total", "pgsi score total"]:
            if k in cols_l:
                pgsi_col = cols_l[k]
                break

        for k in ["condition", "cond"]:
            if k in cols_l:
                cond_col = cols_l[k]
                break

        for k in ["high/low", "risk_group", "risk", "group"]:
            if k in cols_l:
                risk_col = cols_l[k]
                break

        for k in ["gender", "sex"]:
            if k in cols_l:
                gender_col = cols_l[k]
                break

        if pid_col is not None:
            keep = [pid_col]
            rename_map = {pid_col: "pID"}

            if pgsi_col is not None:
                keep.append(pgsi_col)
                rename_map[pgsi_col] = "pgsi_score"
            if cond_col is not None:
                keep.append(cond_col)
                rename_map[cond_col] = "cond_manifest"
            if risk_col is not None:
                keep.append(risk_col)
                rename_map[risk_col] = "risk_group"
            if gender_col is not None:
                keep.append(gender_col)
                rename_map[gender_col] = "gender"

            tmp = df[keep].copy().rename(columns=rename_map)
            tmp["pID"] = normalize_pid_series(tmp["pID"])

            if "pgsi_score" in tmp.columns:
                tmp["pgsi_score"] = pd.to_numeric(tmp["pgsi_score"], errors="coerce")

            manifest_candidates.append((sh, tmp))

    chosen = None
    for sh, df in manifest_candidates:
        if "pgsi_score" in df.columns:
            chosen = (sh, df)
            break
    if chosen is None and len(manifest_candidates):
        chosen = manifest_candidates[0]

    if chosen is not None:
        sh, manifest = chosen
        manifest = manifest.drop_duplicates(subset=["pID"], keep="first")
        print("Loaded manifest:", manifest_path)
        print("Using manifest sheet:", sh)
        print("Manifest columns:", manifest.columns.tolist())
        if "pgsi_score" in manifest.columns:
            print("Manifest non-missing PGSI rows:", int(manifest["pgsi_score"].notna().sum()))
            print(manifest[["pID", "pgsi_score"]].dropna().head(10))
    else:
        print("Manifest found, but no usable participant ID column detected.")
else:
    print("No manifest.xlsx found.")

# ----------------------------
# Utilities
# ----------------------------
def parse_bids_bits(name: str):
    base = Path(name).name
    sub = re.search(r"sub-([A-Za-z0-9]+)", base)
    ses = re.search(r"ses-([A-Za-z0-9]+)", base)
    run = re.search(r"run-([0-9]+)", base)
    task = re.search(r"task-([A-Za-z0-9]+)", base)
    cond = re.search(r"cond-([A-Za-z0-9_]+)", base)

    out = {
        "sub": sub.group(1) if sub else None,
        "ses": ses.group(1) if ses else None,
        "run": run.group(1) if run else None,
        "task": task.group(1) if task else None,
        "condition": cond.group(1) if cond else None,
    }
    if out["ses"] is not None:
        m = re.search(r"(\d+)", out["ses"])
        out["ses_num"] = int(m.group(1)) if m else np.nan
    else:
        out["ses_num"] = np.nan
    return out

def read_beh_sheets(fp: Path):
    xls = pd.ExcelFile(fp)
    sheets = {}
    for sh in xls.sheet_names:
        df = pd.read_excel(fp, sheet_name=sh)
        df.columns = [str(c).strip() for c in df.columns]
        sheets[sh.lower()] = df
    return sheets

def get_sheet(sheets: dict, candidates):
    for c in candidates:
        if c in sheets:
            return sheets[c]
    return None

def safe_json_list(x):
    try:
        return json.loads(x) if isinstance(x, str) else (x if isinstance(x, list) else None)
    except Exception:
        return None

def extract_survey_row(survey_df, participant_id):
    if survey_df is None or len(survey_df) == 0:
        return None

    df = survey_df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    participant_id = normalize_pid_series(pd.Series([participant_id]))[0]

    if len(df) == 1:
        row = df.iloc[0].to_dict()
        row["pID"] = participant_id
        return row

    cols_l = {str(c).strip().lower(): c for c in df.columns}
    q_col = None
    r_col = None

    for k in ["question", "item", "prompt"]:
        if k in cols_l:
            q_col = cols_l[k]
            break
    for k in ["response", "answer", "value", "score"]:
        if k in cols_l:
            r_col = cols_l[k]
            break

    if q_col is not None and r_col is not None:
        out = {"pID": participant_id}
        for _, r in df.iterrows():
            q = str(r[q_col]).strip()
            q_clean = re.sub(r"[^A-Za-z0-9]+", "_", q).strip("_").lower()
            out[f"survey_{q_clean}"] = r[r_col]
        return out

    out = {"pID": participant_id}
    for i, (_, r) in enumerate(df.iterrows(), start=1):
        for c in df.columns:
            cname = re.sub(r"[^A-Za-z0-9]+", "_", str(c)).strip("_").lower()
            out[f"survey_r{i}_{cname}"] = r[c]
    return out

# ----------------------------
# XDF helpers
# ----------------------------
def parse_marker_message(msg):
    s = str(msg)
    parts = s.split(";")
    out = {"code": parts[0].strip()}
    for p in parts[1:]:
        if "=" in p:
            k, v = p.split("=", 1)
            out[k.strip()] = v.strip()
    return out

def _stream_name_lower(s):
    info = s.get("info", {})
    nm = info.get("name", "")
    if isinstance(nm, (list, tuple, np.ndarray)):
        nm = nm[0] if len(nm) else ""
    return str(nm).strip().lower()

def pick_streams(streams):
    eeg = None
    best = -1

    for s in streams:
        ts = s.get("time_series", None)
        if ts is None:
            continue
        arr = np.asarray(ts)
        ch = 1 if arr.ndim == 1 else arr.shape[1]
        if ch > best:
            eeg = s
            best = ch

    markers = None
    for s in streams:
        ts = s.get("time_series", None)
        if ts is None:
            continue
        arr = np.asarray(ts)
        ch = 1 if arr.ndim == 1 else arr.shape[1]
        if ch != 1:
            continue
        nm = _stream_name_lower(s)
        if ("bart" in nm) or ("marker" in nm) or ("markers" in nm) or ("event" in nm):
            markers = s
            break

    if markers is None:
        for s in streams:
            ts = s.get("time_series", None)
            if ts is None:
                continue
            arr = np.asarray(ts)
            ch = 1 if arr.ndim == 1 else arr.shape[1]
            if ch == 1:
                markers = s
                break

    return eeg, markers

def markers_to_df(marker_stream):
    m_ts = np.asarray(marker_stream["time_stamps"], dtype=float)
    m_val = np.asarray(marker_stream["time_series"], dtype=object)

    rows = []
    for t, v in zip(m_ts, m_val):
        msg = v[0] if isinstance(v, (list, tuple, np.ndarray)) else v
        d = parse_marker_message(msg)
        d["lsltime"] = float(t)
        rows.append(d)

    df = pd.DataFrame(rows)
    for k in ["trial", "pump", "loss", "earnings", "total"]:
        if k in df.columns:
            df[k] = pd.to_numeric(df[k], errors="coerce")
    return df

def build_raw_from_xdf_eeg(eeg_stream):
    data = np.asarray(eeg_stream["time_series"])
    t = np.asarray(eeg_stream["time_stamps"], dtype=float)
    if data.ndim == 1:
        data = data[:, None]
    n_samp, n_ch = data.shape

    sfreq_est = 1.0 / np.median(np.diff(t))
    ch_names = EEG_ORDER[:n_ch] if n_ch <= len(EEG_ORDER) else [f"EEG{ii+1:02d}" for ii in range(n_ch)]
    arr = data.T.astype(float)

    peak = float(np.nanmax(np.abs(arr)))
    if peak > 1e3:
        arr *= 1e-6
    elif peak > 5.0:
        arr *= 1e-3

    info = mne.create_info(ch_names=ch_names, sfreq=sfreq_est, ch_types=["eeg"] * n_ch)
    raw = mne.io.RawArray(arr, info, verbose=False)
    return raw, t

def preprocess_raw(raw):
    raw = raw.copy()
    raw.filter(0.1, 30.0, picks="eeg", verbose=False)
    try:
        raw.notch_filter([60], picks="eeg", verbose=False)
    except Exception:
        pass
    try:
        raw.set_eeg_reference("average", verbose=False)
    except Exception:
        pass
    return raw

def lsltime_to_sample(lsltime, eeg_t0, sfreq):
    return int(np.round((lsltime - eeg_t0) * sfreq))

# ----------------------------
# EEG feature extraction
# ----------------------------
def compute_theta_power_window(raw_theta, sample, tmin_s, tmax_s, picks):
    sf = raw_theta.info["sfreq"]
    start = int(sample + np.round(tmin_s * sf))
    stop  = int(sample + np.round(tmax_s * sf))
    start = max(start, 0)
    stop  = min(stop, raw_theta.n_times)
    if stop <= start + 2:
        return np.nan
    seg = raw_theta.get_data(picks=picks, start=start, stop=stop)
    return float(np.mean(seg**2))

def extract_session_theta_metrics(raw):
    rtheta = raw.copy().filter(THETA_BAND[0], THETA_BAND[1], picks="eeg", verbose=False)
    picks = [raw.ch_names.index(ch) for ch in FRONTAL_THETA_CHS if ch in raw.ch_names]
    if len(picks) == 0:
        return np.nan, np.nan
    X = rtheta.get_data(picks=picks)
    theta_ts = np.mean(X**2, axis=0)
    return float(np.mean(theta_ts)), float(np.std(theta_ts, ddof=1))

def extract_p300_metrics(raw, md, eeg_t0):
    keys = ["ses_avgP300_All","ses_avgP300_Fz","ses_avgP300_FCz","ses_avgP300_Cz","ses_avgP300_Pz"]
    if "code" not in md.columns:
        return {k: np.nan for k in keys}

    exp = md[md["code"].astype(str).str.upper().eq("BART_EXPLODE")].copy()
    if len(exp) == 0:
        return {k: np.nan for k in keys}

    sf = raw.info["sfreq"]
    exp = exp.dropna(subset=["lsltime"]).copy()
    exp["sample"] = exp["lsltime"].apply(lambda t: lsltime_to_sample(float(t), eeg_t0, sf)).astype(int)

    min_samp = int(np.ceil((-ERP_TMIN) * sf))
    exp = exp[(exp["sample"] >= min_samp) & (exp["sample"] < raw.n_times)].copy()
    if len(exp) == 0:
        return {k: np.nan for k in keys}

    events = np.c_[exp["sample"].to_numpy(), np.zeros(len(exp), int), np.ones(len(exp), int)]
    epochs = mne.Epochs(
        raw, events, event_id={"EXPLODE": 1},
        tmin=ERP_TMIN, tmax=ERP_TMAX, baseline=ERP_BASELINE,
        preload=True, reject_by_annotation=True, verbose=False
    )
    if len(epochs) == 0:
        return {k: np.nan for k in keys}

    ev = epochs.average()
    times = ev.times
    m1 = np.argmin(np.abs(times - P3_WIN[0]))
    m2 = np.argmin(np.abs(times - P3_WIN[1]))

    out = {}
    vals = []
    for ch in ["Fz","FCz","Cz","Pz"]:
        k = f"ses_avgP300_{ch}"
        if ch in ev.ch_names:
            idx = ev.ch_names.index(ch)
            erp_uV = ev.data[idx] * 1e6
            p3 = float(np.max(erp_uV[m1:m2]))
            out[k] = p3
            vals.append(p3)
        else:
            out[k] = np.nan
    out["ses_avgP300_All"] = float(np.nanmean(vals)) if len(vals) else np.nan
    return out

def extract_event_theta_windows(raw, md, eeg_t0):
    rtheta = raw.copy().filter(THETA_BAND[0], THETA_BAND[1], picks="eeg", verbose=False)
    sf = raw.info["sfreq"]
    picks = [raw.ch_names.index(ch) for ch in FRONTAL_THETA_CHS if ch in raw.ch_names]
    if len(picks) == 0:
        return {
            "pre200_pop_Theta": np.nan, "post200_pop_Theta": np.nan,
            "pre200_collect_Theta": np.nan, "post200_collect_Theta": np.nan,
            "pre200_pmp_Theta": np.nan,
        }

    def _mean_prepost(code, pre=True):
        d = md[md["code"].astype(str).str.upper().eq(code)].dropna(subset=["lsltime"]).copy()
        if len(d) == 0:
            return np.nan
        d["sample"] = d["lsltime"].apply(lambda t: lsltime_to_sample(float(t), eeg_t0, sf)).astype(int)
        vals = []
        for s in d["sample"].tolist():
            v = compute_theta_power_window(rtheta, s, -WIN200, 0.0, picks) if pre else compute_theta_power_window(rtheta, s, 0.0, WIN200, picks)
            vals.append(v)
        return float(np.nanmean(vals)) if len(vals) else np.nan

    return {
        "pre200_pop_Theta": _mean_prepost("BART_EXPLODE", pre=True),
        "post200_pop_Theta": _mean_prepost("BART_EXPLODE", pre=False),
        "pre200_collect_Theta": _mean_prepost("BART_COLLECT", pre=True),
        "post200_collect_Theta": _mean_prepost("BART_COLLECT", pre=False),
        "pre200_pmp_Theta": _mean_prepost("BART_PUMP", pre=True),
    }

def theta_before_each_pump(raw, md, eeg_t0):
    pumps = md[md["code"].astype(str).str.upper().eq("BART_PUMP")].copy()
    if len(pumps) == 0 or ("trial" not in pumps.columns) or ("pump" not in pumps.columns):
        return pd.DataFrame(columns=["trial","pump_number","theta_before_pump","theta_before_pump_Fp1","theta_before_pump_Fp2","frontal_asymmetry"])

    rtheta = raw.copy().filter(THETA_BAND[0], THETA_BAND[1], picks="eeg", verbose=False)
    sf = raw.info["sfreq"]

    pumps["trial"] = pd.to_numeric(pumps["trial"], errors="coerce")
    pumps["pump_number"] = pd.to_numeric(pumps["pump"], errors="coerce")
    pumps = pumps.dropna(subset=["trial","pump_number","lsltime"]).copy()

    pumps["trial"] = pumps["trial"].astype(int)
    pumps["pump_number"] = pumps["pump_number"].astype(int)
    pumps["sample"] = pumps["lsltime"].apply(lambda t: lsltime_to_sample(float(t), eeg_t0, sf)).astype(int)

    min_samp = int(np.ceil(WIN200 * sf))
    pumps = pumps[(pumps["sample"] >= min_samp) & (pumps["sample"] < raw.n_times)].copy()
    if len(pumps) == 0:
        return pd.DataFrame(columns=["trial","pump_number","theta_before_pump","theta_before_pump_Fp1","theta_before_pump_Fp2","frontal_asymmetry"])

    frontal_picks = [raw.ch_names.index(ch) for ch in FRONTAL_THETA_CHS if ch in raw.ch_names]
    fp1_pick = [raw.ch_names.index("Fp1")] if "Fp1" in raw.ch_names else []
    fp2_pick = [raw.ch_names.index("Fp2")] if "Fp2" in raw.ch_names else []

    rows = []
    for _, r in pumps.iterrows():
        s = int(r["sample"])
        pre_frontal = compute_theta_power_window(rtheta, s, -WIN200, 0.0, frontal_picks) if frontal_picks else np.nan
        pre_fp1 = compute_theta_power_window(rtheta, s, -WIN200, 0.0, fp1_pick) if fp1_pick else np.nan
        pre_fp2 = compute_theta_power_window(rtheta, s, -WIN200, 0.0, fp2_pick) if fp2_pick else np.nan
        fa = (np.log(pre_fp2 + 1e-12) - np.log(pre_fp1 + 1e-12)) if (np.isfinite(pre_fp1) and np.isfinite(pre_fp2)) else np.nan

        rows.append({
            "trial": int(r["trial"]),
            "pump_number": int(r["pump_number"]),
            "theta_before_pump": pre_frontal,
            "theta_before_pump_Fp1": pre_fp1,
            "theta_before_pump_Fp2": pre_fp2,
            "frontal_asymmetry": fa
        })

    return pd.DataFrame(rows)

def trial_theta_from_markers(raw, md, eeg_t0):
    sf = raw.info["sfreq"]
    rtheta = raw.copy().filter(THETA_BAND[0], THETA_BAND[1], picks="eeg", verbose=False)
    picks = [raw.ch_names.index(ch) for ch in FRONTAL_THETA_CHS if ch in raw.ch_names]
    if len(picks) == 0:
        return pd.DataFrame(columns=["trial","trial_mTheta","trial_SD_mThetaB"])

    starts = md[md["code"].astype(str).str.upper().eq("BART_TRIAL_START")].copy()
    ends = md[md["code"].astype(str).str.upper().isin(["BART_COLLECT","BART_EXPLODE"])].copy()
    if "trial" not in starts.columns or "trial" not in ends.columns:
        return pd.DataFrame(columns=["trial","trial_mTheta","trial_SD_mThetaB"])

    starts["trial"] = pd.to_numeric(starts["trial"], errors="coerce")
    ends["trial"] = pd.to_numeric(ends["trial"], errors="coerce")
    starts = starts.dropna(subset=["trial","lsltime"]).copy()
    ends = ends.dropna(subset=["trial","lsltime"]).copy()
    if len(starts) == 0 or len(ends) == 0:
        return pd.DataFrame(columns=["trial","trial_mTheta","trial_SD_mThetaB"])

    out = []
    for tnum, g0 in starts.groupby(starts["trial"].astype(int)):
        t_start = float(g0["lsltime"].min())
        g1 = ends[ends["trial"].astype(int) == int(tnum)]
        if len(g1) == 0:
            continue
        t_end = float(g1["lsltime"].min())
        if t_end <= t_start:
            continue

        s0 = lsltime_to_sample(t_start, eeg_t0, sf)
        s1 = lsltime_to_sample(t_end, eeg_t0, sf)
        s0 = max(s0, 0)
        s1 = min(s1, raw.n_times)
        if s1 <= s0 + int(sf * 0.2):
            continue

        seg = rtheta.get_data(picks=picks, start=s0, stop=s1)
        theta_ts = np.mean(seg**2, axis=0)

        out.append({
            "trial": int(tnum),
            "trial_mTheta": float(np.mean(theta_ts)),
            "trial_SD_mThetaB": float(np.std(theta_ts, ddof=1)),
        })

    return pd.DataFrame(out)

# ----------------------------
# BEH -> pump table
# ----------------------------
def beh_to_pump_table(beh_fp: Path):
    sheets = read_beh_sheets(beh_fp)
    trials = get_sheet(sheets, ["trials", "trial"])
    summary = get_sheet(sheets, ["summary"])

    if trials is None or len(trials) == 0:
        return None, None, None

    trials = trials.copy()
    bits = parse_bids_bits(beh_fp.name)

    if "exploded" in trials.columns:
        trials["pop"] = trials["exploded"].astype(str).str.upper().isin(["TRUE","1","YES"]).astype(int)
    else:
        trials["pop"] = 0

    if "collected" in trials.columns:
        trials["collect"] = pd.to_numeric(trials["collected"], errors="coerce").fillna(0).astype(int)
    else:
        trials["collect"] = trials["events"].astype(str).str.contains("collect", case=False).astype(int) if "events" in trials.columns else 0

    for c in [
        "trial","pump_count","trial_earnings","loss_if_pop","trial_value","total_earnings",
        "pump_latency_mean","pump_latency_median","pump_latency_first",
        "collect_latency_from_ready","collect_latency_from_trial_start",
        "nf_high_frac","nf_green_success",
        "risk_after_explosion","risk_after_collect","pump_speed_change",
        "nf_mean_normalized"
    ]:
        if c in trials.columns:
            trials[c] = pd.to_numeric(trials[c], errors="coerce")

    if "nf_high_frac" in trials.columns:
        trials["nf_high_fraction"] = pd.to_numeric(trials["nf_high_frac"], errors="coerce")
    elif "nf_high_fraction" not in trials.columns:
        trials["nf_high_fraction"] = np.nan

    if "nf_green_success" in trials.columns and trials["nf_green_success"].notna().any():
        trials["nf_green_success"] = pd.to_numeric(trials["nf_green_success"], errors="coerce")
    else:
        trials["nf_green_success"] = np.where(trials["nf_high_fraction"] > 0.75, 1, 0).astype(int) if "nf_high_fraction" in trials.columns else np.nan

    block_col = "block" if "block" in trials.columns else None
    if block_col is not None:
        trials = trials.sort_values([block_col, "trial"], kind="mergesort") if "trial" in trials.columns else trials
        g = trials.groupby(block_col)["nf_green_success"]
        trials["nf_bonus_awarded_trial"] = ((g.shift(0)==1) & (g.shift(1)==1) & (g.shift(2)==1)).astype(int)
    else:
        trials["nf_bonus_awarded_trial"] = 0

    if "pump_count" in trials.columns and block_col is not None and "trial" in trials.columns:
        trials["prev_pumps"] = trials.groupby(block_col)["pump_count"].shift(1)
        prev_pop = trials.groupby(block_col)["pop"].shift(1)
        prev_collect = trials.groupby(block_col)["collect"].shift(1)

        trials["risk_after_explosion"] = np.where(prev_pop == 1, trials["pump_count"] - trials["prev_pumps"], np.nan)
        trials["risk_after_collect"] = np.where(prev_collect == 1, trials["pump_count"] - trials["prev_pumps"], np.nan)

        if "pump_latency_mean" in trials.columns:
            trials["pump_speed_change"] = trials["pump_latency_mean"] - trials.groupby(block_col)["pump_latency_mean"].shift(1)
        else:
            trials["pump_speed_change"] = np.nan
    else:
        if "risk_after_explosion" not in trials.columns:
            trials["risk_after_explosion"] = np.nan
        if "risk_after_collect" not in trials.columns:
            trials["risk_after_collect"] = np.nan
        if "pump_speed_change" not in trials.columns:
            trials["pump_speed_change"] = np.nan

    sesPoints = trials["total_earnings"].dropna().max() if "total_earnings" in trials.columns else np.nan
    exp_N = int(trials["pop"].sum())
    collect_N = int(trials["collect"].sum())
    tN = int(len(trials))
    exp_rate = exp_N / tN if tN else np.nan
    collect_rate = collect_N / tN if tN else np.nan
    bonus_N = int(trials["nf_bonus_awarded_trial"].sum()) if "nf_bonus_awarded_trial" in trials.columns else 0

    smPmp = float(trials["pump_count"].mean()) if "pump_count" in trials.columns else np.nan
    SD_smPmp = float(trials["pump_count"].std(ddof=1)) if "pump_count" in trials.columns else np.nan

    trials["adjusted_pumps_trial"] = trials["pump_count"] if "pump_count" in trials.columns else np.nan
    smADJ_pmp = float(trials["adjusted_pumps_trial"].mean()) if "adjusted_pumps_trial" in trials.columns else np.nan
    SD_smADJ_pmp = float(trials["adjusted_pumps_trial"].std(ddof=1)) if "adjusted_pumps_trial" in trials.columns else np.nan

    mpmpL = float(trials["pump_latency_mean"].mean()) if "pump_latency_mean" in trials.columns else np.nan
    SD_mpmpL = float(trials["pump_latency_mean"].std(ddof=1)) if "pump_latency_mean" in trials.columns else np.nan
    medpmpL = float(trials["pump_latency_median"].mean()) if "pump_latency_median" in trials.columns else np.nan
    SD_medpmpL = float(trials["pump_latency_median"].std(ddof=1)) if "pump_latency_median" in trials.columns else np.nan

    def _get_scalar_from_sources(keys, sources):
        for df in sources:
            if df is None or len(df) == 0:
                continue
            cols_lut = {str(c).strip().lower(): c for c in df.columns}
            for k in keys:
                kk = str(k).strip().lower()
                if kk in cols_lut:
                    col = cols_lut[kk]
                    ser = pd.to_numeric(df[col], errors="coerce")
                    if ser.notna().any():
                        return float(ser.dropna().iloc[0])
        return np.nan

    sources = [trials, summary]

    baseline_cols = {
        "baseline_eo_mu": _get_scalar_from_sources(
            ["baseline_eo_mu", "baseline_pre_eo_mu", "pretask_eob_avgtheta", "pretask_eo_avgtheta"], sources),
        "baseline_ec_mu": _get_scalar_from_sources(
            ["baseline_ec_mu", "baseline_pre_ec_mu", "pretask_ecb_avgtheta", "pretask_ec_avgtheta"], sources),
        "baseline_conc_mu": _get_scalar_from_sources(
            ["baseline_conc_mu", "baseline_pre_conc_mu", "pretask_cb_avgtheta", "pretask_conc_avgtheta"], sources),

        "baseline_post_eo_mu": _get_scalar_from_sources(
            ["baseline_post_eo_mu", "posttask_eob_avgtheta", "post_eob_avgtheta", "posttask_eo_avgtheta"], sources),
        "baseline_post_ec_mu": _get_scalar_from_sources(
            ["baseline_post_ec_mu", "posttask_ecb_avgtheta", "post_ecb_avgtheta", "posttask_ec_avgtheta"], sources),
        "baseline_post_conc_mu": _get_scalar_from_sources(
            ["baseline_post_conc_mu", "posttask_cb_avgtheta", "post_cb_avgtheta", "posttask_conc_avgtheta"], sources),
    }

    pump_rows = []
    for _, tr in trials.iterrows():
        n = int(tr["pump_count"]) if pd.notna(tr.get("pump_count", np.nan)) else 0
        if n <= 0:
            continue

        lat_list = safe_json_list(tr.get("pump_latencies_json", None))

        for p in range(1, n + 1):
            row = {}

            row["pID"] = normalize_pid_series(pd.Series([bits["sub"]]))[0]
            row["cond"] = bits["condition"]
            row["sID"] = bits["ses"]
            row["ses_num"] = bits["ses_num"]

            row["pPoints"] = np.nan
            row["pgsi_score"] = np.nan
            row["gender"] = np.nan

            row["sesPoints"] = sesPoints
            row["exp_N"] = exp_N
            row["collect_N"] = collect_N
            row["bonus_N"] = bonus_N
            row["exp_rate"] = exp_rate
            row["collect_rate"] = collect_rate
            row["smPmp"] = smPmp
            row["SD_smPmp"] = SD_smPmp
            row["smADJ_pmp"] = smADJ_pmp
            row["SD_smADJ_pmp"] = SD_smADJ_pmp
            row["mpmpL"] = mpmpL
            row["SD_mpmpL"] = SD_mpmpL
            row["medpmpL"] = medpmpL
            row["SD_medpmpL"] = SD_medpmpL

            row["pretask_EOB_avgTheta"]  = baseline_cols.get("baseline_eo_mu", np.nan)
            row["posttask_EOB_avgTheta"] = baseline_cols.get("baseline_post_eo_mu", np.nan)
            row["pretask_ECB_avgTheta"]  = baseline_cols.get("baseline_ec_mu", np.nan)
            row["posttask_ECB_avgTheta"] = baseline_cols.get("baseline_post_ec_mu", np.nan)
            row["pretask_CB_avgTheta"]   = baseline_cols.get("baseline_conc_mu", np.nan)
            row["posttask_CB_avgTheta"]  = baseline_cols.get("baseline_post_conc_mu", np.nan)

            row["tID"] = int(tr["trial"]) if "trial" in tr and pd.notna(tr["trial"]) else np.nan
            row["pop"] = int(tr["pop"])
            row["collect"] = int(tr["collect"])
            row["pmp_count"] = int(tr["pump_count"]) if pd.notna(tr.get("pump_count", np.nan)) else np.nan
            row["t_earned"] = float(tr["trial_earnings"]) if "trial_earnings" in tr and pd.notna(tr["trial_earnings"]) else np.nan
            row["t_loss"] = float(tr["loss_if_pop"]) if "loss_if_pop" in tr and pd.notna(tr["loss_if_pop"]) else np.nan

            row["collectL_trial"] = float(tr["collect_latency_from_trial_start"]) if "collect_latency_from_trial_start" in tr and pd.notna(tr["collect_latency_from_trial_start"]) else np.nan
            row["collectL_ready"] = float(tr["collect_latency_from_ready"]) if "collect_latency_from_ready" in tr and pd.notna(tr["collect_latency_from_ready"]) else np.nan

            row["nf_high_fraction"] = float(tr["nf_high_fraction"]) if "nf_high_fraction" in tr and pd.notna(tr["nf_high_fraction"]) else np.nan
            row["nf_green_success"] = int(tr["nf_green_success"]) if "nf_green_success" in tr and pd.notna(tr["nf_green_success"]) else np.nan

            row["risk_after_explosion"] = float(tr.get("risk_after_explosion", np.nan)) if pd.notna(tr.get("risk_after_explosion", np.nan)) else np.nan
            row["risk_after_collect"] = float(tr.get("risk_after_collect", np.nan)) if pd.notna(tr.get("risk_after_collect", np.nan)) else np.nan
            row["pump_speed_change"] = float(tr.get("pump_speed_change", np.nan)) if pd.notna(tr.get("pump_speed_change", np.nan)) else np.nan

            row["pump_number"] = int(p)
            row["pmpID"] = f"{bits['sub']}_{bits['ses']}_T{int(row['tID'])}_P{p}" if pd.notna(row["tID"]) else f"{bits['sub']}_{bits['ses']}_P{p}"
            row["pmpL"] = float(lat_list[p-1]) if isinstance(lat_list, list) and len(lat_list) >= p else np.nan

            row["ses_mTheta"] = np.nan
            row["ses_SD_mThetaB"] = np.nan
            row["ses_avgP300_All"] = np.nan
            row["ses_avgP300_Fz"] = np.nan
            row["ses_avgP300_FCz"] = np.nan
            row["ses_avgP300_Cz"] = np.nan
            row["ses_avgP300_Pz"] = np.nan

            row["pre200_pop_Theta"] = np.nan
            row["post200_pop_Theta"] = np.nan
            row["pre200_collect_Theta"] = np.nan
            row["post200_collect_Theta"] = np.nan
            row["pre200_pmp_Theta"] = np.nan

            row["theta_before_pump"] = np.nan
            row["theta_before_pump_Fp1"] = np.nan
            row["theta_before_pump_Fp2"] = np.nan
            row["frontal_asymmetry"] = np.nan

            row["trial_mTheta"] = np.nan
            row["trial_SD_mThetaB"] = np.nan

            pump_rows.append(row)

    pump_df = pd.DataFrame(pump_rows)
    return pump_df, trials, summary

# ----------------------------
# Pair processing
# ----------------------------
def process_pair(beh_fp: Path, xdf_fp: Path | None):
    pump_df, trials_df, summary_df = beh_to_pump_table(beh_fp)
    if pump_df is None or len(pump_df) == 0:
        return None

    if xdf_fp is None:
        return pump_df

    try:
        streams, _ = pyxdf.load_xdf(str(xdf_fp))
        eeg_stream, marker_stream = pick_streams(streams)
        if eeg_stream is None or marker_stream is None:
            return pump_df

        raw, eeg_t = build_raw_from_xdf_eeg(eeg_stream)
        raw = preprocess_raw(raw)
        raw.resample(TARGET_SFREQ, verbose=False)
        eeg_t0 = float(eeg_t[0])

        md = markers_to_df(marker_stream)

        ses_mTheta, ses_SD_mThetaB = extract_session_theta_metrics(raw)
        p300 = extract_p300_metrics(raw, md, eeg_t0)
        ev_theta = extract_event_theta_windows(raw, md, eeg_t0)

        pump_theta = theta_before_each_pump(raw, md, eeg_t0)
        trial_theta = trial_theta_from_markers(raw, md, eeg_t0)

        pump_df["ses_mTheta"] = ses_mTheta
        pump_df["ses_SD_mThetaB"] = ses_SD_mThetaB
        for k, v in p300.items():
            pump_df[k] = v
        for k, v in ev_theta.items():
            pump_df[k] = v

        pump_df["tID"] = pd.to_numeric(pump_df["tID"], errors="coerce").astype("Int64")
        pump_df["pump_number"] = pd.to_numeric(pump_df["pump_number"], errors="coerce").astype("Int64")

        if trial_theta is not None and len(trial_theta) > 0:
            tt = trial_theta.copy()
            tt["tID"] = pd.to_numeric(tt["trial"], errors="coerce").astype("Int64")
            tt = tt.drop(columns=["trial"], errors="ignore")

            for c in ["trial_mTheta", "trial_SD_mThetaB"]:
                if c in pump_df.columns:
                    pump_df = pump_df.drop(columns=[c])

            pump_df = pump_df.merge(tt, on="tID", how="left")

        if pump_theta is not None and len(pump_theta) > 0:
            pt = pump_theta.copy()
            pt["tID"] = pd.to_numeric(pt["trial"], errors="coerce").astype("Int64")
            pt["pump_number"] = pd.to_numeric(pt["pump_number"], errors="coerce").astype("Int64")
            pt = pt.drop(columns=["trial"], errors="ignore")

            for c in ["theta_before_pump", "theta_before_pump_Fp1", "theta_before_pump_Fp2", "frontal_asymmetry"]:
                if c in pump_df.columns:
                    pump_df = pump_df.drop(columns=[c])

            pump_df = pump_df.merge(pt, on=["tID", "pump_number"], how="left")

        for c in ["trial_mTheta","trial_SD_mThetaB","theta_before_pump","theta_before_pump_Fp1","theta_before_pump_Fp2","frontal_asymmetry"]:
            if c not in pump_df.columns:
                pump_df[c] = np.nan

        return pump_df

    except Exception as e:
        print("⚠️ XDF failed:", xdf_fp.name, "->", e)
        return pump_df

# ----------------------------
# Main: find files + build master
# ----------------------------
beh_files = sorted(ROOT.rglob("*_beh.xlsx"))
xdf_files = sorted(ROOT.rglob("*.xdf")) + sorted(ROOT.rglob("*.xdfz"))

print("Found BEH files:", len(beh_files))
print("Found XDF files:", len(xdf_files))

xdf_map = {}
for xf in xdf_files:
    key = xf.name.split("_task")[0]
    xdf_map.setdefault(key, []).append(xf)

all_rows = []
missing_xdf = 0
processed = 0

for bf in beh_files:
    key = bf.name.split("_task")[0]
    xmatch = xdf_map.get(key, [])
    xdf_fp = xmatch[0] if len(xmatch) else None
    if xdf_fp is None:
        missing_xdf += 1

    print("\n---")
    print("BEH:", bf.name)
    print("XDF:", xdf_fp.name if xdf_fp else "MISSING (BEH-only)")

    out = process_pair(bf, xdf_fp)
    if out is not None and len(out) > 0:
        all_rows.append(out)
        processed += 1
        print("Rows:", out.shape)

if len(all_rows) == 0:
    raise RuntimeError("No output rows produced. Check BEH files contain Trials + pump_count > 0.")

master = pd.concat(all_rows, ignore_index=True)
print("\n✅ sessions processed:", processed)
print("⚠️ missing XDF sessions:", missing_xdf)
print("✅ master shape:", master.shape)

# ----------------------------
# Participant total points over all sessions (pPoints)
# ----------------------------
if "pID" in master.columns and "sesPoints" in master.columns and "sID" in master.columns:
    master["pID"] = normalize_pid_series(master["pID"])
    tmp = master[["pID","sID","sesPoints"]].drop_duplicates()
    totals = tmp.groupby("pID")["sesPoints"].sum()
    master["pPoints"] = master["pID"].map(totals)

# ----------------------------
# Build participant-level survey table from session 5 BEH files
# ----------------------------
survey_rows = []

for bf in beh_files:
    bits = parse_bids_bits(bf.name)
    if pd.isna(bits["ses_num"]) or int(bits["ses_num"]) != 5:
        continue

    sheets = read_beh_sheets(bf)
    survey_df = get_sheet(sheets, ["survey", "surveys"])
    if survey_df is None or len(survey_df) == 0:
        continue

    participant_id = normalize_pid_series(pd.Series([bits["sub"]]))[0]
    survey_row = extract_survey_row(survey_df, participant_id)
    if survey_row is not None:
        survey_rows.append(survey_row)

survey_master = pd.DataFrame(survey_rows) if len(survey_rows) else pd.DataFrame(columns=["pID"])

if len(survey_master):
    survey_master["pID"] = normalize_pid_series(survey_master["pID"])
    survey_master = survey_master.drop_duplicates(subset=["pID"], keep="first")
    print("Survey rows loaded:", survey_master.shape)
else:
    print("No survey rows found.")

# ----------------------------
# Merge PGSI + manifest fields
# ----------------------------
if manifest is not None and "pID" in manifest.columns:
    master["pID"] = normalize_pid_series(master["pID"])
    manifest["pID"] = normalize_pid_series(manifest["pID"])

    keep_manifest_cols = [c for c in manifest.columns if c in ["pID", "pgsi_score", "risk_group", "cond_manifest", "gender"]]
    manifest_small = manifest[keep_manifest_cols].drop_duplicates(subset=["pID"], keep="first")

    master_ids = set(master["pID"].dropna().unique())
    manifest_ids = set(manifest_small["pID"].dropna().unique())
    overlap = sorted(master_ids.intersection(manifest_ids))

    print("Master unique pIDs:", len(master_ids))
    print("Manifest unique pIDs:", len(manifest_ids))
    print("Overlapping pIDs:", len(overlap))
    print("First overlapping IDs:", overlap[:10])

    master = master.merge(manifest_small, on="pID", how="left")

    if "pgsi_score" in master.columns:
        print("Non-missing pgsi_score rows after merge:", int(master["pgsi_score"].notna().sum()))
        print(master[["pID", "pgsi_score"]].dropna().drop_duplicates().head(10))

# ----------------------------
# Merge survey responses from session 5
# ----------------------------
if len(survey_master):
    master = master.merge(survey_master, on="pID", how="left")

# ----------------------------
# Ensure final schema columns exist
# ----------------------------
schema_cols = [
    "pID","cond","pPoints","pgsi_score","risk_group","gender","sID","ses_num",
    "sesPoints","ses_mTheta","ses_SD_mThetaB",
    "ses_avgP300_All","ses_avgP300_Fz","ses_avgP300_FCz","ses_avgP300_Cz","ses_avgP300_Pz",
    "smPmp","SD_smPmp","smADJ_pmp","SD_smADJ_pmp",
    "exp_rate","exp_N","bonus_N","collect_N","collect_rate",
    "pretask_EOB_avgTheta","posttask_EOB_avgTheta",
    "pretask_ECB_avgTheta","posttask_ECB_avgTheta",
    "pretask_CB_avgTheta","posttask_CB_avgTheta",
    "tID","trial_mTheta","trial_SD_mThetaB",
    "pop","collect","nf_high_fraction","nf_green_success","t_earned","t_loss",
    "collectL_trial","collectL_ready",
    "pmp_count",
    "pre200_pop_Theta","post200_pop_Theta","pre200_collect_Theta","post200_collect_Theta","pre200_pmp_Theta",
    "risk_after_explosion","risk_after_collect","pump_speed_change",
    "pump_number","pmpID","pmpL",
    "mpmpL","SD_mpmpL","medpmpL","SD_medpmpL",
    "theta_before_pump","theta_before_pump_Fp1","theta_before_pump_Fp2","frontal_asymmetry",
]

for c in schema_cols:
    if c not in master.columns:
        master[c] = np.nan

# Reorder: pID, then survey columns, then schema, then rest
survey_cols = [c for c in master.columns if str(c).lower().startswith("survey_")]

base_order = [c for c in schema_cols if c in master.columns and c != "pID"]
remaining = [c for c in master.columns if c not in (["pID"] + survey_cols + base_order)]

ordered = ["pID"] + survey_cols + base_order + remaining
master = master[ordered]

# Quick checks
print("Columns containing 'pgsi':", [c for c in master.columns if "pgsi" in str(c).lower()])
print("Survey columns found:", len([c for c in master.columns if str(c).lower().startswith("survey_")]))
if "pgsi_score" in master.columns:
    vals = master["pgsi_score"].dropna().unique().tolist()
    try:
        vals = sorted(vals)[:20]
    except Exception:
        vals = vals[:20]
    print("Unique non-missing PGSI values:", vals)

# Save
csv_path = OUT / "BART_master_pump_level.csv"
xlsx_path = OUT / "BART_master_pump_level.xlsx"
master.to_csv(csv_path, index=False)
master.to_excel(xlsx_path, index=False)

print("\n✅ Saved CSV:", csv_path)
print("✅ Saved XLSX:", xlsx_path)

master.head(10)

Loaded manifest: /content/manifest.xlsx
Using manifest sheet: Cohort 1
Manifest columns: ['pID', 'pgsi_score', 'cond_manifest', 'risk_group', 'gender']
Manifest non-missing PGSI rows: 48
     pID  pgsi_score
0   P001         3.0
1   P002         5.0
2   P003         4.0
3   P004         3.0
4   P006         5.0
5   P007         7.0
6   P008         3.0
7   P009         3.0
8   P010         3.0
11  P005         2.0
Found BEH files: 90
Found XDF files: 84

---
BEH: sub-P001_ses-S001_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P001_ses-S001_task-Default_run-001_eeg.xdf
Rows: (536, 62)

---
BEH: sub-P001_ses-S002_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P001_ses-S002_task-Default_run-001_eeg.xdf
Rows: (691, 62)

---
BEH: sub-P001_ses-S003_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P001_ses-S003_task-Default_run-001_eeg.xdf
Rows: (688, 62)

---
BEH: sub-P001_ses-S004_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: MISSING (BEH-only)
Rows: (562, 62)

---
BEH: sub-P001_ses-S00

Rows: (786, 62)

---
BEH: sub-P005_ses-S001_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P005_ses-S001_task-Default_run-001_eeg.xdf
Rows: (633, 62)

---
BEH: sub-P005_ses-S002_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P005_ses-S002_task-Default_run-001_eeg.xdf
Rows: (580, 62)

---
BEH: sub-P005_ses-S003_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P005_ses-S003_task-Default_run-001_eeg.xdf
Rows: (618, 62)

---
BEH: sub-P005_ses-S004_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P005_ses-S004_task-Default_run-001_eeg.xdf
Rows: (621, 62)

---
BEH: sub-P005_ses-S005_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P005_ses-S005_task-Default_run-001_eeg.xdf
Rows: (574, 62)

---
BEH: sub-P006_ses-S001_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P006_ses-S001_task-Default_run-001_eeg.xdf
Rows: (507, 62)

---
BEH: sub-P006_ses-S002_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P006_ses-S002_task-Default_run-001_eeg.xdf
Rows: (527, 62)

---
BEH: sub-P006_ses-S003_task

Rows: (760, 62)

---
BEH: sub-P018_ses-S004_task-BART_cond-nf_sham_run-001_beh.xlsx
XDF: sub-P018_ses-S004_task-Default_run-001_eeg.xdf
Rows: (646, 62)

---
BEH: sub-P018_ses-S005_task-BART_cond-nf_sham_run-001_beh.xlsx
XDF: sub-P018_ses-S005_task-Default_run-001_eeg.xdf
Rows: (807, 62)

---
BEH: sub-P019_ses-S001_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P019_ses-S001_task-Default_run-001_eeg.xdf
Rows: (618, 62)

---
BEH: sub-P019_ses-S002_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P019_ses-S002_task-Default_run-001_eeg.xdf
Rows: (585, 62)

---
BEH: sub-P019_ses-S003_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P019_ses-S003_task-Default_run-001_eeg.xdf
Rows: (515, 62)

---
BEH: sub-P019_ses-S004_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P019_ses-S004_task-Default_run-001_eeg.xdf
Rows: (534, 62)

---
BEH: sub-P019_ses-S005_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P019_ses-S005_task-Default_run-001_eeg.xdf
Rows: (552, 62)

✅ sessions processed: 90
⚠️ mis

/tmp/ipykernel_323/3187548885.py:927: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  manifest["pID"] = normalize_pid_series(manifest["pID"])


Columns containing 'pgsi': ['pgsi_score', 'pgsi_score_x', 'pgsi_score_y']
Survey columns found: 190
Unique non-missing PGSI values: []

✅ Saved CSV: /content/outputs/BART_master_pump_level.csv
✅ Saved XLSX: /content/outputs/BART_master_pump_level.xlsx


,pID,survey_r1_sub,survey_r1_ses,survey_r1_run,survey_r1_task,survey_r1_condition,survey_r1_qid,survey_r1_section,survey_r1_question,survey_r1_response_1to5,...,theta_before_pump_Fp1,theta_before_pump_Fp2,frontal_asymmetry,pgsi_score_x,risk_group_x,gender_x,pgsi_score_y,cond_manifest,risk_group_y,gender_y
0,P001,P001,S005,1,BART,nf_real,S1_Q1,Stress,"Overall, how stressful did you find the tasks ...",1,...,2.317708e-11,3.801747e-11,0.478605,NaN,NaN,NaN,3.0,NF,H,F
1,P001,P001,S005,1,BART,nf_real,S1_Q1,Stress,"Overall, how stressful did you find the tasks ...",1,...,3.875262e-11,4.269164e-11,0.094481,NaN,NaN,NaN,3.0,NF,H,F
2,P001,P001,S005,1,BART,nf_real,S1_Q1,Stress,"Overall, how stressful did you find the tasks ...",1,...,2.006399e-11,5.438994e-11,0.966833,NaN,NaN,NaN,3.0,NF,H,F
3,P001,P001,S005,1,BART,nf_real,S1_Q1,Stress,"Overall, how stressful did you find the tasks ...",1,...,9.258605e-11,1.432722e-10,0.432821,NaN,NaN,NaN,3.0,NF,H,F
4,P001,P001,S005,1,BART,nf_real,S1_Q1,Stress,"Overall, how stressful did you find the tasks ...",1,...,1.047581e-11,1.462728e-11,0.308777,NaN,NaN,NaN,3.0,NF,H,F
5,P001,P001,S005,1,BART,nf_real,S1_Q1,Stress,"Overall, how stressful did you find the tasks ...",1,...,4.157826e-11,4.647099e-11,0.108775,NaN,NaN,NaN,3.0,NF,H,F
6,P001,P001,S005,1,BART,nf_real,S1_Q1,Stress,"Overall, how stressful did you find the tasks ...",1,...,1.502712e-10,1.646237e-10,0.090644,NaN,NaN,NaN,3.0,NF,H,F
7,P001,P001,S005,1,BART,nf_real,S1_Q1,Stress,"Overall, how stressful did you find the tasks ...",1,...,1.371341e-11,6.212980e-12,-0.712877,NaN,NaN,NaN,3.0,NF,H,F
8,P001,P001,S005,1,BART,nf_real,S1_Q1,Stress,"Overall, how stressful did you find the tasks ...",1,...,4.098451e-11,5.847166e-11,0.348199,NaN,NaN,NaN,3.0,NF,H,F
9,P001,P001,S005,1,BART,nf_real,S1_Q1,Stress,"Overall, how stressful did you find the tasks ...",1,...,2.838379e-11,3.433010e-11,0.184292,NaN,NaN,NaN,3.0,NF,H,F


In [ ]:
# ============================================================
# BART MULTIMODAL MASTER PIPELINE (Pump-level)
# Input: *_beh.xlsx + matching *.xdf/*.xdfz + manifest.xlsx
# Output: outputs/BART_master_pump_level.{csv,xlsx}
#
# Includes:
#  - Behavioral pump/trial/session metrics
#  - EEG theta + P300 metrics from XDF
#  - risk_group (H/L) from manifest.xlsx column "high/low"
#  - Survey responses from session 5 BEH survey sheet
# ============================================================

!pip -q install pyxdf mne openpyxl pandas numpy

import re, json
from pathlib import Path
import numpy as np
import pandas as pd
import pyxdf
import mne

# ----------------------------
# CONFIG
# ----------------------------
ROOT = Path("/content") if Path("/content").exists() else Path(".")
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)

TARGET_SFREQ = 512.0
ERP_TMIN, ERP_TMAX = -0.2, 0.8
ERP_BASELINE = (ERP_TMIN, 0.0)
P3_WIN = (0.300, 0.450)
WIN200 = 0.200
THETA_BAND = (4.0, 7.0)

EEG_ORDER = ["Fz","Cz","Pz","POz","Fp1","Fp2","F3","F4","FCz","C3","C4","CPz","P3","P4","O1","O2"]
FRONTAL_THETA_CHS = ["Fp1","Fp2","Fz","F3","F4","FCz"]

# ----------------------------
# ID normalization
# ----------------------------
def normalize_pid_series(s):
    s = s.astype(str).str.strip().str.upper()
    nums = s.str.extract(r"(\d+)")[0]
    out = nums.fillna("").apply(lambda x: f"P{int(x):03d}" if x != "" else np.nan)
    return out.where(out.notna(), s)

# ----------------------------
# Manifest: high/low only
# ----------------------------
manifest = None
manifest_path = next(iter(ROOT.rglob("manifest.xlsx")), None)

if manifest_path is not None and manifest_path.exists():
    manifest_raw = pd.read_excel(manifest_path)
    manifest_raw.columns = [str(c).strip() for c in manifest_raw.columns]

    cols_l = {str(c).strip().lower(): c for c in manifest_raw.columns}
    pid_col = None
    risk_col = None

    for k in ["subject_id", "participant_id", "participant", "subject", "sub", "pid", "id"]:
        if k in cols_l:
            pid_col = cols_l[k]
            break

    for k in ["high/low", "risk_group", "risk", "group"]:
        if k in cols_l:
            risk_col = cols_l[k]
            break

    if pid_col is not None and risk_col is not None:
        manifest = manifest_raw[[pid_col, risk_col]].copy()
        manifest.columns = ["pID", "risk_group"]
        manifest["pID"] = normalize_pid_series(manifest["pID"])
        manifest["risk_group"] = manifest["risk_group"].astype(str).str.strip().str.upper()
        manifest = manifest.drop_duplicates(subset=["pID"], keep="first")
        print("Loaded manifest:", manifest_path)
        print("Manifest rows:", manifest.shape)
        print(manifest.head())
    else:
        print("manifest.xlsx found, but could not detect subject_id and/or high/low columns.")
else:
    print("No manifest.xlsx found.")

# ----------------------------
# Utilities
# ----------------------------
def parse_bids_bits(name: str):
    base = Path(name).name
    sub = re.search(r"sub-([A-Za-z0-9]+)", base)
    ses = re.search(r"ses-([A-Za-z0-9]+)", base)
    run = re.search(r"run-([0-9]+)", base)
    task = re.search(r"task-([A-Za-z0-9]+)", base)
    cond = re.search(r"cond-([A-Za-z0-9_]+)", base)

    out = {
        "sub": sub.group(1) if sub else None,
        "ses": ses.group(1) if ses else None,
        "run": run.group(1) if run else None,
        "task": task.group(1) if task else None,
        "condition": cond.group(1) if cond else None,
    }
    if out["ses"] is not None:
        m = re.search(r"(\d+)", out["ses"])
        out["ses_num"] = int(m.group(1)) if m else np.nan
    else:
        out["ses_num"] = np.nan
    return out

def read_beh_sheets(fp: Path):
    xls = pd.ExcelFile(fp)
    sheets = {}
    for sh in xls.sheet_names:
        df = pd.read_excel(fp, sheet_name=sh)
        df.columns = [str(c).strip() for c in df.columns]
        sheets[sh.lower()] = df
    return sheets

def get_sheet(sheets: dict, candidates):
    for c in candidates:
        if c in sheets:
            return sheets[c]
    return None

def safe_json_list(x):
    try:
        return json.loads(x) if isinstance(x, str) else (x if isinstance(x, list) else None)
    except Exception:
        return None

def extract_survey_row(survey_df, participant_id):
    if survey_df is None or len(survey_df) == 0:
        return None

    df = survey_df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    participant_id = normalize_pid_series(pd.Series([participant_id])).iloc[0]

    if len(df) == 1:
        row = df.iloc[0].to_dict()
        row["pID"] = participant_id
        return row

    cols_l = {str(c).strip().lower(): c for c in df.columns}
    q_col = None
    r_col = None

    for k in ["question", "item", "prompt"]:
        if k in cols_l:
            q_col = cols_l[k]
            break
    for k in ["response", "answer", "value", "score"]:
        if k in cols_l:
            r_col = cols_l[k]
            break

    if q_col is not None and r_col is not None:
        out = {"pID": participant_id}
        for _, r in df.iterrows():
            q = str(r[q_col]).strip()
            q_clean = re.sub(r"[^A-Za-z0-9]+", "_", q).strip("_").lower()
            out[f"survey_{q_clean}"] = r[r_col]
        return out

    out = {"pID": participant_id}
    for i, (_, r) in enumerate(df.iterrows(), start=1):
        for c in df.columns:
            cname = re.sub(r"[^A-Za-z0-9]+", "_", str(c)).strip("_").lower()
            out[f"survey_r{i}_{cname}"] = r[c]
    return out

# ----------------------------
# XDF helpers
# ----------------------------
def parse_marker_message(msg):
    s = str(msg)
    parts = s.split(";")
    out = {"code": parts[0].strip()}
    for p in parts[1:]:
        if "=" in p:
            k, v = p.split("=", 1)
            out[k.strip()] = v.strip()
    return out

def _stream_name_lower(s):
    info = s.get("info", {})
    nm = info.get("name", "")
    if isinstance(nm, (list, tuple, np.ndarray)):
        nm = nm[0] if len(nm) else ""
    return str(nm).strip().lower()

def pick_streams(streams):
    eeg = None
    best = -1

    for s in streams:
        ts = s.get("time_series", None)
        if ts is None:
            continue
        arr = np.asarray(ts)
        ch = 1 if arr.ndim == 1 else arr.shape[1]
        if ch > best:
            eeg = s
            best = ch

    markers = None
    for s in streams:
        ts = s.get("time_series", None)
        if ts is None:
            continue
        arr = np.asarray(ts)
        ch = 1 if arr.ndim == 1 else arr.shape[1]
        if ch != 1:
            continue
        nm = _stream_name_lower(s)
        if ("bart" in nm) or ("marker" in nm) or ("markers" in nm) or ("event" in nm):
            markers = s
            break

    if markers is None:
        for s in streams:
            ts = s.get("time_series", None)
            if ts is None:
                continue
            arr = np.asarray(ts)
            ch = 1 if arr.ndim == 1 else arr.shape[1]
            if ch == 1:
                markers = s
                break

    return eeg, markers

def markers_to_df(marker_stream):
    m_ts = np.asarray(marker_stream["time_stamps"], dtype=float)
    m_val = np.asarray(marker_stream["time_series"], dtype=object)

    rows = []
    for t, v in zip(m_ts, m_val):
        msg = v[0] if isinstance(v, (list, tuple, np.ndarray)) else v
        d = parse_marker_message(msg)
        d["lsltime"] = float(t)
        rows.append(d)

    df = pd.DataFrame(rows)
    for k in ["trial", "pump", "loss", "earnings", "total"]:
        if k in df.columns:
            df[k] = pd.to_numeric(df[k], errors="coerce")
    return df

def build_raw_from_xdf_eeg(eeg_stream):
    data = np.asarray(eeg_stream["time_series"])
    t = np.asarray(eeg_stream["time_stamps"], dtype=float)
    if data.ndim == 1:
        data = data[:, None]
    n_samp, n_ch = data.shape

    sfreq_est = 1.0 / np.median(np.diff(t))
    ch_names = EEG_ORDER[:n_ch] if n_ch <= len(EEG_ORDER) else [f"EEG{ii+1:02d}" for ii in range(n_ch)]
    arr = data.T.astype(float)

    peak = float(np.nanmax(np.abs(arr)))
    if peak > 1e3:
        arr *= 1e-6
    elif peak > 5.0:
        arr *= 1e-3

    info = mne.create_info(ch_names=ch_names, sfreq=sfreq_est, ch_types=["eeg"] * n_ch)
    raw = mne.io.RawArray(arr, info, verbose=False)
    return raw, t

def preprocess_raw(raw):
    raw = raw.copy()
    raw.filter(0.1, 30.0, picks="eeg", verbose=False)
    try:
        raw.notch_filter([60], picks="eeg", verbose=False)
    except Exception:
        pass
    try:
        raw.set_eeg_reference("average", verbose=False)
    except Exception:
        pass
    return raw

def lsltime_to_sample(lsltime, eeg_t0, sfreq):
    return int(np.round((lsltime - eeg_t0) * sfreq))

# ----------------------------
# EEG feature extraction
# ----------------------------
def compute_theta_power_window(raw_theta, sample, tmin_s, tmax_s, picks):
    sf = raw_theta.info["sfreq"]
    start = int(sample + np.round(tmin_s * sf))
    stop  = int(sample + np.round(tmax_s * sf))
    start = max(start, 0)
    stop  = min(stop, raw_theta.n_times)
    if stop <= start + 2:
        return np.nan
    seg = raw_theta.get_data(picks=picks, start=start, stop=stop)
    return float(np.mean(seg**2))

def extract_session_theta_metrics(raw):
    rtheta = raw.copy().filter(THETA_BAND[0], THETA_BAND[1], picks="eeg", verbose=False)
    picks = [raw.ch_names.index(ch) for ch in FRONTAL_THETA_CHS if ch in raw.ch_names]
    if len(picks) == 0:
        return np.nan, np.nan
    X = rtheta.get_data(picks=picks)
    theta_ts = np.mean(X**2, axis=0)
    return float(np.mean(theta_ts)), float(np.std(theta_ts, ddof=1))

def extract_p300_metrics(raw, md, eeg_t0):
    keys = ["ses_avgP300_All","ses_avgP300_Fz","ses_avgP300_FCz","ses_avgP300_Cz","ses_avgP300_Pz"]
    if "code" not in md.columns:
        return {k: np.nan for k in keys}

    exp = md[md["code"].astype(str).str.upper().eq("BART_EXPLODE")].copy()
    if len(exp) == 0:
        return {k: np.nan for k in keys}

    sf = raw.info["sfreq"]
    exp = exp.dropna(subset=["lsltime"]).copy()
    exp["sample"] = exp["lsltime"].apply(lambda t: lsltime_to_sample(float(t), eeg_t0, sf)).astype(int)

    min_samp = int(np.ceil((-ERP_TMIN) * sf))
    exp = exp[(exp["sample"] >= min_samp) & (exp["sample"] < raw.n_times)].copy()
    if len(exp) == 0:
        return {k: np.nan for k in keys}

    events = np.c_[exp["sample"].to_numpy(), np.zeros(len(exp), int), np.ones(len(exp), int)]
    epochs = mne.Epochs(
        raw, events, event_id={"EXPLODE": 1},
        tmin=ERP_TMIN, tmax=ERP_TMAX, baseline=ERP_BASELINE,
        preload=True, reject_by_annotation=True, verbose=False
    )
    if len(epochs) == 0:
        return {k: np.nan for k in keys}

    ev = epochs.average()
    times = ev.times
    m1 = np.argmin(np.abs(times - P3_WIN[0]))
    m2 = np.argmin(np.abs(times - P3_WIN[1]))

    out = {}
    vals = []
    for ch in ["Fz","FCz","Cz","Pz"]:
        k = f"ses_avgP300_{ch}"
        if ch in ev.ch_names:
            idx = ev.ch_names.index(ch)
            erp_uV = ev.data[idx] * 1e6
            p3 = float(np.max(erp_uV[m1:m2]))
            out[k] = p3
            vals.append(p3)
        else:
            out[k] = np.nan
    out["ses_avgP300_All"] = float(np.nanmean(vals)) if len(vals) else np.nan
    return out

def extract_event_theta_windows(raw, md, eeg_t0):
    rtheta = raw.copy().filter(THETA_BAND[0], THETA_BAND[1], picks="eeg", verbose=False)
    sf = raw.info["sfreq"]
    picks = [raw.ch_names.index(ch) for ch in FRONTAL_THETA_CHS if ch in raw.ch_names]
    if len(picks) == 0:
        return {
            "pre200_pop_Theta": np.nan, "post200_pop_Theta": np.nan,
            "pre200_collect_Theta": np.nan, "post200_collect_Theta": np.nan,
            "pre200_pmp_Theta": np.nan,
        }

    def _mean_prepost(code, pre=True):
        d = md[md["code"].astype(str).str.upper().eq(code)].dropna(subset=["lsltime"]).copy()
        if len(d) == 0:
            return np.nan
        d["sample"] = d["lsltime"].apply(lambda t: lsltime_to_sample(float(t), eeg_t0, sf)).astype(int)
        vals = []
        for s in d["sample"].tolist():
            v = compute_theta_power_window(rtheta, s, -WIN200, 0.0, picks) if pre else compute_theta_power_window(rtheta, s, 0.0, WIN200, picks)
            vals.append(v)
        return float(np.nanmean(vals)) if len(vals) else np.nan

    return {
        "pre200_pop_Theta": _mean_prepost("BART_EXPLODE", pre=True),
        "post200_pop_Theta": _mean_prepost("BART_EXPLODE", pre=False),
        "pre200_collect_Theta": _mean_prepost("BART_COLLECT", pre=True),
        "post200_collect_Theta": _mean_prepost("BART_COLLECT", pre=False),
        "pre200_pmp_Theta": _mean_prepost("BART_PUMP", pre=True),
    }

def theta_before_each_pump(raw, md, eeg_t0):
    pumps = md[md["code"].astype(str).str.upper().eq("BART_PUMP")].copy()
    if len(pumps) == 0 or ("trial" not in pumps.columns) or ("pump" not in pumps.columns):
        return pd.DataFrame(columns=["trial","pump_number","theta_before_pump","theta_before_pump_Fp1","theta_before_pump_Fp2","frontal_asymmetry"])

    rtheta = raw.copy().filter(THETA_BAND[0], THETA_BAND[1], picks="eeg", verbose=False)
    sf = raw.info["sfreq"]

    pumps["trial"] = pd.to_numeric(pumps["trial"], errors="coerce")
    pumps["pump_number"] = pd.to_numeric(pumps["pump"], errors="coerce")
    pumps = pumps.dropna(subset=["trial","pump_number","lsltime"]).copy()

    pumps["trial"] = pumps["trial"].astype(int)
    pumps["pump_number"] = pumps["pump_number"].astype(int)
    pumps["sample"] = pumps["lsltime"].apply(lambda t: lsltime_to_sample(float(t), eeg_t0, sf)).astype(int)

    min_samp = int(np.ceil(WIN200 * sf))
    pumps = pumps[(pumps["sample"] >= min_samp) & (pumps["sample"] < raw.n_times)].copy()
    if len(pumps) == 0:
        return pd.DataFrame(columns=["trial","pump_number","theta_before_pump","theta_before_pump_Fp1","theta_before_pump_Fp2","frontal_asymmetry"])

    frontal_picks = [raw.ch_names.index(ch) for ch in FRONTAL_THETA_CHS if ch in raw.ch_names]
    fp1_pick = [raw.ch_names.index("Fp1")] if "Fp1" in raw.ch_names else []
    fp2_pick = [raw.ch_names.index("Fp2")] if "Fp2" in raw.ch_names else []

    rows = []
    for _, r in pumps.iterrows():
        s = int(r["sample"])
        pre_frontal = compute_theta_power_window(rtheta, s, -WIN200, 0.0, frontal_picks) if frontal_picks else np.nan
        pre_fp1 = compute_theta_power_window(rtheta, s, -WIN200, 0.0, fp1_pick) if fp1_pick else np.nan
        pre_fp2 = compute_theta_power_window(rtheta, s, -WIN200, 0.0, fp2_pick) if fp2_pick else np.nan
        fa = (np.log(pre_fp2 + 1e-12) - np.log(pre_fp1 + 1e-12)) if (np.isfinite(pre_fp1) and np.isfinite(pre_fp2)) else np.nan

        rows.append({
            "trial": int(r["trial"]),
            "pump_number": int(r["pump_number"]),
            "theta_before_pump": pre_frontal,
            "theta_before_pump_Fp1": pre_fp1,
            "theta_before_pump_Fp2": pre_fp2,
            "frontal_asymmetry": fa
        })

    return pd.DataFrame(rows)

def trial_theta_from_markers(raw, md, eeg_t0):
    sf = raw.info["sfreq"]
    rtheta = raw.copy().filter(THETA_BAND[0], THETA_BAND[1], picks="eeg", verbose=False)
    picks = [raw.ch_names.index(ch) for ch in FRONTAL_THETA_CHS if ch in raw.ch_names]
    if len(picks) == 0:
        return pd.DataFrame(columns=["trial","trial_mTheta","trial_SD_mThetaB"])

    starts = md[md["code"].astype(str).str.upper().eq("BART_TRIAL_START")].copy()
    ends = md[md["code"].astype(str).str.upper().isin(["BART_COLLECT","BART_EXPLODE"])].copy()
    if "trial" not in starts.columns or "trial" not in ends.columns:
        return pd.DataFrame(columns=["trial","trial_mTheta","trial_SD_mThetaB"])

    starts["trial"] = pd.to_numeric(starts["trial"], errors="coerce")
    ends["trial"] = pd.to_numeric(ends["trial"], errors="coerce")
    starts = starts.dropna(subset=["trial","lsltime"]).copy()
    ends = ends.dropna(subset=["trial","lsltime"]).copy()
    if len(starts) == 0 or len(ends) == 0:
        return pd.DataFrame(columns=["trial","trial_mTheta","trial_SD_mThetaB"])

    out = []
    for tnum, g0 in starts.groupby(starts["trial"].astype(int)):
        t_start = float(g0["lsltime"].min())
        g1 = ends[ends["trial"].astype(int) == int(tnum)]
        if len(g1) == 0:
            continue
        t_end = float(g1["lsltime"].min())
        if t_end <= t_start:
            continue

        s0 = lsltime_to_sample(t_start, eeg_t0, sf)
        s1 = lsltime_to_sample(t_end, eeg_t0, sf)
        s0 = max(s0, 0)
        s1 = min(s1, raw.n_times)
        if s1 <= s0 + int(sf * 0.2):
            continue

        seg = rtheta.get_data(picks=picks, start=s0, stop=s1)
        theta_ts = np.mean(seg**2, axis=0)

        out.append({
            "trial": int(tnum),
            "trial_mTheta": float(np.mean(theta_ts)),
            "trial_SD_mThetaB": float(np.std(theta_ts, ddof=1)),
        })

    return pd.DataFrame(out)

# ----------------------------
# BEH -> pump table
# ----------------------------
def beh_to_pump_table(beh_fp: Path):
    sheets = read_beh_sheets(beh_fp)
    trials = get_sheet(sheets, ["trials", "trial"])
    summary = get_sheet(sheets, ["summary"])

    if trials is None or len(trials) == 0:
        return None, None, None

    trials = trials.copy()
    bits = parse_bids_bits(beh_fp.name)

    if "exploded" in trials.columns:
        trials["pop"] = trials["exploded"].astype(str).str.upper().isin(["TRUE","1","YES"]).astype(int)
    else:
        trials["pop"] = 0

    if "collected" in trials.columns:
        trials["collect"] = pd.to_numeric(trials["collected"], errors="coerce").fillna(0).astype(int)
    else:
        trials["collect"] = trials["events"].astype(str).str.contains("collect", case=False).astype(int) if "events" in trials.columns else 0

    for c in [
        "trial","pump_count","trial_earnings","loss_if_pop","trial_value","total_earnings",
        "pump_latency_mean","pump_latency_median","pump_latency_first",
        "collect_latency_from_ready","collect_latency_from_trial_start",
        "nf_high_frac","nf_green_success",
        "risk_after_explosion","risk_after_collect","pump_speed_change",
        "nf_mean_normalized"
    ]:
        if c in trials.columns:
            trials[c] = pd.to_numeric(trials[c], errors="coerce")

    if "nf_high_frac" in trials.columns:
        trials["nf_high_fraction"] = pd.to_numeric(trials["nf_high_frac"], errors="coerce")
    elif "nf_high_fraction" not in trials.columns:
        trials["nf_high_fraction"] = np.nan

    if "nf_green_success" in trials.columns and trials["nf_green_success"].notna().any():
        trials["nf_green_success"] = pd.to_numeric(trials["nf_green_success"], errors="coerce")
    else:
        trials["nf_green_success"] = np.where(trials["nf_high_fraction"] > 0.75, 1, 0).astype(int) if "nf_high_fraction" in trials.columns else np.nan

    block_col = "block" if "block" in trials.columns else None
    if block_col is not None:
        trials = trials.sort_values([block_col, "trial"], kind="mergesort") if "trial" in trials.columns else trials
        g = trials.groupby(block_col)["nf_green_success"]
        trials["nf_bonus_awarded_trial"] = ((g.shift(0)==1) & (g.shift(1)==1) & (g.shift(2)==1)).astype(int)
    else:
        trials["nf_bonus_awarded_trial"] = 0

    if "pump_count" in trials.columns and block_col is not None and "trial" in trials.columns:
        trials["prev_pumps"] = trials.groupby(block_col)["pump_count"].shift(1)
        prev_pop = trials.groupby(block_col)["pop"].shift(1)
        prev_collect = trials.groupby(block_col)["collect"].shift(1)

        trials["risk_after_explosion"] = np.where(prev_pop == 1, trials["pump_count"] - trials["prev_pumps"], np.nan)
        trials["risk_after_collect"] = np.where(prev_collect == 1, trials["pump_count"] - trials["prev_pumps"], np.nan)

        if "pump_latency_mean" in trials.columns:
            trials["pump_speed_change"] = trials["pump_latency_mean"] - trials.groupby(block_col)["pump_latency_mean"].shift(1)
        else:
            trials["pump_speed_change"] = np.nan
    else:
        if "risk_after_explosion" not in trials.columns:
            trials["risk_after_explosion"] = np.nan
        if "risk_after_collect" not in trials.columns:
            trials["risk_after_collect"] = np.nan
        if "pump_speed_change" not in trials.columns:
            trials["pump_speed_change"] = np.nan

    sesPoints = trials["total_earnings"].dropna().max() if "total_earnings" in trials.columns else np.nan
    exp_N = int(trials["pop"].sum())
    collect_N = int(trials["collect"].sum())
    tN = int(len(trials))
    exp_rate = exp_N / tN if tN else np.nan
    collect_rate = collect_N / tN if tN else np.nan
    bonus_N = int(trials["nf_bonus_awarded_trial"].sum()) if "nf_bonus_awarded_trial" in trials.columns else 0

    smPmp = float(trials["pump_count"].mean()) if "pump_count" in trials.columns else np.nan
    SD_smPmp = float(trials["pump_count"].std(ddof=1)) if "pump_count" in trials.columns else np.nan

    trials["adjusted_pumps_trial"] = trials["pump_count"] if "pump_count" in trials.columns else np.nan
    smADJ_pmp = float(trials["adjusted_pumps_trial"].mean()) if "adjusted_pumps_trial" in trials.columns else np.nan
    SD_smADJ_pmp = float(trials["adjusted_pumps_trial"].std(ddof=1)) if "adjusted_pumps_trial" in trials.columns else np.nan

    mpmpL = float(trials["pump_latency_mean"].mean()) if "pump_latency_mean" in trials.columns else np.nan
    SD_mpmpL = float(trials["pump_latency_mean"].std(ddof=1)) if "pump_latency_mean" in trials.columns else np.nan
    medpmpL = float(trials["pump_latency_median"].mean()) if "pump_latency_median" in trials.columns else np.nan
    SD_medpmpL = float(trials["pump_latency_median"].std(ddof=1)) if "pump_latency_median" in trials.columns else np.nan

    def _get_scalar_from_sources(keys, sources):
        for df in sources:
            if df is None or len(df) == 0:
                continue
            cols_lut = {str(c).strip().lower(): c for c in df.columns}
            for k in keys:
                kk = str(k).strip().lower()
                if kk in cols_lut:
                    col = cols_lut[kk]
                    ser = pd.to_numeric(df[col], errors="coerce")
                    if ser.notna().any():
                        return float(ser.dropna().iloc[0])
        return np.nan

    sources = [trials, summary]

    baseline_cols = {
        "baseline_eo_mu": _get_scalar_from_sources(
            ["baseline_eo_mu", "baseline_pre_eo_mu", "pretask_eob_avgtheta", "pretask_eo_avgtheta"], sources),
        "baseline_ec_mu": _get_scalar_from_sources(
            ["baseline_ec_mu", "baseline_pre_ec_mu", "pretask_ecb_avgtheta", "pretask_ec_avgtheta"], sources),
        "baseline_conc_mu": _get_scalar_from_sources(
            ["baseline_conc_mu", "baseline_pre_conc_mu", "pretask_cb_avgtheta", "pretask_conc_avgtheta"], sources),

        "baseline_post_eo_mu": _get_scalar_from_sources(
            ["baseline_post_eo_mu", "posttask_eob_avgtheta", "post_eob_avgtheta", "posttask_eo_avgtheta"], sources),
        "baseline_post_ec_mu": _get_scalar_from_sources(
            ["baseline_post_ec_mu", "posttask_ecb_avgtheta", "post_ecb_avgtheta", "posttask_ec_avgtheta"], sources),
        "baseline_post_conc_mu": _get_scalar_from_sources(
            ["baseline_post_conc_mu", "posttask_cb_avgtheta", "post_cb_avgtheta", "posttask_conc_avgtheta"], sources),
    }

    pump_rows = []
    for _, tr in trials.iterrows():
        n = int(tr["pump_count"]) if pd.notna(tr.get("pump_count", np.nan)) else 0
        if n <= 0:
            continue

        lat_list = safe_json_list(tr.get("pump_latencies_json", None))

        for p in range(1, n + 1):
            row = {}

            row["pID"] = normalize_pid_series(pd.Series([bits["sub"]])).iloc[0]
            row["cond"] = bits["condition"]
            row["sID"] = bits["ses"]
            row["ses_num"] = bits["ses_num"]

            row["pPoints"] = np.nan

            row["sesPoints"] = sesPoints
            row["exp_N"] = exp_N
            row["collect_N"] = collect_N
            row["bonus_N"] = bonus_N
            row["exp_rate"] = exp_rate
            row["collect_rate"] = collect_rate
            row["smPmp"] = smPmp
            row["SD_smPmp"] = SD_smPmp
            row["smADJ_pmp"] = smADJ_pmp
            row["SD_smADJ_pmp"] = SD_smADJ_pmp
            row["mpmpL"] = mpmpL
            row["SD_mpmpL"] = SD_mpmpL
            row["medpmpL"] = medpmpL
            row["SD_medpmpL"] = SD_medpmpL

            row["pretask_EOB_avgTheta"]  = baseline_cols.get("baseline_eo_mu", np.nan)
            row["posttask_EOB_avgTheta"] = baseline_cols.get("baseline_post_eo_mu", np.nan)
            row["pretask_ECB_avgTheta"]  = baseline_cols.get("baseline_ec_mu", np.nan)
            row["posttask_ECB_avgTheta"] = baseline_cols.get("baseline_post_ec_mu", np.nan)
            row["pretask_CB_avgTheta"]   = baseline_cols.get("baseline_conc_mu", np.nan)
            row["posttask_CB_avgTheta"]  = baseline_cols.get("baseline_post_conc_mu", np.nan)

            row["tID"] = int(tr["trial"]) if "trial" in tr and pd.notna(tr["trial"]) else np.nan
            row["pop"] = int(tr["pop"])
            row["collect"] = int(tr["collect"])
            row["pmp_count"] = int(tr["pump_count"]) if pd.notna(tr.get("pump_count", np.nan)) else np.nan
            row["t_earned"] = float(tr["trial_earnings"]) if "trial_earnings" in tr and pd.notna(tr["trial_earnings"]) else np.nan
            row["t_loss"] = float(tr["loss_if_pop"]) if "loss_if_pop" in tr and pd.notna(tr["loss_if_pop"]) else np.nan

            row["collectL_trial"] = float(tr["collect_latency_from_trial_start"]) if "collect_latency_from_trial_start" in tr and pd.notna(tr["collect_latency_from_trial_start"]) else np.nan
            row["collectL_ready"] = float(tr["collect_latency_from_ready"]) if "collect_latency_from_ready" in tr and pd.notna(tr["collect_latency_from_ready"]) else np.nan

            row["nf_high_fraction"] = float(tr["nf_high_fraction"]) if "nf_high_fraction" in tr and pd.notna(tr["nf_high_fraction"]) else np.nan
            row["nf_green_success"] = int(tr["nf_green_success"]) if "nf_green_success" in tr and pd.notna(tr["nf_green_success"]) else np.nan

            row["risk_after_explosion"] = float(tr.get("risk_after_explosion", np.nan)) if pd.notna(tr.get("risk_after_explosion", np.nan)) else np.nan
            row["risk_after_collect"] = float(tr.get("risk_after_collect", np.nan)) if pd.notna(tr.get("risk_after_collect", np.nan)) else np.nan
            row["pump_speed_change"] = float(tr.get("pump_speed_change", np.nan)) if pd.notna(tr.get("pump_speed_change", np.nan)) else np.nan

            row["pump_number"] = int(p)
            row["pmpID"] = f"{bits['sub']}_{bits['ses']}_T{int(row['tID'])}_P{p}" if pd.notna(row["tID"]) else f"{bits['sub']}_{bits['ses']}_P{p}"
            row["pmpL"] = float(lat_list[p-1]) if isinstance(lat_list, list) and len(lat_list) >= p else np.nan

            row["ses_mTheta"] = np.nan
            row["ses_SD_mThetaB"] = np.nan
            row["ses_avgP300_All"] = np.nan
            row["ses_avgP300_Fz"] = np.nan
            row["ses_avgP300_FCz"] = np.nan
            row["ses_avgP300_Cz"] = np.nan
            row["ses_avgP300_Pz"] = np.nan

            row["pre200_pop_Theta"] = np.nan
            row["post200_pop_Theta"] = np.nan
            row["pre200_collect_Theta"] = np.nan
            row["post200_collect_Theta"] = np.nan
            row["pre200_pmp_Theta"] = np.nan

            row["theta_before_pump"] = np.nan
            row["theta_before_pump_Fp1"] = np.nan
            row["theta_before_pump_Fp2"] = np.nan
            row["frontal_asymmetry"] = np.nan

            row["trial_mTheta"] = np.nan
            row["trial_SD_mThetaB"] = np.nan

            pump_rows.append(row)

    pump_df = pd.DataFrame(pump_rows)
    return pump_df, trials, summary

# ----------------------------
# Pair processing
# ----------------------------
def process_pair(beh_fp: Path, xdf_fp: Path | None):
    pump_df, trials_df, summary_df = beh_to_pump_table(beh_fp)
    if pump_df is None or len(pump_df) == 0:
        return None

    if xdf_fp is None:
        return pump_df

    try:
        streams, _ = pyxdf.load_xdf(str(xdf_fp))
        eeg_stream, marker_stream = pick_streams(streams)
        if eeg_stream is None or marker_stream is None:
            return pump_df

        raw, eeg_t = build_raw_from_xdf_eeg(eeg_stream)
        raw = preprocess_raw(raw)
        raw.resample(TARGET_SFREQ, verbose=False)
        eeg_t0 = float(eeg_t[0])

        md = markers_to_df(marker_stream)

        ses_mTheta, ses_SD_mThetaB = extract_session_theta_metrics(raw)
        p300 = extract_p300_metrics(raw, md, eeg_t0)
        ev_theta = extract_event_theta_windows(raw, md, eeg_t0)

        pump_theta = theta_before_each_pump(raw, md, eeg_t0)
        trial_theta = trial_theta_from_markers(raw, md, eeg_t0)

        pump_df["ses_mTheta"] = ses_mTheta
        pump_df["ses_SD_mThetaB"] = ses_SD_mThetaB
        for k, v in p300.items():
            pump_df[k] = v
        for k, v in ev_theta.items():
            pump_df[k] = v

        pump_df["tID"] = pd.to_numeric(pump_df["tID"], errors="coerce").astype("Int64")
        pump_df["pump_number"] = pd.to_numeric(pump_df["pump_number"], errors="coerce").astype("Int64")

        if trial_theta is not None and len(trial_theta) > 0:
            tt = trial_theta.copy()
            tt["tID"] = pd.to_numeric(tt["trial"], errors="coerce").astype("Int64")
            tt = tt.drop(columns=["trial"], errors="ignore")

            for c in ["trial_mTheta", "trial_SD_mThetaB"]:
                if c in pump_df.columns:
                    pump_df = pump_df.drop(columns=[c])

            pump_df = pump_df.merge(tt, on="tID", how="left")

        if pump_theta is not None and len(pump_theta) > 0:
            pt = pump_theta.copy()
            pt["tID"] = pd.to_numeric(pt["trial"], errors="coerce").astype("Int64")
            pt["pump_number"] = pd.to_numeric(pt["pump_number"], errors="coerce").astype("Int64")
            pt = pt.drop(columns=["trial"], errors="ignore")

            for c in ["theta_before_pump", "theta_before_pump_Fp1", "theta_before_pump_Fp2", "frontal_asymmetry"]:
                if c in pump_df.columns:
                    pump_df = pump_df.drop(columns=[c])

            pump_df = pump_df.merge(pt, on=["tID", "pump_number"], how="left")

        for c in ["trial_mTheta","trial_SD_mThetaB","theta_before_pump","theta_before_pump_Fp1","theta_before_pump_Fp2","frontal_asymmetry"]:
            if c not in pump_df.columns:
                pump_df[c] = np.nan

        return pump_df

    except Exception as e:
        print("⚠️ XDF failed:", xdf_fp.name, "->", e)
        return pump_df

# ----------------------------
# Main: find files + build master
# ----------------------------
beh_files = sorted(ROOT.rglob("*_beh.xlsx"))
xdf_files = sorted(ROOT.rglob("*.xdf")) + sorted(ROOT.rglob("*.xdfz"))

print("Found BEH files:", len(beh_files))
print("Found XDF files:", len(xdf_files))

xdf_map = {}
for xf in xdf_files:
    key = xf.name.split("_task")[0]
    xdf_map.setdefault(key, []).append(xf)

all_rows = []
missing_xdf = 0
processed = 0

for bf in beh_files:
    key = bf.name.split("_task")[0]
    xmatch = xdf_map.get(key, [])
    xdf_fp = xmatch[0] if len(xmatch) else None
    if xdf_fp is None:
        missing_xdf += 1

    print("\n---")
    print("BEH:", bf.name)
    print("XDF:", xdf_fp.name if xdf_fp else "MISSING (BEH-only)")

    out = process_pair(bf, xdf_fp)
    if out is not None and len(out) > 0:
        all_rows.append(out)
        processed += 1
        print("Rows:", out.shape)

if len(all_rows) == 0:
    raise RuntimeError("No output rows produced. Check BEH files contain Trials + pump_count > 0.")

master = pd.concat(all_rows, ignore_index=True)
print("\n✅ sessions processed:", processed)
print("⚠️ missing XDF sessions:", missing_xdf)
print("✅ master shape:", master.shape)

# Participant total points
if "pID" in master.columns and "sesPoints" in master.columns and "sID" in master.columns:
    master["pID"] = normalize_pid_series(master["pID"])
    tmp = master[["pID","sID","sesPoints"]].drop_duplicates()
    totals = tmp.groupby("pID")["sesPoints"].sum()
    master["pPoints"] = master["pID"].map(totals)

# ----------------------------
# Build participant-level survey table from session 5 BEH files
# ----------------------------
survey_rows = []

for bf in beh_files:
    bits = parse_bids_bits(bf.name)
    if pd.isna(bits["ses_num"]) or int(bits["ses_num"]) != 5:
        continue

    sheets = read_beh_sheets(bf)
    survey_df = get_sheet(sheets, ["survey", "surveys"])
    if survey_df is None or len(survey_df) == 0:
        continue

    participant_id = normalize_pid_series(pd.Series([bits["sub"]])).iloc[0]
    survey_row = extract_survey_row(survey_df, participant_id)
    if survey_row is not None:
        survey_rows.append(survey_row)

survey_master = pd.DataFrame(survey_rows) if len(survey_rows) else pd.DataFrame(columns=["pID"])

if len(survey_master):
    survey_master["pID"] = normalize_pid_series(survey_master["pID"])
    survey_master = survey_master.drop_duplicates(subset=["pID"], keep="first")
    print("Survey rows loaded:", survey_master.shape)
else:
    print("No survey rows found.")

# ----------------------------
# Merge only high/low from manifest
# ----------------------------
if manifest is not None and "pID" in manifest.columns:
    master["pID"] = normalize_pid_series(master["pID"])
    manifest["pID"] = normalize_pid_series(manifest["pID"])

    master = master.merge(manifest[["pID", "risk_group"]], on="pID", how="left", suffixes=("", "_manifest"))

    if "risk_group_manifest" in master.columns:
        if "risk_group" in master.columns:
            master["risk_group"] = master["risk_group"].fillna(master["risk_group_manifest"])
            master = master.drop(columns=["risk_group_manifest"])
        else:
            master = master.rename(columns={"risk_group_manifest": "risk_group"})

    if "risk_group" not in master.columns:
        master["risk_group"] = np.nan

    print("Non-missing risk_group rows after merge:", int(master["risk_group"].notna().sum()))
    print(master[["pID", "risk_group"]].dropna().drop_duplicates().head(10))
else:
    if "risk_group" not in master.columns:
        master["risk_group"] = np.nan

# Merge survey responses
if len(survey_master):
    master = master.merge(survey_master, on="pID", how="left")

# ----------------------------
# Ensure final schema columns exist
# ----------------------------
schema_cols = [
    "pID","risk_group","cond","pPoints","sID","ses_num",
    "sesPoints","ses_mTheta","ses_SD_mThetaB",
    "ses_avgP300_All","ses_avgP300_Fz","ses_avgP300_FCz","ses_avgP300_Cz","ses_avgP300_Pz",
    "smPmp","SD_smPmp","smADJ_pmp","SD_smADJ_pmp",
    "exp_rate","exp_N","bonus_N","collect_N","collect_rate",
    "pretask_EOB_avgTheta","posttask_EOB_avgTheta",
    "pretask_ECB_avgTheta","posttask_ECB_avgTheta",
    "pretask_CB_avgTheta","posttask_CB_avgTheta",
    "tID","trial_mTheta","trial_SD_mThetaB",
    "pop","collect","nf_high_fraction","nf_green_success","t_earned","t_loss",
    "collectL_trial","collectL_ready",
    "pmp_count",
    "pre200_pop_Theta","post200_pop_Theta","pre200_collect_Theta","post200_collect_Theta","pre200_pmp_Theta",
    "risk_after_explosion","risk_after_collect","pump_speed_change",
    "pump_number","pmpID","pmpL",
    "mpmpL","SD_mpmpL","medpmpL","SD_medpmpL",
    "theta_before_pump","theta_before_pump_Fp1","theta_before_pump_Fp2","frontal_asymmetry",
]

for c in schema_cols:
    if c not in master.columns:
        master[c] = np.nan

survey_cols = [c for c in master.columns if str(c).lower().startswith("survey_")]
base_order = [c for c in schema_cols if c in master.columns and c != "pID"]
remaining = [c for c in master.columns if c not in (["pID"] + survey_cols + base_order)]

ordered = ["pID"] + survey_cols + base_order + remaining
master = master[ordered]

if "risk_group" not in master.columns:
    master["risk_group"] = np.nan

print("Survey columns found:", len(survey_cols))
print("Unique risk_group values:", sorted(master["risk_group"].dropna().astype(str).unique().tolist()))

# Save
csv_path = OUT / "BART_master_pump_level.csv"
xlsx_path = OUT / "BART_master_pump_level.xlsx"
master.to_csv(csv_path, index=False)
master.to_excel(xlsx_path, index=False)

print("\n✅ Saved CSV:", csv_path)
print("✅ Saved XLSX:", xlsx_path)

master.head(10)

Loaded manifest: /content/manifest.xlsx
Manifest rows: (49, 2)
    pID risk_group
0  P001          H
1  P002          H
2  P003          H
3  P004          H
4  P006          H
Found BEH files: 90
Found XDF files: 84

---
BEH: sub-P001_ses-S001_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P001_ses-S001_task-Default_run-001_eeg.xdf
Rows: (536, 59)

---
BEH: sub-P001_ses-S002_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P001_ses-S002_task-Default_run-001_eeg.xdf
Rows: (691, 59)

---
BEH: sub-P001_ses-S003_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P001_ses-S003_task-Default_run-001_eeg.xdf
Rows: (688, 59)

---
BEH: sub-P001_ses-S004_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: MISSING (BEH-only)
Rows: (562, 59)

---
BEH: sub-P001_ses-S005_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P001_ses-S005_task-Default_run-001_eeg.xdf
Rows: (678, 59)

---
BEH: sub-P003_ses-S001_task-BART_cond-nf_sham_run-001_beh.xlsx
XDF: sub-P003_ses-S001_task-Default_run-001_eeg.xdf
Rows: (305,

Rows: (786, 59)

---
BEH: sub-P005_ses-S001_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P005_ses-S001_task-Default_run-001_eeg.xdf
Rows: (633, 59)

---
BEH: sub-P005_ses-S002_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P005_ses-S002_task-Default_run-001_eeg.xdf
Rows: (580, 59)

---
BEH: sub-P005_ses-S003_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P005_ses-S003_task-Default_run-001_eeg.xdf
Rows: (618, 59)

---
BEH: sub-P005_ses-S004_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P005_ses-S004_task-Default_run-001_eeg.xdf
Rows: (621, 59)

---
BEH: sub-P005_ses-S005_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P005_ses-S005_task-Default_run-001_eeg.xdf
Rows: (574, 59)

---
BEH: sub-P006_ses-S001_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P006_ses-S001_task-Default_run-001_eeg.xdf
Rows: (507, 59)

---
BEH: sub-P006_ses-S002_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P006_ses-S002_task-Default_run-001_eeg.xdf
Rows: (527, 59)

---
BEH: sub-P006_ses-S003_task

Rows: (760, 59)

---
BEH: sub-P018_ses-S004_task-BART_cond-nf_sham_run-001_beh.xlsx
XDF: sub-P018_ses-S004_task-Default_run-001_eeg.xdf
Rows: (646, 59)

---
BEH: sub-P018_ses-S005_task-BART_cond-nf_sham_run-001_beh.xlsx
XDF: sub-P018_ses-S005_task-Default_run-001_eeg.xdf
Rows: (807, 59)

---
BEH: sub-P019_ses-S001_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P019_ses-S001_task-Default_run-001_eeg.xdf
Rows: (618, 59)

---
BEH: sub-P019_ses-S002_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P019_ses-S002_task-Default_run-001_eeg.xdf
Rows: (585, 59)

---
BEH: sub-P019_ses-S003_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P019_ses-S003_task-Default_run-001_eeg.xdf
Rows: (515, 59)

---
BEH: sub-P019_ses-S004_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P019_ses-S004_task-Default_run-001_eeg.xdf
Rows: (534, 59)

---
BEH: sub-P019_ses-S005_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P019_ses-S005_task-Default_run-001_eeg.xdf
Rows: (552, 59)

✅ sessions processed: 90
⚠️ mis

,pID,survey_r1_sub,survey_r1_ses,survey_r1_run,survey_r1_task,survey_r1_condition,survey_r1_qid,survey_r1_section,survey_r1_question,survey_r1_response_1to5,...,pmpID,pmpL,mpmpL,SD_mpmpL,medpmpL,SD_medpmpL,theta_before_pump,theta_before_pump_Fp1,theta_before_pump_Fp2,frontal_asymmetry
0,P001,P001,S005,1,BART,nf_real,S1_Q1,Stress,"Overall, how stressful did you find the tasks ...",1,...,P001_S001_T1_P1,0.0234,0.270922,0.08398,0.237424,0.09864,4.527025e-11,2.317708e-11,3.801747e-11,0.478605
1,P001,P001,S005,1,BART,nf_real,S1_Q1,Stress,"Overall, how stressful did you find the tasks ...",1,...,P001_S001_T1_P1,0.0234,0.270922,0.08398,0.237424,0.09864,2.708999e-11,3.875262e-11,4.269164e-11,0.094481
2,P001,P001,S005,1,BART,nf_real,S1_Q1,Stress,"Overall, how stressful did you find the tasks ...",1,...,P001_S001_T1_P2,1.3350,0.270922,0.08398,0.237424,0.09864,4.678572e-11,2.006399e-11,5.438994e-11,0.966833
3,P001,P001,S005,1,BART,nf_real,S1_Q1,Stress,"Overall, how stressful did you find the tasks ...",1,...,P001_S001_T1_P2,1.3350,0.270922,0.08398,0.237424,0.09864,6.140069e-11,9.258605e-11,1.432722e-10,0.432821
4,P001,P001,S005,1,BART,nf_real,S1_Q1,Stress,"Overall, how stressful did you find the tasks ...",1,...,P001_S001_T1_P3,0.3838,0.270922,0.08398,0.237424,0.09864,2.324976e-11,1.047581e-11,1.462728e-11,0.308777
5,P001,P001,S005,1,BART,nf_real,S1_Q1,Stress,"Overall, how stressful did you find the tasks ...",1,...,P001_S001_T1_P3,0.3838,0.270922,0.08398,0.237424,0.09864,4.378374e-11,4.157826e-11,4.647099e-11,0.108775
6,P001,P001,S005,1,BART,nf_real,S1_Q1,Stress,"Overall, how stressful did you find the tasks ...",1,...,P001_S001_T1_P4,0.3838,0.270922,0.08398,0.237424,0.09864,6.958198e-11,1.502712e-10,1.646237e-10,0.090644
7,P001,P001,S005,1,BART,nf_real,S1_Q1,Stress,"Overall, how stressful did you find the tasks ...",1,...,P001_S001_T1_P4,0.3838,0.270922,0.08398,0.237424,0.09864,1.505707e-11,1.371341e-11,6.212980e-12,-0.712877
8,P001,P001,S005,1,BART,nf_real,S1_Q1,Stress,"Overall, how stressful did you find the tasks ...",1,...,P001_S001_T1_P5,0.4171,0.270922,0.08398,0.237424,0.09864,4.014135e-11,4.098451e-11,5.847166e-11,0.348199
9,P001,P001,S005,1,BART,nf_real,S1_Q1,Stress,"Overall, how stressful did you find the tasks ...",1,...,P001_S001_T1_P5,0.4171,0.270922,0.08398,0.237424,0.09864,3.838771e-11,2.838379e-11,3.433010e-11,0.184292


In [ ]:
# ============================================================
# BART MULTIMODAL MASTER PIPELINE (Pump-level)
# Input: *_beh.xlsx + matching *.xdf/*.xdfz + manifest.xlsx
# Output: outputs/BART_master_pump_level.{csv,xlsx}
#
# Includes:
#  - Behavioral pump/trial/session metrics
#  - EEG theta + P300 metrics from XDF
#  - risk_group (H/L) from manifest.xlsx column "high/low"
#  - Survey responses from session 5 BEH survey sheet
# ============================================================

!pip -q install pyxdf mne openpyxl pandas numpy

import re, json
from pathlib import Path
import numpy as np
import pandas as pd
import pyxdf
import mne

# ----------------------------
# CONFIG
# ----------------------------
ROOT = Path("/content") if Path("/content").exists() else Path(".")
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)

TARGET_SFREQ = 512.0

# ERP epoching (explode)
ERP_TMIN, ERP_TMAX = -0.2, 0.8
ERP_BASELINE = (ERP_TMIN, 0.0)
P3_WIN = (0.300, 0.450)

# Theta windows
WIN200 = 0.200
THETA_BAND = (4.0, 7.0)

# ActiCAP channel order
EEG_ORDER = ["Fz","Cz","Pz","POz","Fp1","Fp2","F3","F4","FCz","C3","C4","CPz","P3","P4","O1","O2"]
FRONTAL_THETA_CHS = ["Fp1","Fp2","Fz","F3","F4","FCz"]

# ----------------------------
# ID normalization
# ----------------------------
def normalize_pid_series(s):
    s = s.astype(str).str.strip().str.upper()
    nums = s.str.extract(r"(\d+)")[0]
    out = nums.fillna("").apply(lambda x: f"P{int(x):03d}" if x != "" else np.nan)
    return out.where(out.notna(), s)

# ----------------------------
# Manifest: only high/low
# ----------------------------
manifest = None
manifest_path = next(iter(ROOT.rglob("manifest.xlsx")), None)

if manifest_path is not None and manifest_path.exists():
    manifest_raw = pd.read_excel(manifest_path)
    manifest_raw.columns = [str(c).strip() for c in manifest_raw.columns]

    cols_l = {str(c).strip().lower(): c for c in manifest_raw.columns}
    pid_col = None
    risk_col = None

    for k in ["subject_id", "participant_id", "participant", "subject", "sub", "pid", "id"]:
        if k in cols_l:
            pid_col = cols_l[k]
            break

    for k in ["high/low", "risk_group", "risk", "group"]:
        if k in cols_l:
            risk_col = cols_l[k]
            break

    if pid_col is not None and risk_col is not None:
        manifest = manifest_raw[[pid_col, risk_col]].copy()
        manifest.columns = ["pID", "risk_group"]
        manifest["pID"] = normalize_pid_series(manifest["pID"])
        manifest["risk_group"] = manifest["risk_group"].astype(str).str.strip().str.upper()
        manifest = manifest.drop_duplicates(subset=["pID"], keep="first")
        print("Loaded manifest:", manifest_path)
        print("Manifest rows:", manifest.shape)
        print(manifest.head())
    else:
        print("manifest.xlsx found, but could not detect subject_id and/or high/low columns.")
else:
    print("No manifest.xlsx found.")

# ----------------------------
# Utilities
# ----------------------------
def parse_bids_bits(name: str):
    base = Path(name).name
    sub = re.search(r"sub-([A-Za-z0-9]+)", base)
    ses = re.search(r"ses-([A-Za-z0-9]+)", base)
    run = re.search(r"run-([0-9]+)", base)
    task = re.search(r"task-([A-Za-z0-9]+)", base)
    cond = re.search(r"cond-([A-Za-z0-9_]+)", base)

    out = {
        "sub": sub.group(1) if sub else None,
        "ses": ses.group(1) if ses else None,
        "run": run.group(1) if run else None,
        "task": task.group(1) if task else None,
        "condition": cond.group(1) if cond else None,
    }
    if out["ses"] is not None:
        m = re.search(r"(\d+)", out["ses"])
        out["ses_num"] = int(m.group(1)) if m else np.nan
    else:
        out["ses_num"] = np.nan
    return out

def read_beh_sheets(fp: Path):
    xls = pd.ExcelFile(fp)
    sheets = {}
    for sh in xls.sheet_names:
        df = pd.read_excel(fp, sheet_name=sh)
        df.columns = [str(c).strip() for c in df.columns]
        sheets[sh.lower()] = df
    return sheets

def get_sheet(sheets: dict, candidates):
    for c in candidates:
        if c in sheets:
            return sheets[c]
    return None

def safe_json_list(x):
    try:
        return json.loads(x) if isinstance(x, str) else (x if isinstance(x, list) else None)
    except Exception:
        return None

def extract_survey_row(survey_df, participant_id):
    if survey_df is None or len(survey_df) == 0:
        return None

    df = survey_df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    participant_id = normalize_pid_series(pd.Series([participant_id]))[0]

    if len(df) == 1:
        row = df.iloc[0].to_dict()
        row["pID"] = participant_id
        return row

    cols_l = {str(c).strip().lower(): c for c in df.columns}
    q_col = None
    r_col = None

    for k in ["question", "item", "prompt"]:
        if k in cols_l:
            q_col = cols_l[k]
            break
    for k in ["response", "answer", "value", "score"]:
        if k in cols_l:
            r_col = cols_l[k]
            break

    if q_col is not None and r_col is not None:
        out = {"pID": participant_id}
        for _, r in df.iterrows():
            q = str(r[q_col]).strip()
            q_clean = re.sub(r"[^A-Za-z0-9]+", "_", q).strip("_").lower()
            out[f"survey_{q_clean}"] = r[r_col]
        return out

    out = {"pID": participant_id}
    for i, (_, r) in enumerate(df.iterrows(), start=1):
        for c in df.columns:
            cname = re.sub(r"[^A-Za-z0-9]+", "_", str(c)).strip("_").lower()
            out[f"survey_r{i}_{cname}"] = r[c]
    return out

# ----------------------------
# XDF helpers
# ----------------------------
def parse_marker_message(msg):
    s = str(msg)
    parts = s.split(";")
    out = {"code": parts[0].strip()}
    for p in parts[1:]:
        if "=" in p:
            k, v = p.split("=", 1)
            out[k.strip()] = v.strip()
    return out

def _stream_name_lower(s):
    info = s.get("info", {})
    nm = info.get("name", "")
    if isinstance(nm, (list, tuple, np.ndarray)):
        nm = nm[0] if len(nm) else ""
    return str(nm).strip().lower()

def pick_streams(streams):
    eeg = None
    best = -1

    for s in streams:
        ts = s.get("time_series", None)
        if ts is None:
            continue
        arr = np.asarray(ts)
        ch = 1 if arr.ndim == 1 else arr.shape[1]
        if ch > best:
            eeg = s
            best = ch

    markers = None
    for s in streams:
        ts = s.get("time_series", None)
        if ts is None:
            continue
        arr = np.asarray(ts)
        ch = 1 if arr.ndim == 1 else arr.shape[1]
        if ch != 1:
            continue
        nm = _stream_name_lower(s)
        if ("bart" in nm) or ("marker" in nm) or ("markers" in nm) or ("event" in nm):
            markers = s
            break

    if markers is None:
        for s in streams:
            ts = s.get("time_series", None)
            if ts is None:
                continue
            arr = np.asarray(ts)
            ch = 1 if arr.ndim == 1 else arr.shape[1]
            if ch == 1:
                markers = s
                break

    return eeg, markers

def markers_to_df(marker_stream):
    m_ts = np.asarray(marker_stream["time_stamps"], dtype=float)
    m_val = np.asarray(marker_stream["time_series"], dtype=object)

    rows = []
    for t, v in zip(m_ts, m_val):
        msg = v[0] if isinstance(v, (list, tuple, np.ndarray)) else v
        d = parse_marker_message(msg)
        d["lsltime"] = float(t)
        rows.append(d)

    df = pd.DataFrame(rows)
    for k in ["trial", "pump", "loss", "earnings", "total"]:
        if k in df.columns:
            df[k] = pd.to_numeric(df[k], errors="coerce")
    return df

def build_raw_from_xdf_eeg(eeg_stream):
    data = np.asarray(eeg_stream["time_series"])
    t = np.asarray(eeg_stream["time_stamps"], dtype=float)
    if data.ndim == 1:
        data = data[:, None]
    n_samp, n_ch = data.shape

    sfreq_est = 1.0 / np.median(np.diff(t))
    ch_names = EEG_ORDER[:n_ch] if n_ch <= len(EEG_ORDER) else [f"EEG{ii+1:02d}" for ii in range(n_ch)]
    arr = data.T.astype(float)

    peak = float(np.nanmax(np.abs(arr)))
    if peak > 1e3:
        arr *= 1e-6
    elif peak > 5.0:
        arr *= 1e-3

    info = mne.create_info(ch_names=ch_names, sfreq=sfreq_est, ch_types=["eeg"] * n_ch)
    raw = mne.io.RawArray(arr, info, verbose=False)
    return raw, t

def preprocess_raw(raw):
    raw = raw.copy()
    raw.filter(0.1, 30.0, picks="eeg", verbose=False)
    try:
        raw.notch_filter([60], picks="eeg", verbose=False)
    except Exception:
        pass
    try:
        raw.set_eeg_reference("average", verbose=False)
    except Exception:
        pass
    return raw

def lsltime_to_sample(lsltime, eeg_t0, sfreq):
    return int(np.round((lsltime - eeg_t0) * sfreq))

# ----------------------------
# EEG feature extraction
# ----------------------------
def compute_theta_power_window(raw_theta, sample, tmin_s, tmax_s, picks):
    sf = raw_theta.info["sfreq"]
    start = int(sample + np.round(tmin_s * sf))
    stop  = int(sample + np.round(tmax_s * sf))
    start = max(start, 0)
    stop  = min(stop, raw_theta.n_times)
    if stop <= start + 2:
        return np.nan
    seg = raw_theta.get_data(picks=picks, start=start, stop=stop)
    return float(np.mean(seg**2))

def extract_session_theta_metrics(raw):
    rtheta = raw.copy().filter(THETA_BAND[0], THETA_BAND[1], picks="eeg", verbose=False)
    picks = [raw.ch_names.index(ch) for ch in FRONTAL_THETA_CHS if ch in raw.ch_names]
    if len(picks) == 0:
        return np.nan, np.nan
    X = rtheta.get_data(picks=picks)
    theta_ts = np.mean(X**2, axis=0)
    return float(np.mean(theta_ts)), float(np.std(theta_ts, ddof=1))

def extract_p300_metrics(raw, md, eeg_t0):
    keys = ["ses_avgP300_All","ses_avgP300_Fz","ses_avgP300_FCz","ses_avgP300_Cz","ses_avgP300_Pz"]
    if "code" not in md.columns:
        return {k: np.nan for k in keys}

    exp = md[md["code"].astype(str).str.upper().eq("BART_EXPLODE")].copy()
    if len(exp) == 0:
        return {k: np.nan for k in keys}

    sf = raw.info["sfreq"]
    exp = exp.dropna(subset=["lsltime"]).copy()
    exp["sample"] = exp["lsltime"].apply(lambda t: lsltime_to_sample(float(t), eeg_t0, sf)).astype(int)

    min_samp = int(np.ceil((-ERP_TMIN) * sf))
    exp = exp[(exp["sample"] >= min_samp) & (exp["sample"] < raw.n_times)].copy()
    if len(exp) == 0:
        return {k: np.nan for k in keys}

    events = np.c_[exp["sample"].to_numpy(), np.zeros(len(exp), int), np.ones(len(exp), int)]
    epochs = mne.Epochs(
        raw, events, event_id={"EXPLODE": 1},
        tmin=ERP_TMIN, tmax=ERP_TMAX, baseline=ERP_BASELINE,
        preload=True, reject_by_annotation=True, verbose=False
    )
    if len(epochs) == 0:
        return {k: np.nan for k in keys}

    ev = epochs.average()
    times = ev.times
    m1 = np.argmin(np.abs(times - P3_WIN[0]))
    m2 = np.argmin(np.abs(times - P3_WIN[1]))

    out = {}
    vals = []
    for ch in ["Fz","FCz","Cz","Pz"]:
        k = f"ses_avgP300_{ch}"
        if ch in ev.ch_names:
            idx = ev.ch_names.index(ch)
            erp_uV = ev.data[idx] * 1e6
            p3 = float(np.max(erp_uV[m1:m2]))
            out[k] = p3
            vals.append(p3)
        else:
            out[k] = np.nan
    out["ses_avgP300_All"] = float(np.nanmean(vals)) if len(vals) else np.nan
    return out

def extract_event_theta_windows(raw, md, eeg_t0):
    rtheta = raw.copy().filter(THETA_BAND[0], THETA_BAND[1], picks="eeg", verbose=False)
    sf = raw.info["sfreq"]
    picks = [raw.ch_names.index(ch) for ch in FRONTAL_THETA_CHS if ch in raw.ch_names]
    if len(picks) == 0:
        return {
            "pre200_pop_Theta": np.nan, "post200_pop_Theta": np.nan,
            "pre200_collect_Theta": np.nan, "post200_collect_Theta": np.nan,
            "pre200_pmp_Theta": np.nan,
        }

    def _mean_prepost(code, pre=True):
        d = md[md["code"].astype(str).str.upper().eq(code)].dropna(subset=["lsltime"]).copy()
        if len(d) == 0:
            return np.nan
        d["sample"] = d["lsltime"].apply(lambda t: lsltime_to_sample(float(t), eeg_t0, sf)).astype(int)
        vals = []
        for s in d["sample"].tolist():
            v = compute_theta_power_window(rtheta, s, -WIN200, 0.0, picks) if pre else compute_theta_power_window(rtheta, s, 0.0, WIN200, picks)
            vals.append(v)
        return float(np.nanmean(vals)) if len(vals) else np.nan

    return {
        "pre200_pop_Theta": _mean_prepost("BART_EXPLODE", pre=True),
        "post200_pop_Theta": _mean_prepost("BART_EXPLODE", pre=False),
        "pre200_collect_Theta": _mean_prepost("BART_COLLECT", pre=True),
        "post200_collect_Theta": _mean_prepost("BART_COLLECT", pre=False),
        "pre200_pmp_Theta": _mean_prepost("BART_PUMP", pre=True),
    }

def theta_before_each_pump(raw, md, eeg_t0):
    pumps = md[md["code"].astype(str).str.upper().eq("BART_PUMP")].copy()
    if len(pumps) == 0 or ("trial" not in pumps.columns) or ("pump" not in pumps.columns):
        return pd.DataFrame(columns=["trial","pump_number","theta_before_pump","theta_before_pump_Fp1","theta_before_pump_Fp2","frontal_asymmetry"])

    rtheta = raw.copy().filter(THETA_BAND[0], THETA_BAND[1], picks="eeg", verbose=False)
    sf = raw.info["sfreq"]

    pumps["trial"] = pd.to_numeric(pumps["trial"], errors="coerce")
    pumps["pump_number"] = pd.to_numeric(pumps["pump"], errors="coerce")
    pumps = pumps.dropna(subset=["trial","pump_number","lsltime"]).copy()

    pumps["trial"] = pumps["trial"].astype(int)
    pumps["pump_number"] = pumps["pump_number"].astype(int)
    pumps["sample"] = pumps["lsltime"].apply(lambda t: lsltime_to_sample(float(t), eeg_t0, sf)).astype(int)

    min_samp = int(np.ceil(WIN200 * sf))
    pumps = pumps[(pumps["sample"] >= min_samp) & (pumps["sample"] < raw.n_times)].copy()
    if len(pumps) == 0:
        return pd.DataFrame(columns=["trial","pump_number","theta_before_pump","theta_before_pump_Fp1","theta_before_pump_Fp2","frontal_asymmetry"])

    frontal_picks = [raw.ch_names.index(ch) for ch in FRONTAL_THETA_CHS if ch in raw.ch_names]
    fp1_pick = [raw.ch_names.index("Fp1")] if "Fp1" in raw.ch_names else []
    fp2_pick = [raw.ch_names.index("Fp2")] if "Fp2" in raw.ch_names else []

    rows = []
    for _, r in pumps.iterrows():
        s = int(r["sample"])
        pre_frontal = compute_theta_power_window(rtheta, s, -WIN200, 0.0, frontal_picks) if frontal_picks else np.nan
        pre_fp1 = compute_theta_power_window(rtheta, s, -WIN200, 0.0, fp1_pick) if fp1_pick else np.nan
        pre_fp2 = compute_theta_power_window(rtheta, s, -WIN200, 0.0, fp2_pick) if fp2_pick else np.nan
        fa = (np.log(pre_fp2 + 1e-12) - np.log(pre_fp1 + 1e-12)) if (np.isfinite(pre_fp1) and np.isfinite(pre_fp2)) else np.nan

        rows.append({
            "trial": int(r["trial"]),
            "pump_number": int(r["pump_number"]),
            "theta_before_pump": pre_frontal,
            "theta_before_pump_Fp1": pre_fp1,
            "theta_before_pump_Fp2": pre_fp2,
            "frontal_asymmetry": fa
        })

    return pd.DataFrame(rows)

def trial_theta_from_markers(raw, md, eeg_t0):
    sf = raw.info["sfreq"]
    rtheta = raw.copy().filter(THETA_BAND[0], THETA_BAND[1], picks="eeg", verbose=False)
    picks = [raw.ch_names.index(ch) for ch in FRONTAL_THETA_CHS if ch in raw.ch_names]
    if len(picks) == 0:
        return pd.DataFrame(columns=["trial","trial_mTheta","trial_SD_mThetaB"])

    starts = md[md["code"].astype(str).str.upper().eq("BART_TRIAL_START")].copy()
    ends = md[md["code"].astype(str).str.upper().isin(["BART_COLLECT","BART_EXPLODE"])].copy()
    if "trial" not in starts.columns or "trial" not in ends.columns:
        return pd.DataFrame(columns=["trial","trial_mTheta","trial_SD_mThetaB"])

    starts["trial"] = pd.to_numeric(starts["trial"], errors="coerce")
    ends["trial"] = pd.to_numeric(ends["trial"], errors="coerce")
    starts = starts.dropna(subset=["trial","lsltime"]).copy()
    ends = ends.dropna(subset=["trial","lsltime"]).copy()
    if len(starts) == 0 or len(ends) == 0:
        return pd.DataFrame(columns=["trial","trial_mTheta","trial_SD_mThetaB"])

    out = []
    for tnum, g0 in starts.groupby(starts["trial"].astype(int)):
        t_start = float(g0["lsltime"].min())
        g1 = ends[ends["trial"].astype(int) == int(tnum)]
        if len(g1) == 0:
            continue
        t_end = float(g1["lsltime"].min())
        if t_end <= t_start:
            continue

        s0 = lsltime_to_sample(t_start, eeg_t0, sf)
        s1 = lsltime_to_sample(t_end, eeg_t0, sf)
        s0 = max(s0, 0)
        s1 = min(s1, raw.n_times)
        if s1 <= s0 + int(sf * 0.2):
            continue

        seg = rtheta.get_data(picks=picks, start=s0, stop=s1)
        theta_ts = np.mean(seg**2, axis=0)

        out.append({
            "trial": int(tnum),
            "trial_mTheta": float(np.mean(theta_ts)),
            "trial_SD_mThetaB": float(np.std(theta_ts, ddof=1)),
        })

    return pd.DataFrame(out)

# ----------------------------
# BEH -> pump table
# ----------------------------
def beh_to_pump_table(beh_fp: Path):
    sheets = read_beh_sheets(beh_fp)
    trials = get_sheet(sheets, ["trials", "trial"])
    summary = get_sheet(sheets, ["summary"])

    if trials is None or len(trials) == 0:
        return None, None, None

    trials = trials.copy()
    bits = parse_bids_bits(beh_fp.name)

    if "exploded" in trials.columns:
        trials["pop"] = trials["exploded"].astype(str).str.upper().isin(["TRUE","1","YES"]).astype(int)
    else:
        trials["pop"] = 0

    if "collected" in trials.columns:
        trials["collect"] = pd.to_numeric(trials["collected"], errors="coerce").fillna(0).astype(int)
    else:
        trials["collect"] = trials["events"].astype(str).str.contains("collect", case=False).astype(int) if "events" in trials.columns else 0

    for c in [
        "trial","pump_count","trial_earnings","loss_if_pop","trial_value","total_earnings",
        "pump_latency_mean","pump_latency_median","pump_latency_first",
        "collect_latency_from_ready","collect_latency_from_trial_start",
        "nf_high_frac","nf_green_success",
        "risk_after_explosion","risk_after_collect","pump_speed_change",
        "nf_mean_normalized"
    ]:
        if c in trials.columns:
            trials[c] = pd.to_numeric(trials[c], errors="coerce")

    if "nf_high_frac" in trials.columns:
        trials["nf_high_fraction"] = pd.to_numeric(trials["nf_high_frac"], errors="coerce")
    elif "nf_high_fraction" not in trials.columns:
        trials["nf_high_fraction"] = np.nan

    if "nf_green_success" in trials.columns and trials["nf_green_success"].notna().any():
        trials["nf_green_success"] = pd.to_numeric(trials["nf_green_success"], errors="coerce")
    else:
        trials["nf_green_success"] = np.where(trials["nf_high_fraction"] > 0.75, 1, 0).astype(int) if "nf_high_fraction" in trials.columns else np.nan

    block_col = "block" if "block" in trials.columns else None
    if block_col is not None:
        trials = trials.sort_values([block_col, "trial"], kind="mergesort") if "trial" in trials.columns else trials
        g = trials.groupby(block_col)["nf_green_success"]
        trials["nf_bonus_awarded_trial"] = ((g.shift(0)==1) & (g.shift(1)==1) & (g.shift(2)==1)).astype(int)
    else:
        trials["nf_bonus_awarded_trial"] = 0

    if "pump_count" in trials.columns and block_col is not None and "trial" in trials.columns:
        trials["prev_pumps"] = trials.groupby(block_col)["pump_count"].shift(1)
        prev_pop = trials.groupby(block_col)["pop"].shift(1)
        prev_collect = trials.groupby(block_col)["collect"].shift(1)

        trials["risk_after_explosion"] = np.where(prev_pop == 1, trials["pump_count"] - trials["prev_pumps"], np.nan)
        trials["risk_after_collect"] = np.where(prev_collect == 1, trials["pump_count"] - trials["prev_pumps"], np.nan)

        if "pump_latency_mean" in trials.columns:
            trials["pump_speed_change"] = trials["pump_latency_mean"] - trials.groupby(block_col)["pump_latency_mean"].shift(1)
        else:
            trials["pump_speed_change"] = np.nan
    else:
        if "risk_after_explosion" not in trials.columns:
            trials["risk_after_explosion"] = np.nan
        if "risk_after_collect" not in trials.columns:
            trials["risk_after_collect"] = np.nan
        if "pump_speed_change" not in trials.columns:
            trials["pump_speed_change"] = np.nan

    sesPoints = trials["total_earnings"].dropna().max() if "total_earnings" in trials.columns else np.nan
    exp_N = int(trials["pop"].sum())
    collect_N = int(trials["collect"].sum())
    tN = int(len(trials))
    exp_rate = exp_N / tN if tN else np.nan
    collect_rate = collect_N / tN if tN else np.nan
    bonus_N = int(trials["nf_bonus_awarded_trial"].sum()) if "nf_bonus_awarded_trial" in trials.columns else 0

    smPmp = float(trials["pump_count"].mean()) if "pump_count" in trials.columns else np.nan
    SD_smPmp = float(trials["pump_count"].std(ddof=1)) if "pump_count" in trials.columns else np.nan

    trials["adjusted_pumps_trial"] = trials["pump_count"] if "pump_count" in trials.columns else np.nan
    smADJ_pmp = float(trials["adjusted_pumps_trial"].mean()) if "adjusted_pumps_trial" in trials.columns else np.nan
    SD_smADJ_pmp = float(trials["adjusted_pumps_trial"].std(ddof=1)) if "adjusted_pumps_trial" in trials.columns else np.nan

    mpmpL = float(trials["pump_latency_mean"].mean()) if "pump_latency_mean" in trials.columns else np.nan
    SD_mpmpL = float(trials["pump_latency_mean"].std(ddof=1)) if "pump_latency_mean" in trials.columns else np.nan
    medpmpL = float(trials["pump_latency_median"].mean()) if "pump_latency_median" in trials.columns else np.nan
    SD_medpmpL = float(trials["pump_latency_median"].std(ddof=1)) if "pump_latency_median" in trials.columns else np.nan

    def _get_scalar_from_sources(keys, sources):
        for df in sources:
            if df is None or len(df) == 0:
                continue
            cols_lut = {str(c).strip().lower(): c for c in df.columns}
            for k in keys:
                kk = str(k).strip().lower()
                if kk in cols_lut:
                    col = cols_lut[kk]
                    ser = pd.to_numeric(df[col], errors="coerce")
                    if ser.notna().any():
                        return float(ser.dropna().iloc[0])
        return np.nan

    sources = [trials, summary]

    baseline_cols = {
        "baseline_eo_mu": _get_scalar_from_sources(
            ["baseline_eo_mu", "baseline_pre_eo_mu", "pretask_eob_avgtheta", "pretask_eo_avgtheta"], sources),
        "baseline_ec_mu": _get_scalar_from_sources(
            ["baseline_ec_mu", "baseline_pre_ec_mu", "pretask_ecb_avgtheta", "pretask_ec_avgtheta"], sources),
        "baseline_conc_mu": _get_scalar_from_sources(
            ["baseline_conc_mu", "baseline_pre_conc_mu", "pretask_cb_avgtheta", "pretask_conc_avgtheta"], sources),

        "baseline_post_eo_mu": _get_scalar_from_sources(
            ["baseline_post_eo_mu", "posttask_eob_avgtheta", "post_eob_avgtheta", "posttask_eo_avgtheta"], sources),
        "baseline_post_ec_mu": _get_scalar_from_sources(
            ["baseline_post_ec_mu", "posttask_ecb_avgtheta", "post_ecb_avgtheta", "posttask_ec_avgtheta"], sources),
        "baseline_post_conc_mu": _get_scalar_from_sources(
            ["baseline_post_conc_mu", "posttask_cb_avgtheta", "post_cb_avgtheta", "posttask_conc_avgtheta"], sources),
    }

    pump_rows = []
    for _, tr in trials.iterrows():
        n = int(tr["pump_count"]) if pd.notna(tr.get("pump_count", np.nan)) else 0
        if n <= 0:
            continue

        lat_list = safe_json_list(tr.get("pump_latencies_json", None))

        for p in range(1, n + 1):
            row = {}

            row["pID"] = normalize_pid_series(pd.Series([bits["sub"]]))[0]
            row["cond"] = bits["condition"]
            row["sID"] = bits["ses"]
            row["ses_num"] = bits["ses_num"]

            row["pPoints"] = np.nan
            row["risk_group"] = np.nan

            row["sesPoints"] = sesPoints
            row["exp_N"] = exp_N
            row["collect_N"] = collect_N
            row["bonus_N"] = bonus_N
            row["exp_rate"] = exp_rate
            row["collect_rate"] = collect_rate
            row["smPmp"] = smPmp
            row["SD_smPmp"] = SD_smPmp
            row["smADJ_pmp"] = smADJ_pmp
            row["SD_smADJ_pmp"] = SD_smADJ_pmp
            row["mpmpL"] = mpmpL
            row["SD_mpmpL"] = SD_mpmpL
            row["medpmpL"] = medpmpL
            row["SD_medpmpL"] = SD_medpmpL

            row["pretask_EOB_avgTheta"]  = baseline_cols.get("baseline_eo_mu", np.nan)
            row["posttask_EOB_avgTheta"] = baseline_cols.get("baseline_post_eo_mu", np.nan)
            row["pretask_ECB_avgTheta"]  = baseline_cols.get("baseline_ec_mu", np.nan)
            row["posttask_ECB_avgTheta"] = baseline_cols.get("baseline_post_ec_mu", np.nan)
            row["pretask_CB_avgTheta"]   = baseline_cols.get("baseline_conc_mu", np.nan)
            row["posttask_CB_avgTheta"]  = baseline_cols.get("baseline_post_conc_mu", np.nan)

            row["tID"] = int(tr["trial"]) if "trial" in tr and pd.notna(tr["trial"]) else np.nan
            row["pop"] = int(tr["pop"])
            row["collect"] = int(tr["collect"])
            row["pmp_count"] = int(tr["pump_count"]) if pd.notna(tr.get("pump_count", np.nan)) else np.nan
            row["t_earned"] = float(tr["trial_earnings"]) if "trial_earnings" in tr and pd.notna(tr["trial_earnings"]) else np.nan
            row["t_loss"] = float(tr["loss_if_pop"]) if "loss_if_pop" in tr and pd.notna(tr["loss_if_pop"]) else np.nan

            row["collectL_trial"] = float(tr["collect_latency_from_trial_start"]) if "collect_latency_from_trial_start" in tr and pd.notna(tr["collect_latency_from_trial_start"]) else np.nan
            row["collectL_ready"] = float(tr["collect_latency_from_ready"]) if "collect_latency_from_ready" in tr and pd.notna(tr["collect_latency_from_ready"]) else np.nan

            row["nf_high_fraction"] = float(tr["nf_high_fraction"]) if "nf_high_fraction" in tr and pd.notna(tr["nf_high_fraction"]) else np.nan
            row["nf_green_success"] = int(tr["nf_green_success"]) if "nf_green_success" in tr and pd.notna(tr["nf_green_success"]) else np.nan

            row["risk_after_explosion"] = float(tr.get("risk_after_explosion", np.nan)) if pd.notna(tr.get("risk_after_explosion", np.nan)) else np.nan
            row["risk_after_collect"] = float(tr.get("risk_after_collect", np.nan)) if pd.notna(tr.get("risk_after_collect", np.nan)) else np.nan
            row["pump_speed_change"] = float(tr.get("pump_speed_change", np.nan)) if pd.notna(tr.get("pump_speed_change", np.nan)) else np.nan

            row["pump_number"] = int(p)
            row["pmpID"] = f"{bits['sub']}_{bits['ses']}_T{int(row['tID'])}_P{p}" if pd.notna(row["tID"]) else f"{bits['sub']}_{bits['ses']}_P{p}"
            row["pmpL"] = float(lat_list[p-1]) if isinstance(lat_list, list) and len(lat_list) >= p else np.nan

            row["ses_mTheta"] = np.nan
            row["ses_SD_mThetaB"] = np.nan
            row["ses_avgP300_All"] = np.nan
            row["ses_avgP300_Fz"] = np.nan
            row["ses_avgP300_FCz"] = np.nan
            row["ses_avgP300_Cz"] = np.nan
            row["ses_avgP300_Pz"] = np.nan

            row["pre200_pop_Theta"] = np.nan
            row["post200_pop_Theta"] = np.nan
            row["pre200_collect_Theta"] = np.nan
            row["post200_collect_Theta"] = np.nan
            row["pre200_pmp_Theta"] = np.nan

            row["theta_before_pump"] = np.nan
            row["theta_before_pump_Fp1"] = np.nan
            row["theta_before_pump_Fp2"] = np.nan
            row["frontal_asymmetry"] = np.nan

            row["trial_mTheta"] = np.nan
            row["trial_SD_mThetaB"] = np.nan

            pump_rows.append(row)

    pump_df = pd.DataFrame(pump_rows)
    return pump_df, trials, summary

# ----------------------------
# Pair processing
# ----------------------------
def process_pair(beh_fp: Path, xdf_fp: Path | None):
    pump_df, trials_df, summary_df = beh_to_pump_table(beh_fp)
    if pump_df is None or len(pump_df) == 0:
        return None

    if xdf_fp is None:
        return pump_df

    try:
        streams, _ = pyxdf.load_xdf(str(xdf_fp))
        eeg_stream, marker_stream = pick_streams(streams)
        if eeg_stream is None or marker_stream is None:
            return pump_df

        raw, eeg_t = build_raw_from_xdf_eeg(eeg_stream)
        raw = preprocess_raw(raw)
        raw.resample(TARGET_SFREQ, verbose=False)
        eeg_t0 = float(eeg_t[0])

        md = markers_to_df(marker_stream)

        ses_mTheta, ses_SD_mThetaB = extract_session_theta_metrics(raw)
        p300 = extract_p300_metrics(raw, md, eeg_t0)
        ev_theta = extract_event_theta_windows(raw, md, eeg_t0)

        pump_theta = theta_before_each_pump(raw, md, eeg_t0)
        trial_theta = trial_theta_from_markers(raw, md, eeg_t0)

        pump_df["ses_mTheta"] = ses_mTheta
        pump_df["ses_SD_mThetaB"] = ses_SD_mThetaB
        for k, v in p300.items():
            pump_df[k] = v
        for k, v in ev_theta.items():
            pump_df[k] = v

        pump_df["tID"] = pd.to_numeric(pump_df["tID"], errors="coerce").astype("Int64")
        pump_df["pump_number"] = pd.to_numeric(pump_df["pump_number"], errors="coerce").astype("Int64")

        if trial_theta is not None and len(trial_theta) > 0:
            tt = trial_theta.copy()
            tt["tID"] = pd.to_numeric(tt["trial"], errors="coerce").astype("Int64")
            tt = tt.drop(columns=["trial"], errors="ignore")

            for c in ["trial_mTheta", "trial_SD_mThetaB"]:
                if c in pump_df.columns:
                    pump_df = pump_df.drop(columns=[c])

            pump_df = pump_df.merge(tt, on="tID", how="left")

        if pump_theta is not None and len(pump_theta) > 0:
            pt = pump_theta.copy()
            pt["tID"] = pd.to_numeric(pt["trial"], errors="coerce").astype("Int64")
            pt["pump_number"] = pd.to_numeric(pt["pump_number"], errors="coerce").astype("Int64")
            pt = pt.drop(columns=["trial"], errors="ignore")

            for c in ["theta_before_pump", "theta_before_pump_Fp1", "theta_before_pump_Fp2", "frontal_asymmetry"]:
                if c in pump_df.columns:
                    pump_df = pump_df.drop(columns=[c])

            pump_df = pump_df.merge(pt, on=["tID", "pump_number"], how="left")

        for c in ["trial_mTheta","trial_SD_mThetaB","theta_before_pump","theta_before_pump_Fp1","theta_before_pump_Fp2","frontal_asymmetry"]:
            if c not in pump_df.columns:
                pump_df[c] = np.nan

        return pump_df

    except Exception as e:
        print("⚠️ XDF failed:", xdf_fp.name, "->", e)
        return pump_df

# ----------------------------
# Main: find files + build master
# ----------------------------
beh_files = sorted(ROOT.rglob("*_beh.xlsx"))
xdf_files = sorted(ROOT.rglob("*.xdf")) + sorted(ROOT.rglob("*.xdfz"))

print("Found BEH files:", len(beh_files))
print("Found XDF files:", len(xdf_files))

xdf_map = {}
for xf in xdf_files:
    key = xf.name.split("_task")[0]
    xdf_map.setdefault(key, []).append(xf)

all_rows = []
missing_xdf = 0
processed = 0

for bf in beh_files:
    key = bf.name.split("_task")[0]
    xmatch = xdf_map.get(key, [])
    xdf_fp = xmatch[0] if len(xmatch) else None
    if xdf_fp is None:
        missing_xdf += 1

    print("\n---")
    print("BEH:", bf.name)
    print("XDF:", xdf_fp.name if xdf_fp else "MISSING (BEH-only)")

    out = process_pair(bf, xdf_fp)
    if out is not None and len(out) > 0:
        all_rows.append(out)
        processed += 1
        print("Rows:", out.shape)

if len(all_rows) == 0:
    raise RuntimeError("No output rows produced. Check BEH files contain Trials + pump_count > 0.")

master = pd.concat(all_rows, ignore_index=True)
print("\n✅ sessions processed:", processed)
print("⚠️ missing XDF sessions:", missing_xdf)
print("✅ master shape:", master.shape)

# Participant total points
if "pID" in master.columns and "sesPoints" in master.columns and "sID" in master.columns:
    master["pID"] = normalize_pid_series(master["pID"])
    tmp = master[["pID","sID","sesPoints"]].drop_duplicates()
    totals = tmp.groupby("pID")["sesPoints"].sum()
    master["pPoints"] = master["pID"].map(totals)

# ----------------------------
# Build participant-level survey table from session 5 BEH files
# ----------------------------
survey_rows = []

for bf in beh_files:
    bits = parse_bids_bits(bf.name)
    if pd.isna(bits["ses_num"]) or int(bits["ses_num"]) != 5:
        continue

    sheets = read_beh_sheets(bf)
    survey_df = get_sheet(sheets, ["survey", "surveys"])
    if survey_df is None or len(survey_df) == 0:
        continue

    participant_id = normalize_pid_series(pd.Series([bits["sub"]]))[0]
    survey_row = extract_survey_row(survey_df, participant_id)
    if survey_row is not None:
        survey_rows.append(survey_row)

survey_master = pd.DataFrame(survey_rows) if len(survey_rows) else pd.DataFrame(columns=["pID"])

if len(survey_master):
    survey_master["pID"] = normalize_pid_series(survey_master["pID"])
    survey_master = survey_master.drop_duplicates(subset=["pID"], keep="first")
    print("Survey rows loaded:", survey_master.shape)
else:
    print("No survey rows found.")

# ----------------------------
# Merge only high/low from manifest
# ----------------------------
if manifest is not None and "pID" in manifest.columns:
    master["pID"] = normalize_pid_series(master["pID"])
    manifest["pID"] = normalize_pid_series(manifest["pID"])

    master = master.merge(manifest[["pID", "risk_group"]], on="pID", how="left")

    print("Non-missing risk_group rows after merge:", int(master["risk_group"].notna().sum()))
    print(master[["pID", "risk_group"]].dropna().drop_duplicates().head(10))

# Merge survey responses
if len(survey_master):
    master = master.merge(survey_master, on="pID", how="left")

# ----------------------------
# Ensure final schema columns exist
# ----------------------------
schema_cols = [
    "pID","risk_group","cond","pPoints","sID","ses_num",
    "sesPoints","ses_mTheta","ses_SD_mThetaB",
    "ses_avgP300_All","ses_avgP300_Fz","ses_avgP300_FCz","ses_avgP300_Cz","ses_avgP300_Pz",
    "smPmp","SD_smPmp","smADJ_pmp","SD_smADJ_pmp",
    "exp_rate","exp_N","bonus_N","collect_N","collect_rate",
    "pretask_EOB_avgTheta","posttask_EOB_avgTheta",
    "pretask_ECB_avgTheta","posttask_ECB_avgTheta",
    "pretask_CB_avgTheta","posttask_CB_avgTheta",
    "tID","trial_mTheta","trial_SD_mThetaB",
    "pop","collect","nf_high_fraction","nf_green_success","t_earned","t_loss",
    "collectL_trial","collectL_ready",
    "pmp_count",
    "pre200_pop_Theta","post200_pop_Theta","pre200_collect_Theta","post200_collect_Theta","pre200_pmp_Theta",
    "risk_after_explosion","risk_after_collect","pump_speed_change",
    "pump_number","pmpID","pmpL",
    "mpmpL","SD_mpmpL","medpmpL","SD_medpmpL",
    "theta_before_pump","theta_before_pump_Fp1","theta_before_pump_Fp2","frontal_asymmetry",
]

for c in schema_cols:
    if c not in master.columns:
        master[c] = np.nan

# Reorder: pID, survey columns, schema, rest
survey_cols = [c for c in master.columns if str(c).lower().startswith("survey_")]
base_order = [c for c in schema_cols if c in master.columns and c != "pID"]
remaining = [c for c in master.columns if c not in (["pID"] + survey_cols + base_order)]

ordered = ["pID"] + survey_cols + base_order + remaining
master = master[ordered]

print("Survey columns found:", len(survey_cols))
print("Unique risk_group values:", sorted(master["risk_group"].dropna().astype(str).unique().tolist()) if "risk_group" in master.columns else [])

# Save
csv_path = OUT / "BART_master_pump_level.csv"
xlsx_path = OUT / "BART_master_pump_level.xlsx"
master.to_csv(csv_path, index=False)
master.to_excel(xlsx_path, index=False)

print("\n✅ Saved CSV:", csv_path)
print("✅ Saved XLSX:", xlsx_path)

master.head(10)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 28.1 MB/s eta 0:00:00
Loaded manifest: /content/manifest.xlsx
Manifest rows: (49, 2)
    pID risk_group
0  P001          H
1  P002          H
2  P003          H
3  P004          H
4  P006          H
Found BEH files: 230
Found XDF files: 227

---
BEH: sub-P001_ses-S001_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P001_ses-S001_task-Default_run-001_eeg.xdf
Rows: (536, 60)

---
BEH: sub-P001_ses-S002_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P001_ses-S002_task-Default_run-001_eeg.xdf
Rows: (691, 60)

---
BEH: sub-P001_ses-S003_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P001_ses-S003_task-Default_run-001_eeg.xdf
Rows: (688, 60)

---
BEH: sub-P001_ses-S004_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: MISSING (BEH-only)
Rows: (562, 60)

---
BEH: sub-P001_ses-S005_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P001_ses-S005_task-Default_run-001_eeg.xdf
Rows: (678, 60)

---
BEH: sub-P003_ses-S001_task-BART_cond-nf_sham_r

Rows: (786, 60)

---
BEH: sub-P005_ses-S001_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P005_ses-S001_task-Default_run-001_eeg.xdf
Rows: (633, 60)

---
BEH: sub-P005_ses-S002_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P005_ses-S002_task-Default_run-001_eeg.xdf
Rows: (580, 60)

---
BEH: sub-P005_ses-S003_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P005_ses-S003_task-Default_run-001_eeg.xdf
Rows: (618, 60)

---
BEH: sub-P005_ses-S004_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P005_ses-S004_task-Default_run-001_eeg.xdf
Rows: (621, 60)

---
BEH: sub-P005_ses-S005_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P005_ses-S005_task-Default_run-001_eeg.xdf
Rows: (574, 60)

---
BEH: sub-P006_ses-S001_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P006_ses-S001_task-Default_run-001_eeg.xdf
Rows: (507, 60)

---
BEH: sub-P006_ses-S002_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P006_ses-S002_task-Default_run-001_eeg.xdf
Rows: (527, 60)

---
BEH: sub-P006_ses-S003_task

ERROR:pyxdf.pyxdf:found likely XDF file corruption (unpack requires a buffer of 8 bytes), scanning forward to next boundary chunk.


Rows: (719, 60)

---
BEH: sub-P016_ses-S005_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P016_ses-S005_task-Default_run-001_eeg.xdf


ERROR:pyxdf.pyxdf:found likely XDF file corruption (unpack requires a buffer of 8 bytes), scanning forward to next boundary chunk.


Rows: (782, 60)

---
BEH: sub-P017_ses-S001_task-BART_cond-nf_sham_run-001_beh.xlsx
XDF: sub-P017_ses-S001_task-Default_run-001_eeg.xdf


... (13 outputs left out)

Rows: (868, 60)

---
BEH: sub-P036_ses-S005_task-BART_cond-nf_sham_run-001_beh.xlsx
XDF: sub-P036_ses-S005_task-Default_run-001_eeg.xdf


ERROR:pyxdf.pyxdf:found likely XDF file corruption (unpack requires a buffer of 8 bytes), scanning forward to next boundary chunk.


Rows: (761, 60)

---
BEH: sub-P037_ses-S001_task-BART_cond-nf_sham_run-001_beh.xlsx
XDF: sub-P037_ses-S001_task-Default_run-001_eeg.xdf
Rows: (584, 60)

---
BEH: sub-P037_ses-S002_task-BART_cond-nf_sham_run-001_beh.xlsx
XDF: sub-P037_ses-S002_task-Default_run-001_eeg.xdf
Rows: (730, 60)

---
BEH: sub-P037_ses-S003_task-BART_cond-nf_sham_run-001_beh.xlsx
XDF: sub-P037_ses-S003_task-Default_run-001_eeg.xdf
Rows: (618, 60)

---
BEH: sub-P037_ses-S004_task-BART_cond-nf_sham_run-001_beh.xlsx
XDF: sub-P037_ses-S004_task-Default_run-001_eeg.xdf
Rows: (638, 60)

---
BEH: sub-P037_ses-S005_task-BART_cond-nf_sham_run-001_beh.xlsx
XDF: sub-P037_ses-S005_task-Default_run-001_eeg.xdf
Rows: (559, 60)

---
BEH: sub-P038_ses-S001_task-BART_cond-nf_sham_run-001_beh.xlsx
XDF: sub-P038_ses-S001_task-Default_run-001_eeg.xdf
Rows: (549, 60)

---
BEH: sub-P038_ses-S002_task-BART_cond-nf_sham_run-001_beh.xlsx
XDF: sub-P038_ses-S002_task-Default_run-001_eeg.xdf
Rows: (544, 60)

---
BEH: sub-P038_ses-S003_task

KeyError: 'risk_group'

In [ ]:
# ============================================================
# BART MULTIMODAL MASTER PIPELINE (Pump-level)
# Input: *_beh.xlsx + matching *.xdf/*.xdfz + manifest.xlsx
# Output: outputs/BART_master_pump_level.{csv,xlsx}
#
# Includes:
#  - Behavioral pump/trial/session metrics
#  - EEG theta + P300 metrics from XDF
#  - risk_group (H/L) from manifest.xlsx column "high/low"
#  - Survey responses from session 5 BEH survey sheet
# ============================================================

!pip -q install pyxdf mne openpyxl pandas numpy

import re, json
from pathlib import Path
import numpy as np
import pandas as pd
import pyxdf
import mne

# ----------------------------
# CONFIG
# ----------------------------
ROOT = Path("/content") if Path("/content").exists() else Path(".")
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)

TARGET_SFREQ = 512.0

# ERP epoching (explode)
ERP_TMIN, ERP_TMAX = -0.2, 0.8
ERP_BASELINE = (ERP_TMIN, 0.0)
P3_WIN = (0.300, 0.450)

# Theta windows
WIN200 = 0.200
THETA_BAND = (4.0, 7.0)

# ActiCAP channel order
EEG_ORDER = ["Fz","Cz","Pz","POz","Fp1","Fp2","F3","F4","FCz","C3","C4","CPz","P3","P4","O1","O2"]
FRONTAL_THETA_CHS = ["Fp1","Fp2","Fz","F3","F4","FCz"]

# ----------------------------
# ID normalization
# ----------------------------
def normalize_pid_series(s):
    s = s.astype(str).str.strip().str.upper()
    nums = s.str.extract(r"(\d+)")[0]
    out = nums.fillna("").apply(lambda x: f"P{int(x):03d}" if x != "" else np.nan)
    return out.where(out.notna(), s)

# ----------------------------
# Manifest: only high/low
# ----------------------------
manifest = None
manifest_path = next(iter(ROOT.rglob("manifest.xlsx")), None)

if manifest_path is not None and manifest_path.exists():
    manifest_raw = pd.read_excel(manifest_path)
    manifest_raw.columns = [str(c).strip() for c in manifest_raw.columns]

    cols_l = {str(c).strip().lower(): c for c in manifest_raw.columns}
    pid_col = None
    risk_col = None

    for k in ["subject_id", "participant_id", "participant", "subject", "sub", "pid", "id"]:
        if k in cols_l:
            pid_col = cols_l[k]
            break

    for k in ["high/low", "risk_group", "risk", "group"]:
        if k in cols_l:
            risk_col = cols_l[k]
            break

    if pid_col is not None and risk_col is not None:
        manifest = manifest_raw[[pid_col, risk_col]].copy()
        manifest.columns = ["pID", "risk_group"]
        manifest["pID"] = normalize_pid_series(manifest["pID"])
        manifest["risk_group"] = manifest["risk_group"].astype(str).str.strip().str.upper()
        manifest = manifest.drop_duplicates(subset=["pID"], keep="first")
        print("Loaded manifest:", manifest_path)
        print("Manifest rows:", manifest.shape)
        print(manifest.head())
    else:
        print("manifest.xlsx found, but could not detect subject_id and/or high/low columns.")
else:
    print("No manifest.xlsx found.")

# ----------------------------
# Utilities
# ----------------------------
def parse_bids_bits(name: str):
    base = Path(name).name
    sub = re.search(r"sub-([A-Za-z0-9]+)", base)
    ses = re.search(r"ses-([A-Za-z0-9]+)", base)
    run = re.search(r"run-([0-9]+)", base)
    task = re.search(r"task-([A-Za-z0-9]+)", base)
    cond = re.search(r"cond-([A-Za-z0-9_]+)", base)

    out = {
        "sub": sub.group(1) if sub else None,
        "ses": ses.group(1) if ses else None,
        "run": run.group(1) if run else None,
        "task": task.group(1) if task else None,
        "condition": cond.group(1) if cond else None,
    }
    if out["ses"] is not None:
        m = re.search(r"(\d+)", out["ses"])
        out["ses_num"] = int(m.group(1)) if m else np.nan
    else:
        out["ses_num"] = np.nan
    return out

def read_beh_sheets(fp: Path):
    xls = pd.ExcelFile(fp)
    sheets = {}
    for sh in xls.sheet_names:
        df = pd.read_excel(fp, sheet_name=sh)
        df.columns = [str(c).strip() for c in df.columns]
        sheets[sh.lower()] = df
    return sheets

def get_sheet(sheets: dict, candidates):
    for c in candidates:
        if c in sheets:
            return sheets[c]
    return None

def safe_json_list(x):
    try:
        return json.loads(x) if isinstance(x, str) else (x if isinstance(x, list) else None)
    except Exception:
        return None

def extract_survey_row(survey_df, participant_id):
    if survey_df is None or len(survey_df) == 0:
        return None

    df = survey_df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    participant_id = normalize_pid_series(pd.Series([participant_id]))[0]

    if len(df) == 1:
        row = df.iloc[0].to_dict()
        row["pID"] = participant_id
        return row

    cols_l = {str(c).strip().lower(): c for c in df.columns}
    q_col = None
    r_col = None

    for k in ["question", "item", "prompt"]:
        if k in cols_l:
            q_col = cols_l[k]
            break
    for k in ["response", "answer", "value", "score"]:
        if k in cols_l:
            r_col = cols_l[k]
            break

    if q_col is not None and r_col is not None:
        out = {"pID": participant_id}
        for _, r in df.iterrows():
            q = str(r[q_col]).strip()
            q_clean = re.sub(r"[^A-Za-z0-9]+", "_", q).strip("_").lower()
            out[f"survey_{q_clean}"] = r[r_col]
        return out

    out = {"pID": participant_id}
    for i, (_, r) in enumerate(df.iterrows(), start=1):
        for c in df.columns:
            cname = re.sub(r"[^A-Za-z0-9]+", "_", str(c)).strip("_").lower()
            out[f"survey_r{i}_{cname}"] = r[c]
    return out

# ----------------------------
# XDF helpers
# ----------------------------
def parse_marker_message(msg):
    s = str(msg)
    parts = s.split(";")
    out = {"code": parts[0].strip()}
    for p in parts[1:]:
        if "=" in p:
            k, v = p.split("=", 1)
            out[k.strip()] = v.strip()
    return out

def _stream_name_lower(s):
    info = s.get("info", {})
    nm = info.get("name", "")
    if isinstance(nm, (list, tuple, np.ndarray)):
        nm = nm[0] if len(nm) else ""
    return str(nm).strip().lower()

def pick_streams(streams):
    eeg = None
    best = -1

    for s in streams:
        ts = s.get("time_series", None)
        if ts is None:
            continue
        arr = np.asarray(ts)
        ch = 1 if arr.ndim == 1 else arr.shape[1]
        if ch > best:
            eeg = s
            best = ch

    markers = None
    for s in streams:
        ts = s.get("time_series", None)
        if ts is None:
            continue
        arr = np.asarray(ts)
        ch = 1 if arr.ndim == 1 else arr.shape[1]
        if ch != 1:
            continue
        nm = _stream_name_lower(s)
        if ("bart" in nm) or ("marker" in nm) or ("markers" in nm) or ("event" in nm):
            markers = s
            break

    if markers is None:
        for s in streams:
            ts = s.get("time_series", None)
            if ts is None:
                continue
            arr = np.asarray(ts)
            ch = 1 if arr.ndim == 1 else arr.shape[1]
            if ch == 1:
                markers = s
                break

    return eeg, markers

def markers_to_df(marker_stream):
    m_ts = np.asarray(marker_stream["time_stamps"], dtype=float)
    m_val = np.asarray(marker_stream["time_series"], dtype=object)

    rows = []
    for t, v in zip(m_ts, m_val):
        msg = v[0] if isinstance(v, (list, tuple, np.ndarray)) else v
        d = parse_marker_message(msg)
        d["lsltime"] = float(t)
        rows.append(d)

    df = pd.DataFrame(rows)
    for k in ["trial", "pump", "loss", "earnings", "total"]:
        if k in df.columns:
            df[k] = pd.to_numeric(df[k], errors="coerce")
    return df

def build_raw_from_xdf_eeg(eeg_stream):
    data = np.asarray(eeg_stream["time_series"])
    t = np.asarray(eeg_stream["time_stamps"], dtype=float)
    if data.ndim == 1:
        data = data[:, None]
    n_samp, n_ch = data.shape

    sfreq_est = 1.0 / np.median(np.diff(t))
    ch_names = EEG_ORDER[:n_ch] if n_ch <= len(EEG_ORDER) else [f"EEG{ii+1:02d}" for ii in range(n_ch)]
    arr = data.T.astype(float)

    peak = float(np.nanmax(np.abs(arr)))
    if peak > 1e3:
        arr *= 1e-6
    elif peak > 5.0:
        arr *= 1e-3

    info = mne.create_info(ch_names=ch_names, sfreq=sfreq_est, ch_types=["eeg"] * n_ch)
    raw = mne.io.RawArray(arr, info, verbose=False)
    return raw, t

def preprocess_raw(raw):
    raw = raw.copy()
    raw.filter(0.1, 30.0, picks="eeg", verbose=False)
    try:
        raw.notch_filter([60], picks="eeg", verbose=False)
    except Exception:
        pass
    try:
        raw.set_eeg_reference("average", verbose=False)
    except Exception:
        pass
    return raw

def lsltime_to_sample(lsltime, eeg_t0, sfreq):
    return int(np.round((lsltime - eeg_t0) * sfreq))

# ----------------------------
# EEG feature extraction
# ----------------------------
def compute_theta_power_window(raw_theta, sample, tmin_s, tmax_s, picks):
    sf = raw_theta.info["sfreq"]
    start = int(sample + np.round(tmin_s * sf))
    stop  = int(sample + np.round(tmax_s * sf))
    start = max(start, 0)
    stop  = min(stop, raw_theta.n_times)
    if stop <= start + 2:
        return np.nan
    seg = raw_theta.get_data(picks=picks, start=start, stop=stop)
    return float(np.mean(seg**2))

def extract_session_theta_metrics(raw):
    rtheta = raw.copy().filter(THETA_BAND[0], THETA_BAND[1], picks="eeg", verbose=False)
    picks = [raw.ch_names.index(ch) for ch in FRONTAL_THETA_CHS if ch in raw.ch_names]
    if len(picks) == 0:
        return np.nan, np.nan
    X = rtheta.get_data(picks=picks)
    theta_ts = np.mean(X**2, axis=0)
    return float(np.mean(theta_ts)), float(np.std(theta_ts, ddof=1))

def extract_p300_metrics(raw, md, eeg_t0):
    keys = ["ses_avgP300_All","ses_avgP300_Fz","ses_avgP300_FCz","ses_avgP300_Cz","ses_avgP300_Pz"]
    if "code" not in md.columns:
        return {k: np.nan for k in keys}

    exp = md[md["code"].astype(str).str.upper().eq("BART_EXPLODE")].copy()
    if len(exp) == 0:
        return {k: np.nan for k in keys}

    sf = raw.info["sfreq"]
    exp = exp.dropna(subset=["lsltime"]).copy()
    exp["sample"] = exp["lsltime"].apply(lambda t: lsltime_to_sample(float(t), eeg_t0, sf)).astype(int)

    min_samp = int(np.ceil((-ERP_TMIN) * sf))
    exp = exp[(exp["sample"] >= min_samp) & (exp["sample"] < raw.n_times)].copy()
    if len(exp) == 0:
        return {k: np.nan for k in keys}

    events = np.c_[exp["sample"].to_numpy(), np.zeros(len(exp), int), np.ones(len(exp), int)]
    epochs = mne.Epochs(
        raw, events, event_id={"EXPLODE": 1},
        tmin=ERP_TMIN, tmax=ERP_TMAX, baseline=ERP_BASELINE,
        preload=True, reject_by_annotation=True, verbose=False
    )
    if len(epochs) == 0:
        return {k: np.nan for k in keys}

    ev = epochs.average()
    times = ev.times
    m1 = np.argmin(np.abs(times - P3_WIN[0]))
    m2 = np.argmin(np.abs(times - P3_WIN[1]))

    out = {}
    vals = []
    for ch in ["Fz","FCz","Cz","Pz"]:
        k = f"ses_avgP300_{ch}"
        if ch in ev.ch_names:
            idx = ev.ch_names.index(ch)
            erp_uV = ev.data[idx] * 1e6
            p3 = float(np.max(erp_uV[m1:m2]))
            out[k] = p3
            vals.append(p3)
        else:
            out[k] = np.nan
    out["ses_avgP300_All"] = float(np.nanmean(vals)) if len(vals) else np.nan
    return out

def extract_event_theta_windows(raw, md, eeg_t0):
    rtheta = raw.copy().filter(THETA_BAND[0], THETA_BAND[1], picks="eeg", verbose=False)
    sf = raw.info["sfreq"]
    picks = [raw.ch_names.index(ch) for ch in FRONTAL_THETA_CHS if ch in raw.ch_names]
    if len(picks) == 0:
        return {
            "pre200_pop_Theta": np.nan, "post200_pop_Theta": np.nan,
            "pre200_collect_Theta": np.nan, "post200_collect_Theta": np.nan,
            "pre200_pmp_Theta": np.nan,
        }

    def _mean_prepost(code, pre=True):
        d = md[md["code"].astype(str).str.upper().eq(code)].dropna(subset=["lsltime"]).copy()
        if len(d) == 0:
            return np.nan
        d["sample"] = d["lsltime"].apply(lambda t: lsltime_to_sample(float(t), eeg_t0, sf)).astype(int)
        vals = []
        for s in d["sample"].tolist():
            v = compute_theta_power_window(rtheta, s, -WIN200, 0.0, picks) if pre else compute_theta_power_window(rtheta, s, 0.0, WIN200, picks)
            vals.append(v)
        return float(np.nanmean(vals)) if len(vals) else np.nan

    return {
        "pre200_pop_Theta": _mean_prepost("BART_EXPLODE", pre=True),
        "post200_pop_Theta": _mean_prepost("BART_EXPLODE", pre=False),
        "pre200_collect_Theta": _mean_prepost("BART_COLLECT", pre=True),
        "post200_collect_Theta": _mean_prepost("BART_COLLECT", pre=False),
        "pre200_pmp_Theta": _mean_prepost("BART_PUMP", pre=True),
    }

def theta_before_each_pump(raw, md, eeg_t0):
    pumps = md[md["code"].astype(str).str.upper().eq("BART_PUMP")].copy()
    if len(pumps) == 0 or ("trial" not in pumps.columns) or ("pump" not in pumps.columns):
        return pd.DataFrame(columns=["trial","pump_number","theta_before_pump","theta_before_pump_Fp1","theta_before_pump_Fp2","frontal_asymmetry"])

    rtheta = raw.copy().filter(THETA_BAND[0], THETA_BAND[1], picks="eeg", verbose=False)
    sf = raw.info["sfreq"]

    pumps["trial"] = pd.to_numeric(pumps["trial"], errors="coerce")
    pumps["pump_number"] = pd.to_numeric(pumps["pump"], errors="coerce")
    pumps = pumps.dropna(subset=["trial","pump_number","lsltime"]).copy()

    pumps["trial"] = pumps["trial"].astype(int)
    pumps["pump_number"] = pumps["pump_number"].astype(int)
    pumps["sample"] = pumps["lsltime"].apply(lambda t: lsltime_to_sample(float(t), eeg_t0, sf)).astype(int)

    min_samp = int(np.ceil(WIN200 * sf))
    pumps = pumps[(pumps["sample"] >= min_samp) & (pumps["sample"] < raw.n_times)].copy()
    if len(pumps) == 0:
        return pd.DataFrame(columns=["trial","pump_number","theta_before_pump","theta_before_pump_Fp1","theta_before_pump_Fp2","frontal_asymmetry"])

    frontal_picks = [raw.ch_names.index(ch) for ch in FRONTAL_THETA_CHS if ch in raw.ch_names]
    fp1_pick = [raw.ch_names.index("Fp1")] if "Fp1" in raw.ch_names else []
    fp2_pick = [raw.ch_names.index("Fp2")] if "Fp2" in raw.ch_names else []

    rows = []
    for _, r in pumps.iterrows():
        s = int(r["sample"])
        pre_frontal = compute_theta_power_window(rtheta, s, -WIN200, 0.0, frontal_picks) if frontal_picks else np.nan
        pre_fp1 = compute_theta_power_window(rtheta, s, -WIN200, 0.0, fp1_pick) if fp1_pick else np.nan
        pre_fp2 = compute_theta_power_window(rtheta, s, -WIN200, 0.0, fp2_pick) if fp2_pick else np.nan
        fa = (np.log(pre_fp2 + 1e-12) - np.log(pre_fp1 + 1e-12)) if (np.isfinite(pre_fp1) and np.isfinite(pre_fp2)) else np.nan

        rows.append({
            "trial": int(r["trial"]),
            "pump_number": int(r["pump_number"]),
            "theta_before_pump": pre_frontal,
            "theta_before_pump_Fp1": pre_fp1,
            "theta_before_pump_Fp2": pre_fp2,
            "frontal_asymmetry": fa
        })

    return pd.DataFrame(rows)

def trial_theta_from_markers(raw, md, eeg_t0):
    sf = raw.info["sfreq"]
    rtheta = raw.copy().filter(THETA_BAND[0], THETA_BAND[1], picks="eeg", verbose=False)
    picks = [raw.ch_names.index(ch) for ch in FRONTAL_THETA_CHS if ch in raw.ch_names]
    if len(picks) == 0:
        return pd.DataFrame(columns=["trial","trial_mTheta","trial_SD_mThetaB"])

    starts = md[md["code"].astype(str).str.upper().eq("BART_TRIAL_START")].copy()
    ends = md[md["code"].astype(str).str.upper().isin(["BART_COLLECT","BART_EXPLODE"])].copy()
    if "trial" not in starts.columns or "trial" not in ends.columns:
        return pd.DataFrame(columns=["trial","trial_mTheta","trial_SD_mThetaB"])

    starts["trial"] = pd.to_numeric(starts["trial"], errors="coerce")
    ends["trial"] = pd.to_numeric(ends["trial"], errors="coerce")
    starts = starts.dropna(subset=["trial","lsltime"]).copy()
    ends = ends.dropna(subset=["trial","lsltime"]).copy()
    if len(starts) == 0 or len(ends) == 0:
        return pd.DataFrame(columns=["trial","trial_mTheta","trial_SD_mThetaB"])

    out = []
    for tnum, g0 in starts.groupby(starts["trial"].astype(int)):
        t_start = float(g0["lsltime"].min())
        g1 = ends[ends["trial"].astype(int) == int(tnum)]
        if len(g1) == 0:
            continue
        t_end = float(g1["lsltime"].min())
        if t_end <= t_start:
            continue

        s0 = lsltime_to_sample(t_start, eeg_t0, sf)
        s1 = lsltime_to_sample(t_end, eeg_t0, sf)
        s0 = max(s0, 0)
        s1 = min(s1, raw.n_times)
        if s1 <= s0 + int(sf * 0.2):
            continue

        seg = rtheta.get_data(picks=picks, start=s0, stop=s1)
        theta_ts = np.mean(seg**2, axis=0)

        out.append({
            "trial": int(tnum),
            "trial_mTheta": float(np.mean(theta_ts)),
            "trial_SD_mThetaB": float(np.std(theta_ts, ddof=1)),
        })

    return pd.DataFrame(out)

# ----------------------------
# BEH -> pump table
# ----------------------------
def beh_to_pump_table(beh_fp: Path):
    sheets = read_beh_sheets(beh_fp)
    trials = get_sheet(sheets, ["trials", "trial"])
    summary = get_sheet(sheets, ["summary"])

    if trials is None or len(trials) == 0:
        return None, None, None

    trials = trials.copy()
    bits = parse_bids_bits(beh_fp.name)

    if "exploded" in trials.columns:
        trials["pop"] = trials["exploded"].astype(str).str.upper().isin(["TRUE","1","YES"]).astype(int)
    else:
        trials["pop"] = 0

    if "collected" in trials.columns:
        trials["collect"] = pd.to_numeric(trials["collected"], errors="coerce").fillna(0).astype(int)
    else:
        trials["collect"] = trials["events"].astype(str).str.contains("collect", case=False).astype(int) if "events" in trials.columns else 0

    for c in [
        "trial","pump_count","trial_earnings","loss_if_pop","trial_value","total_earnings",
        "pump_latency_mean","pump_latency_median","pump_latency_first",
        "collect_latency_from_ready","collect_latency_from_trial_start",
        "nf_high_frac","nf_green_success",
        "risk_after_explosion","risk_after_collect","pump_speed_change",
        "nf_mean_normalized"
    ]:
        if c in trials.columns:
            trials[c] = pd.to_numeric(trials[c], errors="coerce")

    if "nf_high_frac" in trials.columns:
        trials["nf_high_fraction"] = pd.to_numeric(trials["nf_high_frac"], errors="coerce")
    elif "nf_high_fraction" not in trials.columns:
        trials["nf_high_fraction"] = np.nan

    if "nf_green_success" in trials.columns and trials["nf_green_success"].notna().any():
        trials["nf_green_success"] = pd.to_numeric(trials["nf_green_success"], errors="coerce")
    else:
        trials["nf_green_success"] = np.where(trials["nf_high_fraction"] > 0.75, 1, 0).astype(int) if "nf_high_fraction" in trials.columns else np.nan

    block_col = "block" if "block" in trials.columns else None
    if block_col is not None:
        trials = trials.sort_values([block_col, "trial"], kind="mergesort") if "trial" in trials.columns else trials
        g = trials.groupby(block_col)["nf_green_success"]
        trials["nf_bonus_awarded_trial"] = ((g.shift(0)==1) & (g.shift(1)==1) & (g.shift(2)==1)).astype(int)
    else:
        trials["nf_bonus_awarded_trial"] = 0

    if "pump_count" in trials.columns and block_col is not None and "trial" in trials.columns:
        trials["prev_pumps"] = trials.groupby(block_col)["pump_count"].shift(1)
        prev_pop = trials.groupby(block_col)["pop"].shift(1)
        prev_collect = trials.groupby(block_col)["collect"].shift(1)

        trials["risk_after_explosion"] = np.where(prev_pop == 1, trials["pump_count"] - trials["prev_pumps"], np.nan)
        trials["risk_after_collect"] = np.where(prev_collect == 1, trials["pump_count"] - trials["prev_pumps"], np.nan)

        if "pump_latency_mean" in trials.columns:
            trials["pump_speed_change"] = trials["pump_latency_mean"] - trials.groupby(block_col)["pump_latency_mean"].shift(1)
        else:
            trials["pump_speed_change"] = np.nan
    else:
        if "risk_after_explosion" not in trials.columns:
            trials["risk_after_explosion"] = np.nan
        if "risk_after_collect" not in trials.columns:
            trials["risk_after_collect"] = np.nan
        if "pump_speed_change" not in trials.columns:
            trials["pump_speed_change"] = np.nan

    sesPoints = trials["total_earnings"].dropna().max() if "total_earnings" in trials.columns else np.nan
    exp_N = int(trials["pop"].sum())
    collect_N = int(trials["collect"].sum())
    tN = int(len(trials))
    exp_rate = exp_N / tN if tN else np.nan
    collect_rate = collect_N / tN if tN else np.nan
    bonus_N = int(trials["nf_bonus_awarded_trial"].sum()) if "nf_bonus_awarded_trial" in trials.columns else 0

    smPmp = float(trials["pump_count"].mean()) if "pump_count" in trials.columns else np.nan
    SD_smPmp = float(trials["pump_count"].std(ddof=1)) if "pump_count" in trials.columns else np.nan

    trials["adjusted_pumps_trial"] = trials["pump_count"] if "pump_count" in trials.columns else np.nan
    smADJ_pmp = float(trials["adjusted_pumps_trial"].mean()) if "adjusted_pumps_trial" in trials.columns else np.nan
    SD_smADJ_pmp = float(trials["adjusted_pumps_trial"].std(ddof=1)) if "adjusted_pumps_trial" in trials.columns else np.nan

    mpmpL = float(trials["pump_latency_mean"].mean()) if "pump_latency_mean" in trials.columns else np.nan
    SD_mpmpL = float(trials["pump_latency_mean"].std(ddof=1)) if "pump_latency_mean" in trials.columns else np.nan
    medpmpL = float(trials["pump_latency_median"].mean()) if "pump_latency_median" in trials.columns else np.nan
    SD_medpmpL = float(trials["pump_latency_median"].std(ddof=1)) if "pump_latency_median" in trials.columns else np.nan

    def _get_scalar_from_sources(keys, sources):
        for df in sources:
            if df is None or len(df) == 0:
                continue
            cols_lut = {str(c).strip().lower(): c for c in df.columns}
            for k in keys:
                kk = str(k).strip().lower()
                if kk in cols_lut:
                    col = cols_lut[kk]
                    ser = pd.to_numeric(df[col], errors="coerce")
                    if ser.notna().any():
                        return float(ser.dropna().iloc[0])
        return np.nan

    sources = [trials, summary]

    baseline_cols = {
        "baseline_eo_mu": _get_scalar_from_sources(
            ["baseline_eo_mu", "baseline_pre_eo_mu", "pretask_eob_avgtheta", "pretask_eo_avgtheta"], sources),
        "baseline_ec_mu": _get_scalar_from_sources(
            ["baseline_ec_mu", "baseline_pre_ec_mu", "pretask_ecb_avgtheta", "pretask_ec_avgtheta"], sources),
        "baseline_conc_mu": _get_scalar_from_sources(
            ["baseline_conc_mu", "baseline_pre_conc_mu", "pretask_cb_avgtheta", "pretask_conc_avgtheta"], sources),

        "baseline_post_eo_mu": _get_scalar_from_sources(
            ["baseline_post_eo_mu", "posttask_eob_avgtheta", "post_eob_avgtheta", "posttask_eo_avgtheta"], sources),
        "baseline_post_ec_mu": _get_scalar_from_sources(
            ["baseline_post_ec_mu", "posttask_ecb_avgtheta", "post_ecb_avgtheta", "posttask_ec_avgtheta"], sources),
        "baseline_post_conc_mu": _get_scalar_from_sources(
            ["baseline_post_conc_mu", "posttask_cb_avgtheta", "post_cb_avgtheta", "posttask_conc_avgtheta"], sources),
    }

    pump_rows = []
    for _, tr in trials.iterrows():
        n = int(tr["pump_count"]) if pd.notna(tr.get("pump_count", np.nan)) else 0
        if n <= 0:
            continue

        lat_list = safe_json_list(tr.get("pump_latencies_json", None))

        for p in range(1, n + 1):
            row = {}

            row["pID"] = normalize_pid_series(pd.Series([bits["sub"]]))[0]
            row["cond"] = bits["condition"]
            row["sID"] = bits["ses"]
            row["ses_num"] = bits["ses_num"]

            row["pPoints"] = np.nan
            # removed placeholder risk_group here to avoid merge suffix bug

            row["sesPoints"] = sesPoints
            row["exp_N"] = exp_N
            row["collect_N"] = collect_N
            row["bonus_N"] = bonus_N
            row["exp_rate"] = exp_rate
            row["collect_rate"] = collect_rate
            row["smPmp"] = smPmp
            row["SD_smPmp"] = SD_smPmp
            row["smADJ_pmp"] = smADJ_pmp
            row["SD_smADJ_pmp"] = SD_smADJ_pmp
            row["mpmpL"] = mpmpL
            row["SD_mpmpL"] = SD_mpmpL
            row["medpmpL"] = medpmpL
            row["SD_medpmpL"] = SD_medpmpL

            row["pretask_EOB_avgTheta"]  = baseline_cols.get("baseline_eo_mu", np.nan)
            row["posttask_EOB_avgTheta"] = baseline_cols.get("baseline_post_eo_mu", np.nan)
            row["pretask_ECB_avgTheta"]  = baseline_cols.get("baseline_ec_mu", np.nan)
            row["posttask_ECB_avgTheta"] = baseline_cols.get("baseline_post_ec_mu", np.nan)
            row["pretask_CB_avgTheta"]   = baseline_cols.get("baseline_conc_mu", np.nan)
            row["posttask_CB_avgTheta"]  = baseline_cols.get("baseline_post_conc_mu", np.nan)

            row["tID"] = int(tr["trial"]) if "trial" in tr and pd.notna(tr["trial"]) else np.nan
            row["pop"] = int(tr["pop"])
            row["collect"] = int(tr["collect"])
            row["pmp_count"] = int(tr["pump_count"]) if pd.notna(tr.get("pump_count", np.nan)) else np.nan
            row["t_earned"] = float(tr["trial_earnings"]) if "trial_earnings" in tr and pd.notna(tr["trial_earnings"]) else np.nan
            row["t_loss"] = float(tr["loss_if_pop"]) if "loss_if_pop" in tr and pd.notna(tr["loss_if_pop"]) else np.nan

            row["collectL_trial"] = float(tr["collect_latency_from_trial_start"]) if "collect_latency_from_trial_start" in tr and pd.notna(tr["collect_latency_from_trial_start"]) else np.nan
            row["collectL_ready"] = float(tr["collect_latency_from_ready"]) if "collect_latency_from_ready" in tr and pd.notna(tr["collect_latency_from_ready"]) else np.nan

            row["nf_high_fraction"] = float(tr["nf_high_fraction"]) if "nf_high_fraction" in tr and pd.notna(tr["nf_high_fraction"]) else np.nan
            row["nf_green_success"] = int(tr["nf_green_success"]) if "nf_green_success" in tr and pd.notna(tr["nf_green_success"]) else np.nan

            row["risk_after_explosion"] = float(tr.get("risk_after_explosion", np.nan)) if pd.notna(tr.get("risk_after_explosion", np.nan)) else np.nan
            row["risk_after_collect"] = float(tr.get("risk_after_collect", np.nan)) if pd.notna(tr.get("risk_after_collect", np.nan)) else np.nan
            row["pump_speed_change"] = float(tr.get("pump_speed_change", np.nan)) if pd.notna(tr.get("pump_speed_change", np.nan)) else np.nan

            row["pump_number"] = int(p)
            row["pmpID"] = f"{bits['sub']}_{bits['ses']}_T{int(row['tID'])}_P{p}" if pd.notna(row["tID"]) else f"{bits['sub']}_{bits['ses']}_P{p}"
            row["pmpL"] = float(lat_list[p-1]) if isinstance(lat_list, list) and len(lat_list) >= p else np.nan

            row["ses_mTheta"] = np.nan
            row["ses_SD_mThetaB"] = np.nan
            row["ses_avgP300_All"] = np.nan
            row["ses_avgP300_Fz"] = np.nan
            row["ses_avgP300_FCz"] = np.nan
            row["ses_avgP300_Cz"] = np.nan
            row["ses_avgP300_Pz"] = np.nan

            row["pre200_pop_Theta"] = np.nan
            row["post200_pop_Theta"] = np.nan
            row["pre200_collect_Theta"] = np.nan
            row["post200_collect_Theta"] = np.nan
            row["pre200_pmp_Theta"] = np.nan

            row["theta_before_pump"] = np.nan
            row["theta_before_pump_Fp1"] = np.nan
            row["theta_before_pump_Fp2"] = np.nan
            row["frontal_asymmetry"] = np.nan

            row["trial_mTheta"] = np.nan
            row["trial_SD_mThetaB"] = np.nan

            pump_rows.append(row)

    pump_df = pd.DataFrame(pump_rows)
    return pump_df, trials, summary

# ----------------------------
# Pair processing
# ----------------------------
def process_pair(beh_fp: Path, xdf_fp: Path | None):
    pump_df, trials_df, summary_df = beh_to_pump_table(beh_fp)
    if pump_df is None or len(pump_df) == 0:
        return None

    if xdf_fp is None:
        return pump_df

    try:
        streams, _ = pyxdf.load_xdf(str(xdf_fp))
        eeg_stream, marker_stream = pick_streams(streams)
        if eeg_stream is None or marker_stream is None:
            return pump_df

        raw, eeg_t = build_raw_from_xdf_eeg(eeg_stream)
        raw = preprocess_raw(raw)
        raw.resample(TARGET_SFREQ, verbose=False)
        eeg_t0 = float(eeg_t[0])

        md = markers_to_df(marker_stream)

        ses_mTheta, ses_SD_mThetaB = extract_session_theta_metrics(raw)
        p300 = extract_p300_metrics(raw, md, eeg_t0)
        ev_theta = extract_event_theta_windows(raw, md, eeg_t0)

        pump_theta = theta_before_each_pump(raw, md, eeg_t0)
        trial_theta = trial_theta_from_markers(raw, md, eeg_t0)

        pump_df["ses_mTheta"] = ses_mTheta
        pump_df["ses_SD_mThetaB"] = ses_SD_mThetaB
        for k, v in p300.items():
            pump_df[k] = v
        for k, v in ev_theta.items():
            pump_df[k] = v

        pump_df["tID"] = pd.to_numeric(pump_df["tID"], errors="coerce").astype("Int64")
        pump_df["pump_number"] = pd.to_numeric(pump_df["pump_number"], errors="coerce").astype("Int64")

        if trial_theta is not None and len(trial_theta) > 0:
            tt = trial_theta.copy()
            tt["tID"] = pd.to_numeric(tt["trial"], errors="coerce").astype("Int64")
            tt = tt.drop(columns=["trial"], errors="ignore")

            for c in ["trial_mTheta", "trial_SD_mThetaB"]:
                if c in pump_df.columns:
                    pump_df = pump_df.drop(columns=[c])

            pump_df = pump_df.merge(tt, on="tID", how="left")

        if pump_theta is not None and len(pump_theta) > 0:
            pt = pump_theta.copy()
            pt["tID"] = pd.to_numeric(pt["trial"], errors="coerce").astype("Int64")
            pt["pump_number"] = pd.to_numeric(pt["pump_number"], errors="coerce").astype("Int64")
            pt = pt.drop(columns=["trial"], errors="ignore")

            for c in ["theta_before_pump", "theta_before_pump_Fp1", "theta_before_pump_Fp2", "frontal_asymmetry"]:
                if c in pump_df.columns:
                    pump_df = pump_df.drop(columns=[c])

            pump_df = pump_df.merge(pt, on=["tID", "pump_number"], how="left")

        for c in ["trial_mTheta","trial_SD_mThetaB","theta_before_pump","theta_before_pump_Fp1","theta_before_pump_Fp2","frontal_asymmetry"]:
            if c not in pump_df.columns:
                pump_df[c] = np.nan

        return pump_df

    except Exception as e:
        print("⚠️ XDF failed:", xdf_fp.name, "->", e)
        return pump_df

# ----------------------------
# Main: find files + build master
# ----------------------------
beh_files = sorted(ROOT.rglob("*_beh.xlsx"))
xdf_files = sorted(ROOT.rglob("*.xdf")) + sorted(ROOT.rglob("*.xdfz"))

print("Found BEH files:", len(beh_files))
print("Found XDF files:", len(xdf_files))

xdf_map = {}
for xf in xdf_files:
    key = xf.name.split("_task")[0]
    xdf_map.setdefault(key, []).append(xf)

all_rows = []
missing_xdf = 0
processed = 0

for bf in beh_files:
    key = bf.name.split("_task")[0]
    xmatch = xdf_map.get(key, [])
    xdf_fp = xmatch[0] if len(xmatch) else None
    if xdf_fp is None:
        missing_xdf += 1

    print("\n---")
    print("BEH:", bf.name)
    print("XDF:", xdf_fp.name if xdf_fp else "MISSING (BEH-only)")

    out = process_pair(bf, xdf_fp)
    if out is not None and len(out) > 0:
        all_rows.append(out)
        processed += 1
        print("Rows:", out.shape)

if len(all_rows) == 0:
    raise RuntimeError("No output rows produced. Check BEH files contain Trials + pump_count > 0.")

master = pd.concat(all_rows, ignore_index=True)
print("\n✅ sessions processed:", processed)
print("⚠️ missing XDF sessions:", missing_xdf)
print("✅ master shape:", master.shape)

# Participant total points
if "pID" in master.columns and "sesPoints" in master.columns and "sID" in master.columns:
    master["pID"] = normalize_pid_series(master["pID"])
    tmp = master[["pID","sID","sesPoints"]].drop_duplicates()
    totals = tmp.groupby("pID")["sesPoints"].sum()
    master["pPoints"] = master["pID"].map(totals)

# ----------------------------
# Build participant-level survey table from session 5 BEH files
# ----------------------------
survey_rows = []

for bf in beh_files:
    bits = parse_bids_bits(bf.name)
    if pd.isna(bits["ses_num"]) or int(bits["ses_num"]) != 5:
        continue

    sheets = read_beh_sheets(bf)
    survey_df = get_sheet(sheets, ["survey", "surveys"])
    if survey_df is None or len(survey_df) == 0:
        continue

    participant_id = normalize_pid_series(pd.Series([bits["sub"]]))[0]
    survey_row = extract_survey_row(survey_df, participant_id)
    if survey_row is not None:
        survey_rows.append(survey_row)

survey_master = pd.DataFrame(survey_rows) if len(survey_rows) else pd.DataFrame(columns=["pID"])

if len(survey_master):
    survey_master["pID"] = normalize_pid_series(survey_master["pID"])
    survey_master = survey_master.drop_duplicates(subset=["pID"], keep="first")
    print("Survey rows loaded:", survey_master.shape)
else:
    print("No survey rows found.")

# ----------------------------
# Merge only high/low from manifest
# ----------------------------
if manifest is not None and "pID" in manifest.columns:
    master["pID"] = normalize_pid_series(master["pID"])
    manifest["pID"] = normalize_pid_series(manifest["pID"])

    # defensive: remove any existing risk_group before merge
    master = master.drop(columns=["risk_group"], errors="ignore")

    master = master.merge(manifest[["pID", "risk_group"]], on="pID", how="left")

    print("Non-missing risk_group rows after merge:", int(master["risk_group"].notna().sum()))
    print(master[["pID", "risk_group"]].dropna().drop_duplicates().head(10))

# Merge survey responses
if len(survey_master):
    master = master.merge(survey_master, on="pID", how="left")

# ----------------------------
# Ensure final schema columns exist
# ----------------------------
schema_cols = [
    "pID","risk_group","cond","pPoints","sID","ses_num",
    "sesPoints","ses_mTheta","ses_SD_mThetaB",
    "ses_avgP300_All","ses_avgP300_Fz","ses_avgP300_FCz","ses_avgP300_Cz","ses_avgP300_Pz",
    "smPmp","SD_smPmp","smADJ_pmp","SD_smADJ_pmp",
    "exp_rate","exp_N","bonus_N","collect_N","collect_rate",
    "pretask_EOB_avgTheta","posttask_EOB_avgTheta",
    "pretask_ECB_avgTheta","posttask_ECB_avgTheta",
    "pretask_CB_avgTheta","posttask_CB_avgTheta",
    "tID","trial_mTheta","trial_SD_mThetaB",
    "pop","collect","nf_high_fraction","nf_green_success","t_earned","t_loss",
    "collectL_trial","collectL_ready",
    "pmp_count",
    "pre200_pop_Theta","post200_pop_Theta","pre200_collect_Theta","post200_collect_Theta","pre200_pmp_Theta",
    "risk_after_explosion","risk_after_collect","pump_speed_change",
    "pump_number","pmpID","pmpL",
    "mpmpL","SD_mpmpL","medpmpL","SD_medpmpL",
    "theta_before_pump","theta_before_pump_Fp1","theta_before_pump_Fp2","frontal_asymmetry",
]

for c in schema_cols:
    if c not in master.columns:
        master[c] = np.nan

# Reorder: pID, survey columns, schema, rest
survey_cols = [c for c in master.columns if str(c).lower().startswith("survey_")]
base_order = [c for c in schema_cols if c in master.columns and c != "pID"]
remaining = [c for c in master.columns if c not in (["pID"] + survey_cols + base_order)]

ordered = ["pID"] + survey_cols + base_order + remaining
master = master[ordered]

print("Survey columns found:", len(survey_cols))
print("Unique risk_group values:", sorted(master["risk_group"].dropna().astype(str).unique().tolist()) if "risk_group" in master.columns else [])

# Save
csv_path = OUT / "BART_master_pump_level.csv"
xlsx_path = OUT / "BART_master_pump_level.xlsx"
master.to_csv(csv_path, index=False)
master.to_excel(xlsx_path, index=False)

print("\n✅ Saved CSV:", csv_path)
print("✅ Saved XLSX:", xlsx_path)

master.head(10)

Loaded manifest: /content/manifest.xlsx
Manifest rows: (49, 2)
    pID risk_group
0  P001          H
1  P002          H
2  P003          H
3  P004          H
4  P006          H
Found BEH files: 230
Found XDF files: 227

---
BEH: sub-P001_ses-S001_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P001_ses-S001_task-Default_run-001_eeg.xdf
Rows: (536, 59)

---
BEH: sub-P001_ses-S002_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P001_ses-S002_task-Default_run-001_eeg.xdf
Rows: (691, 59)

---
BEH: sub-P001_ses-S003_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P001_ses-S003_task-Default_run-001_eeg.xdf
Rows: (688, 59)

---
BEH: sub-P001_ses-S004_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: MISSING (BEH-only)
Rows: (562, 59)

---
BEH: sub-P001_ses-S005_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P001_ses-S005_task-Default_run-001_eeg.xdf
Rows: (678, 59)

---
BEH: sub-P003_ses-S001_task-BART_cond-nf_sham_run-001_beh.xlsx
XDF: sub-P003_ses-S001_task-Default_run-001_eeg.xdf
Rows: (30

Rows: (786, 59)

---
BEH: sub-P005_ses-S001_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P005_ses-S001_task-Default_run-001_eeg.xdf
Rows: (633, 59)

---
BEH: sub-P005_ses-S002_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P005_ses-S002_task-Default_run-001_eeg.xdf
Rows: (580, 59)

---
BEH: sub-P005_ses-S003_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P005_ses-S003_task-Default_run-001_eeg.xdf
Rows: (618, 59)

---
BEH: sub-P005_ses-S004_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P005_ses-S004_task-Default_run-001_eeg.xdf
Rows: (621, 59)

---
BEH: sub-P005_ses-S005_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P005_ses-S005_task-Default_run-001_eeg.xdf
Rows: (574, 59)

---
BEH: sub-P006_ses-S001_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P006_ses-S001_task-Default_run-001_eeg.xdf
Rows: (507, 59)

---
BEH: sub-P006_ses-S002_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P006_ses-S002_task-Default_run-001_eeg.xdf
Rows: (527, 59)

---
BEH: sub-P006_ses-S003_task

ERROR:pyxdf.pyxdf:found likely XDF file corruption (unpack requires a buffer of 8 bytes), scanning forward to next boundary chunk.


Rows: (719, 59)

---
BEH: sub-P016_ses-S005_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P016_ses-S005_task-Default_run-001_eeg.xdf


ERROR:pyxdf.pyxdf:found likely XDF file corruption (unpack requires a buffer of 8 bytes), scanning forward to next boundary chunk.


Rows: (782, 59)

---
BEH: sub-P017_ses-S001_task-BART_cond-nf_sham_run-001_beh.xlsx
XDF: sub-P017_ses-S001_task-Default_run-001_eeg.xdf


... (12 outputs left out)

ERROR:pyxdf.pyxdf:found likely XDF file corruption (unpack requires a buffer of 8 bytes), scanning forward to next boundary chunk.


Rows: (868, 59)

---
BEH: sub-P036_ses-S005_task-BART_cond-nf_sham_run-001_beh.xlsx
XDF: sub-P036_ses-S005_task-Default_run-001_eeg.xdf


ERROR:pyxdf.pyxdf:found likely XDF file corruption (unpack requires a buffer of 8 bytes), scanning forward to next boundary chunk.


Rows: (761, 59)

---
BEH: sub-P037_ses-S001_task-BART_cond-nf_sham_run-001_beh.xlsx
XDF: sub-P037_ses-S001_task-Default_run-001_eeg.xdf
Rows: (584, 59)

---
BEH: sub-P037_ses-S002_task-BART_cond-nf_sham_run-001_beh.xlsx
XDF: sub-P037_ses-S002_task-Default_run-001_eeg.xdf
Rows: (730, 59)

---
BEH: sub-P037_ses-S003_task-BART_cond-nf_sham_run-001_beh.xlsx
XDF: sub-P037_ses-S003_task-Default_run-001_eeg.xdf
Rows: (618, 59)

---
BEH: sub-P037_ses-S004_task-BART_cond-nf_sham_run-001_beh.xlsx
XDF: sub-P037_ses-S004_task-Default_run-001_eeg.xdf
Rows: (638, 59)

---
BEH: sub-P037_ses-S005_task-BART_cond-nf_sham_run-001_beh.xlsx
XDF: sub-P037_ses-S005_task-Default_run-001_eeg.xdf
Rows: (559, 59)

---
BEH: sub-P038_ses-S001_task-BART_cond-nf_sham_run-001_beh.xlsx
XDF: sub-P038_ses-S001_task-Default_run-001_eeg.xdf
Rows: (549, 59)

---
BEH: sub-P038_ses-S002_task-BART_cond-nf_sham_run-001_beh.xlsx
XDF: sub-P038_ses-S002_task-Default_run-001_eeg.xdf
Rows: (544, 59)

---
BEH: sub-P038_ses-S003_task

In [ ]:
# ============================================================
# BART MULTIMODAL MASTER PIPELINE (Pump-level) -- LOW RAM VERSION
# Input: *_beh.xlsx + matching *.xdf/*.xdfz + manifest.xlsx
# Output: outputs/BART_master_pump_level.csv
#
# Changes for RAM safety:
#  - writes session outputs incrementally to CSV
#  - avoids storing all_rows in memory
#  - deletes large objects after each session
#  - runs garbage collection
#  - optional: skip XLSX export until CSV is confirmed
# ============================================================

!pip -q install pyxdf mne openpyxl pandas numpy

import re, json, gc
from pathlib import Path
import numpy as np
import pandas as pd
import pyxdf
import mne

# ----------------------------
# CONFIG
# ----------------------------
ROOT = Path("/content") if Path("/content").exists() else Path(".")
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)

TARGET_SFREQ = 512.0

ERP_TMIN, ERP_TMAX = -0.2, 0.8
ERP_BASELINE = (ERP_TMIN, 0.0)
P3_WIN = (0.300, 0.450)

WIN200 = 0.200
THETA_BAND = (4.0, 7.0)

EEG_ORDER = ["Fz","Cz","Pz","POz","Fp1","Fp2","F3","F4","FCz","C3","C4","CPz","P3","P4","O1","O2"]
FRONTAL_THETA_CHS = ["Fp1","Fp2","Fz","F3","F4","FCz"]

CSV_PATH = OUT / "BART_master_pump_level.csv"
XLSX_PATH = OUT / "BART_master_pump_level.xlsx"

WRITE_XLSX_AT_END = False   # keep False for low RAM
INCLUDE_XDFZ = False        # set True only if you need .xdfz files

# ----------------------------
# ID normalization
# ----------------------------
def normalize_pid_series(s):
    s = s.astype(str).str.strip().str.upper()
    nums = s.str.extract(r"(\d+)")[0]
    out = nums.fillna("").apply(lambda x: f"P{int(x):03d}" if x != "" else np.nan)
    return out.where(out.notna(), s)

# ----------------------------
# Manifest
# ----------------------------
manifest = None
manifest_path = next(iter(ROOT.rglob("manifest.xlsx")), None)

if manifest_path is not None and manifest_path.exists():
    manifest_raw = pd.read_excel(manifest_path)
    manifest_raw.columns = [str(c).strip() for c in manifest_raw.columns]

    cols_l = {str(c).strip().lower(): c for c in manifest_raw.columns}
    pid_col = None
    risk_col = None

    for k in ["subject_id", "participant_id", "participant", "subject", "sub", "pid", "id"]:
        if k in cols_l:
            pid_col = cols_l[k]
            break

    for k in ["high/low", "risk_group", "risk", "group"]:
        if k in cols_l:
            risk_col = cols_l[k]
            break

    if pid_col is not None and risk_col is not None:
        manifest = manifest_raw[[pid_col, risk_col]].copy()
        manifest.columns = ["pID", "risk_group"]
        manifest["pID"] = normalize_pid_series(manifest["pID"])
        manifest["risk_group"] = manifest["risk_group"].astype(str).str.strip().str.upper()
        manifest = manifest.drop_duplicates(subset=["pID"], keep="first")
        print("Loaded manifest:", manifest_path)
        print("Manifest rows:", manifest.shape)
    else:
        print("manifest.xlsx found, but could not detect subject_id and/or high/low columns.")
else:
    print("No manifest.xlsx found.")

# ----------------------------
# Utilities
# ----------------------------
def parse_bids_bits(name: str):
    base = Path(name).name
    sub = re.search(r"sub-([A-Za-z0-9]+)", base)
    ses = re.search(r"ses-([A-Za-z0-9]+)", base)
    run = re.search(r"run-([0-9]+)", base)
    task = re.search(r"task-([A-Za-z0-9]+)", base)
    cond = re.search(r"cond-([A-Za-z0-9_]+)", base)

    out = {
        "sub": sub.group(1) if sub else None,
        "ses": ses.group(1) if ses else None,
        "run": run.group(1) if run else None,
        "task": task.group(1) if task else None,
        "condition": cond.group(1) if cond else None,
    }
    if out["ses"] is not None:
        m = re.search(r"(\d+)", out["ses"])
        out["ses_num"] = int(m.group(1)) if m else np.nan
    else:
        out["ses_num"] = np.nan
    return out

def read_beh_sheets(fp: Path):
    xls = pd.ExcelFile(fp)
    sheets = {}
    for sh in xls.sheet_names:
        df = pd.read_excel(fp, sheet_name=sh)
        df.columns = [str(c).strip() for c in df.columns]
        sheets[sh.lower()] = df
    return sheets

def get_sheet(sheets: dict, candidates):
    for c in candidates:
        if c in sheets:
            return sheets[c]
    return None

def safe_json_list(x):
    try:
        return json.loads(x) if isinstance(x, str) else (x if isinstance(x, list) else None)
    except Exception:
        return None

def extract_survey_row(survey_df, participant_id):
    if survey_df is None or len(survey_df) == 0:
        return None

    df = survey_df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    participant_id = normalize_pid_series(pd.Series([participant_id]))[0]

    if len(df) == 1:
        row = df.iloc[0].to_dict()
        row["pID"] = participant_id
        return row

    cols_l = {str(c).strip().lower(): c for c in df.columns}
    q_col = None
    r_col = None

    for k in ["question", "item", "prompt"]:
        if k in cols_l:
            q_col = cols_l[k]
            break
    for k in ["response", "answer", "value", "score"]:
        if k in cols_l:
            r_col = cols_l[k]
            break

    if q_col is not None and r_col is not None:
        out = {"pID": participant_id}
        for _, r in df.iterrows():
            q = str(r[q_col]).strip()
            q_clean = re.sub(r"[^A-Za-z0-9]+", "_", q).strip("_").lower()
            out[f"survey_{q_clean}"] = r[r_col]
        return out

    out = {"pID": participant_id}
    for i, (_, r) in enumerate(df.iterrows(), start=1):
        for c in df.columns:
            cname = re.sub(r"[^A-Za-z0-9]+", "_", str(c)).strip("_").lower()
            out[f"survey_r{i}_{cname}"] = r[c]
    return out

def downcast_df(df):
    for c in df.columns:
        if pd.api.types.is_float_dtype(df[c]):
            df[c] = pd.to_numeric(df[c], downcast="float")
        elif pd.api.types.is_integer_dtype(df[c]):
            df[c] = pd.to_numeric(df[c], downcast="integer")
    return df

# ----------------------------
# XDF helpers
# ----------------------------
def parse_marker_message(msg):
    s = str(msg)
    parts = s.split(";")
    out = {"code": parts[0].strip()}
    for p in parts[1:]:
        if "=" in p:
            k, v = p.split("=", 1)
            out[k.strip()] = v.strip()
    return out

def _stream_name_lower(s):
    info = s.get("info", {})
    nm = info.get("name", "")
    if isinstance(nm, (list, tuple, np.ndarray)):
        nm = nm[0] if len(nm) else ""
    return str(nm).strip().lower()

def pick_streams(streams):
    eeg = None
    best = -1

    for s in streams:
        ts = s.get("time_series", None)
        if ts is None:
            continue
        arr = np.asarray(ts)
        ch = 1 if arr.ndim == 1 else arr.shape[1]
        if ch > best:
            eeg = s
            best = ch

    markers = None
    for s in streams:
        ts = s.get("time_series", None)
        if ts is None:
            continue
        arr = np.asarray(ts)
        ch = 1 if arr.ndim == 1 else arr.shape[1]
        if ch != 1:
            continue
        nm = _stream_name_lower(s)
        if ("bart" in nm) or ("marker" in nm) or ("markers" in nm) or ("event" in nm):
            markers = s
            break

    if markers is None:
        for s in streams:
            ts = s.get("time_series", None)
            if ts is None:
                continue
            arr = np.asarray(ts)
            ch = 1 if arr.ndim == 1 else arr.shape[1]
            if ch == 1:
                markers = s
                break

    return eeg, markers

def markers_to_df(marker_stream):
    m_ts = np.asarray(marker_stream["time_stamps"], dtype=float)
    m_val = np.asarray(marker_stream["time_series"], dtype=object)

    rows = []
    for t, v in zip(m_ts, m_val):
        msg = v[0] if isinstance(v, (list, tuple, np.ndarray)) else v
        d = parse_marker_message(msg)
        d["lsltime"] = float(t)
        rows.append(d)

    df = pd.DataFrame(rows)
    for k in ["trial", "pump", "loss", "earnings", "total"]:
        if k in df.columns:
            df[k] = pd.to_numeric(df[k], errors="coerce")
    return df

def build_raw_from_xdf_eeg(eeg_stream):
    data = np.asarray(eeg_stream["time_series"])
    t = np.asarray(eeg_stream["time_stamps"], dtype=float)
    if data.ndim == 1:
        data = data[:, None]
    n_samp, n_ch = data.shape

    sfreq_est = 1.0 / np.median(np.diff(t))
    ch_names = EEG_ORDER[:n_ch] if n_ch <= len(EEG_ORDER) else [f"EEG{ii+1:02d}" for ii in range(n_ch)]
    arr = data.T.astype(np.float32)

    peak = float(np.nanmax(np.abs(arr)))
    if peak > 1e3:
        arr *= 1e-6
    elif peak > 5.0:
        arr *= 1e-3

    info = mne.create_info(ch_names=ch_names, sfreq=sfreq_est, ch_types=["eeg"] * n_ch)
    raw = mne.io.RawArray(arr, info, verbose=False)
    return raw, t

def preprocess_raw(raw):
    raw = raw.copy()
    raw.filter(0.1, 30.0, picks="eeg", verbose=False)
    try:
        raw.notch_filter([60], picks="eeg", verbose=False)
    except Exception:
        pass
    try:
        raw.set_eeg_reference("average", verbose=False)
    except Exception:
        pass
    return raw

def lsltime_to_sample(lsltime, eeg_t0, sfreq):
    return int(np.round((lsltime - eeg_t0) * sfreq))

# ----------------------------
# EEG feature extraction
# ----------------------------
def compute_theta_power_window(raw_theta, sample, tmin_s, tmax_s, picks):
    sf = raw_theta.info["sfreq"]
    start = int(sample + np.round(tmin_s * sf))
    stop = int(sample + np.round(tmax_s * sf))
    start = max(start, 0)
    stop = min(stop, raw_theta.n_times)
    if stop <= start + 2:
        return np.nan
    seg = raw_theta.get_data(picks=picks, start=start, stop=stop)
    return float(np.mean(seg**2))

def extract_session_theta_metrics(raw):
    rtheta = raw.copy().filter(THETA_BAND[0], THETA_BAND[1], picks="eeg", verbose=False)
    picks = [raw.ch_names.index(ch) for ch in FRONTAL_THETA_CHS if ch in raw.ch_names]
    if len(picks) == 0:
        return np.nan, np.nan
    X = rtheta.get_data(picks=picks)
    theta_ts = np.mean(X**2, axis=0)
    return float(np.mean(theta_ts)), float(np.std(theta_ts, ddof=1))

def extract_p300_metrics(raw, md, eeg_t0):
    keys = ["ses_avgP300_All","ses_avgP300_Fz","ses_avgP300_FCz","ses_avgP300_Cz","ses_avgP300_Pz"]
    if "code" not in md.columns:
        return {k: np.nan for k in keys}

    exp = md[md["code"].astype(str).str.upper().eq("BART_EXPLODE")].copy()
    if len(exp) == 0:
        return {k: np.nan for k in keys}

    sf = raw.info["sfreq"]
    exp = exp.dropna(subset=["lsltime"]).copy()
    exp["sample"] = exp["lsltime"].apply(lambda t: lsltime_to_sample(float(t), eeg_t0, sf)).astype(int)

    min_samp = int(np.ceil((-ERP_TMIN) * sf))
    exp = exp[(exp["sample"] >= min_samp) & (exp["sample"] < raw.n_times)].copy()
    if len(exp) == 0:
        return {k: np.nan for k in keys}

    events = np.c_[exp["sample"].to_numpy(), np.zeros(len(exp), int), np.ones(len(exp), int)]
    epochs = mne.Epochs(
        raw, events, event_id={"EXPLODE": 1},
        tmin=ERP_TMIN, tmax=ERP_TMAX, baseline=ERP_BASELINE,
        preload=True, reject_by_annotation=True, verbose=False
    )
    if len(epochs) == 0:
        return {k: np.nan for k in keys}

    ev = epochs.average()
    times = ev.times
    m1 = np.argmin(np.abs(times - P3_WIN[0]))
    m2 = np.argmin(np.abs(times - P3_WIN[1]))

    out = {}
    vals = []
    for ch in ["Fz","FCz","Cz","Pz"]:
        k = f"ses_avgP300_{ch}"
        if ch in ev.ch_names:
            idx = ev.ch_names.index(ch)
            erp_uV = ev.data[idx] * 1e6
            p3 = float(np.max(erp_uV[m1:m2]))
            out[k] = p3
            vals.append(p3)
        else:
            out[k] = np.nan
    out["ses_avgP300_All"] = float(np.nanmean(vals)) if len(vals) else np.nan
    return out

def extract_event_theta_windows(raw, md, eeg_t0):
    rtheta = raw.copy().filter(THETA_BAND[0], THETA_BAND[1], picks="eeg", verbose=False)
    sf = raw.info["sfreq"]
    picks = [raw.ch_names.index(ch) for ch in FRONTAL_THETA_CHS if ch in raw.ch_names]
    if len(picks) == 0:
        return {
            "pre200_pop_Theta": np.nan, "post200_pop_Theta": np.nan,
            "pre200_collect_Theta": np.nan, "post200_collect_Theta": np.nan,
            "pre200_pmp_Theta": np.nan,
        }

    def _mean_prepost(code, pre=True):
        d = md[md["code"].astype(str).str.upper().eq(code)].dropna(subset=["lsltime"]).copy()
        if len(d) == 0:
            return np.nan
        d["sample"] = d["lsltime"].apply(lambda t: lsltime_to_sample(float(t), eeg_t0, sf)).astype(int)
        vals = []
        for s in d["sample"].tolist():
            v = compute_theta_power_window(rtheta, s, -WIN200, 0.0, picks) if pre else compute_theta_power_window(rtheta, s, 0.0, WIN200, picks)
            vals.append(v)
        return float(np.nanmean(vals)) if len(vals) else np.nan

    return {
        "pre200_pop_Theta": _mean_prepost("BART_EXPLODE", pre=True),
        "post200_pop_Theta": _mean_prepost("BART_EXPLODE", pre=False),
        "pre200_collect_Theta": _mean_prepost("BART_COLLECT", pre=True),
        "post200_collect_Theta": _mean_prepost("BART_COLLECT", pre=False),
        "pre200_pmp_Theta": _mean_prepost("BART_PUMP", pre=True),
    }

def theta_before_each_pump(raw, md, eeg_t0):
    pumps = md[md["code"].astype(str).str.upper().eq("BART_PUMP")].copy()
    if len(pumps) == 0 or ("trial" not in pumps.columns) or ("pump" not in pumps.columns):
        return pd.DataFrame(columns=["trial","pump_number","theta_before_pump","theta_before_pump_Fp1","theta_before_pump_Fp2","frontal_asymmetry"])

    rtheta = raw.copy().filter(THETA_BAND[0], THETA_BAND[1], picks="eeg", verbose=False)
    sf = raw.info["sfreq"]

    pumps["trial"] = pd.to_numeric(pumps["trial"], errors="coerce")
    pumps["pump_number"] = pd.to_numeric(pumps["pump"], errors="coerce")
    pumps = pumps.dropna(subset=["trial","pump_number","lsltime"]).copy()

    pumps["trial"] = pumps["trial"].astype(int)
    pumps["pump_number"] = pumps["pump_number"].astype(int)
    pumps["sample"] = pumps["lsltime"].apply(lambda t: lsltime_to_sample(float(t), eeg_t0, sf)).astype(int)

    min_samp = int(np.ceil(WIN200 * sf))
    pumps = pumps[(pumps["sample"] >= min_samp) & (pumps["sample"] < raw.n_times)].copy()
    if len(pumps) == 0:
        return pd.DataFrame(columns=["trial","pump_number","theta_before_pump","theta_before_pump_Fp1","theta_before_pump_Fp2","frontal_asymmetry"])

    frontal_picks = [raw.ch_names.index(ch) for ch in FRONTAL_THETA_CHS if ch in raw.ch_names]
    fp1_pick = [raw.ch_names.index("Fp1")] if "Fp1" in raw.ch_names else []
    fp2_pick = [raw.ch_names.index("Fp2")] if "Fp2" in raw.ch_names else []

    rows = []
    for _, r in pumps.iterrows():
        s = int(r["sample"])
        pre_frontal = compute_theta_power_window(rtheta, s, -WIN200, 0.0, frontal_picks) if frontal_picks else np.nan
        pre_fp1 = compute_theta_power_window(rtheta, s, -WIN200, 0.0, fp1_pick) if fp1_pick else np.nan
        pre_fp2 = compute_theta_power_window(rtheta, s, -WIN200, 0.0, fp2_pick) if fp2_pick else np.nan
        fa = (np.log(pre_fp2 + 1e-12) - np.log(pre_fp1 + 1e-12)) if (np.isfinite(pre_fp1) and np.isfinite(pre_fp2)) else np.nan

        rows.append({
            "trial": int(r["trial"]),
            "pump_number": int(r["pump_number"]),
            "theta_before_pump": pre_frontal,
            "theta_before_pump_Fp1": pre_fp1,
            "theta_before_pump_Fp2": pre_fp2,
            "frontal_asymmetry": fa
        })

    return pd.DataFrame(rows)

def trial_theta_from_markers(raw, md, eeg_t0):
    sf = raw.info["sfreq"]
    rtheta = raw.copy().filter(THETA_BAND[0], THETA_BAND[1], picks="eeg", verbose=False)
    picks = [raw.ch_names.index(ch) for ch in FRONTAL_THETA_CHS if ch in raw.ch_names]
    if len(picks) == 0:
        return pd.DataFrame(columns=["trial","trial_mTheta","trial_SD_mThetaB"])

    starts = md[md["code"].astype(str).str.upper().eq("BART_TRIAL_START")].copy()
    ends = md[md["code"].astype(str).str.upper().isin(["BART_COLLECT","BART_EXPLODE"])].copy()
    if "trial" not in starts.columns or "trial" not in ends.columns:
        return pd.DataFrame(columns=["trial","trial_mTheta","trial_SD_mThetaB"])

    starts["trial"] = pd.to_numeric(starts["trial"], errors="coerce")
    ends["trial"] = pd.to_numeric(ends["trial"], errors="coerce")
    starts = starts.dropna(subset=["trial","lsltime"]).copy()
    ends = ends.dropna(subset=["trial","lsltime"]).copy()
    if len(starts) == 0 or len(ends) == 0:
        return pd.DataFrame(columns=["trial","trial_mTheta","trial_SD_mThetaB"])

    out = []
    for tnum, g0 in starts.groupby(starts["trial"].astype(int)):
        t_start = float(g0["lsltime"].min())
        g1 = ends[ends["trial"].astype(int) == int(tnum)]
        if len(g1) == 0:
            continue
        t_end = float(g1["lsltime"].min())
        if t_end <= t_start:
            continue

        s0 = lsltime_to_sample(t_start, eeg_t0, sf)
        s1 = lsltime_to_sample(t_end, eeg_t0, sf)
        s0 = max(s0, 0)
        s1 = min(s1, raw.n_times)
        if s1 <= s0 + int(sf * 0.2):
            continue

        seg = rtheta.get_data(picks=picks, start=s0, stop=s1)
        theta_ts = np.mean(seg**2, axis=0)

        out.append({
            "trial": int(tnum),
            "trial_mTheta": float(np.mean(theta_ts)),
            "trial_SD_mThetaB": float(np.std(theta_ts, ddof=1)),
        })

    return pd.DataFrame(out)

# ----------------------------
# BEH -> pump table
# ----------------------------
def beh_to_pump_table(beh_fp: Path):
    sheets = read_beh_sheets(beh_fp)
    trials = get_sheet(sheets, ["trials", "trial"])
    summary = get_sheet(sheets, ["summary"])

    if trials is None or len(trials) == 0:
        return None, None, None

    trials = trials.copy()
    bits = parse_bids_bits(beh_fp.name)

    if "exploded" in trials.columns:
        trials["pop"] = trials["exploded"].astype(str).str.upper().isin(["TRUE","1","YES"]).astype(int)
    else:
        trials["pop"] = 0

    if "collected" in trials.columns:
        trials["collect"] = pd.to_numeric(trials["collected"], errors="coerce").fillna(0).astype(int)
    else:
        trials["collect"] = trials["events"].astype(str).str.contains("collect", case=False).astype(int) if "events" in trials.columns else 0

    for c in [
        "trial","pump_count","trial_earnings","loss_if_pop","trial_value","total_earnings",
        "pump_latency_mean","pump_latency_median","pump_latency_first",
        "collect_latency_from_ready","collect_latency_from_trial_start",
        "nf_high_frac","nf_green_success",
        "risk_after_explosion","risk_after_collect","pump_speed_change",
        "nf_mean_normalized"
    ]:
        if c in trials.columns:
            trials[c] = pd.to_numeric(trials[c], errors="coerce")

    if "nf_high_frac" in trials.columns:
        trials["nf_high_fraction"] = pd.to_numeric(trials["nf_high_frac"], errors="coerce")
    elif "nf_high_fraction" not in trials.columns:
        trials["nf_high_fraction"] = np.nan

    if "nf_green_success" in trials.columns and trials["nf_green_success"].notna().any():
        trials["nf_green_success"] = pd.to_numeric(trials["nf_green_success"], errors="coerce")
    else:
        trials["nf_green_success"] = np.where(trials["nf_high_fraction"] > 0.75, 1, 0).astype(int) if "nf_high_fraction" in trials.columns else np.nan

    block_col = "block" if "block" in trials.columns else None
    if block_col is not None:
        trials = trials.sort_values([block_col, "trial"], kind="mergesort") if "trial" in trials.columns else trials
        g = trials.groupby(block_col)["nf_green_success"]
        trials["nf_bonus_awarded_trial"] = ((g.shift(0)==1) & (g.shift(1)==1) & (g.shift(2)==1)).astype(int)
    else:
        trials["nf_bonus_awarded_trial"] = 0

    if "pump_count" in trials.columns and block_col is not None and "trial" in trials.columns:
        trials["prev_pumps"] = trials.groupby(block_col)["pump_count"].shift(1)
        prev_pop = trials.groupby(block_col)["pop"].shift(1)
        prev_collect = trials.groupby(block_col)["collect"].shift(1)

        trials["risk_after_explosion"] = np.where(prev_pop == 1, trials["pump_count"] - trials["prev_pumps"], np.nan)
        trials["risk_after_collect"] = np.where(prev_collect == 1, trials["pump_count"] - trials["prev_pumps"], np.nan)

        if "pump_latency_mean" in trials.columns:
            trials["pump_speed_change"] = trials["pump_latency_mean"] - trials.groupby(block_col)["pump_latency_mean"].shift(1)
        else:
            trials["pump_speed_change"] = np.nan
    else:
        if "risk_after_explosion" not in trials.columns:
            trials["risk_after_explosion"] = np.nan
        if "risk_after_collect" not in trials.columns:
            trials["risk_after_collect"] = np.nan
        if "pump_speed_change" not in trials.columns:
            trials["pump_speed_change"] = np.nan

    sesPoints = trials["total_earnings"].dropna().max() if "total_earnings" in trials.columns else np.nan
    exp_N = int(trials["pop"].sum())
    collect_N = int(trials["collect"].sum())
    tN = int(len(trials))
    exp_rate = exp_N / tN if tN else np.nan
    collect_rate = collect_N / tN if tN else np.nan
    bonus_N = int(trials["nf_bonus_awarded_trial"].sum()) if "nf_bonus_awarded_trial" in trials.columns else 0

    smPmp = float(trials["pump_count"].mean()) if "pump_count" in trials.columns else np.nan
    SD_smPmp = float(trials["pump_count"].std(ddof=1)) if "pump_count" in trials.columns else np.nan

    trials["adjusted_pumps_trial"] = trials["pump_count"] if "pump_count" in trials.columns else np.nan
    smADJ_pmp = float(trials["adjusted_pumps_trial"].mean()) if "adjusted_pumps_trial" in trials.columns else np.nan
    SD_smADJ_pmp = float(trials["adjusted_pumps_trial"].std(ddof=1)) if "adjusted_pumps_trial" in trials.columns else np.nan

    mpmpL = float(trials["pump_latency_mean"].mean()) if "pump_latency_mean" in trials.columns else np.nan
    SD_mpmpL = float(trials["pump_latency_mean"].std(ddof=1)) if "pump_latency_mean" in trials.columns else np.nan
    medpmpL = float(trials["pump_latency_median"].mean()) if "pump_latency_median" in trials.columns else np.nan
    SD_medpmpL = float(trials["pump_latency_median"].std(ddof=1)) if "pump_latency_median" in trials.columns else np.nan

    def _get_scalar_from_sources(keys, sources):
        for df in sources:
            if df is None or len(df) == 0:
                continue
            cols_lut = {str(c).strip().lower(): c for c in df.columns}
            for k in keys:
                kk = str(k).strip().lower()
                if kk in cols_lut:
                    col = cols_lut[kk]
                    ser = pd.to_numeric(df[col], errors="coerce")
                    if ser.notna().any():
                        return float(ser.dropna().iloc[0])
        return np.nan

    sources = [trials, summary]

    baseline_cols = {
        "baseline_eo_mu": _get_scalar_from_sources(
            ["baseline_eo_mu", "baseline_pre_eo_mu", "pretask_eob_avgtheta", "pretask_eo_avgtheta"], sources),
        "baseline_ec_mu": _get_scalar_from_sources(
            ["baseline_ec_mu", "baseline_pre_ec_mu", "pretask_ecb_avgtheta", "pretask_ec_avgtheta"], sources),
        "baseline_conc_mu": _get_scalar_from_sources(
            ["baseline_conc_mu", "baseline_pre_conc_mu", "pretask_cb_avgtheta", "pretask_conc_avgtheta"], sources),
        "baseline_post_eo_mu": _get_scalar_from_sources(
            ["baseline_post_eo_mu", "posttask_eob_avgtheta", "post_eob_avgtheta", "posttask_eo_avgtheta"], sources),
        "baseline_post_ec_mu": _get_scalar_from_sources(
            ["baseline_post_ec_mu", "posttask_ecb_avgtheta", "post_ecb_avgtheta", "posttask_ec_avgtheta"], sources),
        "baseline_post_conc_mu": _get_scalar_from_sources(
            ["baseline_post_conc_mu", "posttask_cb_avgtheta", "post_cb_avgtheta", "posttask_conc_avgtheta"], sources),
    }

    pump_rows = []
    for _, tr in trials.iterrows():
        n = int(tr["pump_count"]) if pd.notna(tr.get("pump_count", np.nan)) else 0
        if n <= 0:
            continue

        lat_list = safe_json_list(tr.get("pump_latencies_json", None))

        for p in range(1, n + 1):
            row = {}
            row["pID"] = normalize_pid_series(pd.Series([bits["sub"]]))[0]
            row["cond"] = bits["condition"]
            row["sID"] = bits["ses"]
            row["ses_num"] = bits["ses_num"]
            row["pPoints"] = np.nan

            row["sesPoints"] = sesPoints
            row["exp_N"] = exp_N
            row["collect_N"] = collect_N
            row["bonus_N"] = bonus_N
            row["exp_rate"] = exp_rate
            row["collect_rate"] = collect_rate
            row["smPmp"] = smPmp
            row["SD_smPmp"] = SD_smPmp
            row["smADJ_pmp"] = smADJ_pmp
            row["SD_smADJ_pmp"] = SD_smADJ_pmp
            row["mpmpL"] = mpmpL
            row["SD_mpmpL"] = SD_mpmpL
            row["medpmpL"] = medpmpL
            row["SD_medpmpL"] = SD_medpmpL

            row["pretask_EOB_avgTheta"] = baseline_cols.get("baseline_eo_mu", np.nan)
            row["posttask_EOB_avgTheta"] = baseline_cols.get("baseline_post_eo_mu", np.nan)
            row["pretask_ECB_avgTheta"] = baseline_cols.get("baseline_ec_mu", np.nan)
            row["posttask_ECB_avgTheta"] = baseline_cols.get("baseline_post_ec_mu", np.nan)
            row["pretask_CB_avgTheta"] = baseline_cols.get("baseline_conc_mu", np.nan)
            row["posttask_CB_avgTheta"] = baseline_cols.get("baseline_post_conc_mu", np.nan)

            row["tID"] = int(tr["trial"]) if "trial" in tr and pd.notna(tr["trial"]) else np.nan
            row["pop"] = int(tr["pop"])
            row["collect"] = int(tr["collect"])
            row["pmp_count"] = int(tr["pump_count"]) if pd.notna(tr.get("pump_count", np.nan)) else np.nan
            row["t_earned"] = float(tr["trial_earnings"]) if "trial_earnings" in tr and pd.notna(tr["trial_earnings"]) else np.nan
            row["t_loss"] = float(tr["loss_if_pop"]) if "loss_if_pop" in tr and pd.notna(tr["loss_if_pop"]) else np.nan

            row["collectL_trial"] = float(tr["collect_latency_from_trial_start"]) if "collect_latency_from_trial_start" in tr and pd.notna(tr["collect_latency_from_trial_start"]) else np.nan
            row["collectL_ready"] = float(tr["collect_latency_from_ready"]) if "collect_latency_from_ready" in tr and pd.notna(tr["collect_latency_from_ready"]) else np.nan

            row["nf_high_fraction"] = float(tr["nf_high_fraction"]) if "nf_high_fraction" in tr and pd.notna(tr["nf_high_fraction"]) else np.nan
            row["nf_green_success"] = int(tr["nf_green_success"]) if "nf_green_success" in tr and pd.notna(tr["nf_green_success"]) else np.nan

            row["risk_after_explosion"] = float(tr.get("risk_after_explosion", np.nan)) if pd.notna(tr.get("risk_after_explosion", np.nan)) else np.nan
            row["risk_after_collect"] = float(tr.get("risk_after_collect", np.nan)) if pd.notna(tr.get("risk_after_collect", np.nan)) else np.nan
            row["pump_speed_change"] = float(tr.get("pump_speed_change", np.nan)) if pd.notna(tr.get("pump_speed_change", np.nan)) else np.nan

            row["pump_number"] = int(p)
            row["pmpID"] = f"{bits['sub']}_{bits['ses']}_T{int(row['tID'])}_P{p}" if pd.notna(row["tID"]) else f"{bits['sub']}_{bits['ses']}_P{p}"
            row["pmpL"] = float(lat_list[p-1]) if isinstance(lat_list, list) and len(lat_list) >= p else np.nan

            row["ses_mTheta"] = np.nan
            row["ses_SD_mThetaB"] = np.nan
            row["ses_avgP300_All"] = np.nan
            row["ses_avgP300_Fz"] = np.nan
            row["ses_avgP300_FCz"] = np.nan
            row["ses_avgP300_Cz"] = np.nan
            row["ses_avgP300_Pz"] = np.nan
            row["pre200_pop_Theta"] = np.nan
            row["post200_pop_Theta"] = np.nan
            row["pre200_collect_Theta"] = np.nan
            row["post200_collect_Theta"] = np.nan
            row["pre200_pmp_Theta"] = np.nan
            row["theta_before_pump"] = np.nan
            row["theta_before_pump_Fp1"] = np.nan
            row["theta_before_pump_Fp2"] = np.nan
            row["frontal_asymmetry"] = np.nan
            row["trial_mTheta"] = np.nan
            row["trial_SD_mThetaB"] = np.nan

            pump_rows.append(row)

    pump_df = pd.DataFrame(pump_rows)
    pump_df = downcast_df(pump_df)
    return pump_df, trials, summary

# ----------------------------
# Pair processing
# ----------------------------
def process_pair(beh_fp: Path, xdf_fp):
    pump_df, _, _ = beh_to_pump_table(beh_fp)
    if pump_df is None or len(pump_df) == 0:
        return None

    if xdf_fp is None:
        return pump_df

    try:
        streams, _ = pyxdf.load_xdf(str(xdf_fp))
        eeg_stream, marker_stream = pick_streams(streams)
        if eeg_stream is None or marker_stream is None:
            return pump_df

        raw, eeg_t = build_raw_from_xdf_eeg(eeg_stream)
        raw = preprocess_raw(raw)
        raw.resample(TARGET_SFREQ, verbose=False)
        eeg_t0 = float(eeg_t[0])

        md = markers_to_df(marker_stream)

        ses_mTheta, ses_SD_mThetaB = extract_session_theta_metrics(raw)
        p300 = extract_p300_metrics(raw, md, eeg_t0)
        ev_theta = extract_event_theta_windows(raw, md, eeg_t0)
        pump_theta = theta_before_each_pump(raw, md, eeg_t0)
        trial_theta = trial_theta_from_markers(raw, md, eeg_t0)

        pump_df["ses_mTheta"] = ses_mTheta
        pump_df["ses_SD_mThetaB"] = ses_SD_mThetaB
        for k, v in p300.items():
            pump_df[k] = v
        for k, v in ev_theta.items():
            pump_df[k] = v

        pump_df["tID"] = pd.to_numeric(pump_df["tID"], errors="coerce").astype("Int64")
        pump_df["pump_number"] = pd.to_numeric(pump_df["pump_number"], errors="coerce").astype("Int64")

        if trial_theta is not None and len(trial_theta) > 0:
            tt = trial_theta.copy()
            tt["tID"] = pd.to_numeric(tt["trial"], errors="coerce").astype("Int64")
            tt = tt.drop(columns=["trial"], errors="ignore")
            pump_df = pump_df.drop(columns=["trial_mTheta", "trial_SD_mThetaB"], errors="ignore")
            pump_df = pump_df.merge(tt, on="tID", how="left")

        if pump_theta is not None and len(pump_theta) > 0:
            pt = pump_theta.copy()
            pt["tID"] = pd.to_numeric(pt["trial"], errors="coerce").astype("Int64")
            pt["pump_number"] = pd.to_numeric(pt["pump_number"], errors="coerce").astype("Int64")
            pt = pt.drop(columns=["trial"], errors="ignore")
            pump_df = pump_df.drop(columns=["theta_before_pump", "theta_before_pump_Fp1", "theta_before_pump_Fp2", "frontal_asymmetry"], errors="ignore")
            pump_df = pump_df.merge(pt, on=["tID", "pump_number"], how="left")

        for c in ["trial_mTheta","trial_SD_mThetaB","theta_before_pump","theta_before_pump_Fp1","theta_before_pump_Fp2","frontal_asymmetry"]:
            if c not in pump_df.columns:
                pump_df[c] = np.nan

        pump_df = downcast_df(pump_df)
        return pump_df

    except Exception as e:
        print("⚠️ XDF failed:", xdf_fp.name, "->", e)
        return pump_df

    finally:
        for var in ["streams", "eeg_stream", "marker_stream", "raw", "eeg_t", "md", "pump_theta", "trial_theta", "p300", "ev_theta"]:
            if var in locals():
                del locals()[var]
        gc.collect()

# ----------------------------
# Find files
# ----------------------------
beh_files = sorted(ROOT.rglob("*_beh.xlsx"))
xdf_files = sorted(ROOT.rglob("*.xdf"))
if INCLUDE_XDFZ:
    xdf_files += sorted(ROOT.rglob("*.xdfz"))

print("Found BEH files:", len(beh_files))
print("Found XDF files:", len(xdf_files))

xdf_map = {}
for xf in xdf_files:
    key = xf.name.split("_task")[0]
    xdf_map.setdefault(key, []).append(xf)

# ----------------------------
# Build survey table first
# ----------------------------
survey_rows = []
for bf in beh_files:
    bits = parse_bids_bits(bf.name)
    if pd.isna(bits["ses_num"]) or int(bits["ses_num"]) != 5:
        continue

    sheets = read_beh_sheets(bf)
    survey_df = get_sheet(sheets, ["survey", "surveys"])
    if survey_df is None or len(survey_df) == 0:
        continue

    participant_id = normalize_pid_series(pd.Series([bits["sub"]]))[0]
    survey_row = extract_survey_row(survey_df, participant_id)
    if survey_row is not None:
        survey_rows.append(survey_row)

survey_master = pd.DataFrame(survey_rows) if len(survey_rows) else pd.DataFrame(columns=["pID"])
if len(survey_master):
    survey_master["pID"] = normalize_pid_series(survey_master["pID"])
    survey_master = survey_master.drop_duplicates(subset=["pID"], keep="first")

# ----------------------------
# Process session-by-session and write incrementally
# ----------------------------
if CSV_PATH.exists():
    CSV_PATH.unlink()

processed = 0
missing_xdf = 0
header_written = False
participant_points_rows = []

for bf in beh_files:
    key = bf.name.split("_task")[0]
    xmatch = xdf_map.get(key, [])
    xdf_fp = xmatch[0] if len(xmatch) else None
    if xdf_fp is None:
        missing_xdf += 1

    print("\n---")
    print("BEH:", bf.name)
    print("XDF:", xdf_fp.name if xdf_fp else "MISSING (BEH-only)")

    out = process_pair(bf, xdf_fp)
    if out is None or len(out) == 0:
        gc.collect()
        continue

    out["pID"] = normalize_pid_series(out["pID"])

    if manifest is not None and "pID" in manifest.columns:
        out = out.drop(columns=["risk_group"], errors="ignore")
        out = out.merge(manifest[["pID", "risk_group"]], on="pID", how="left")

    if len(survey_master):
        out = out.merge(survey_master, on="pID", how="left")

    # store unique session points for later participant total calculation
    if {"pID", "sID", "sesPoints"}.issubset(out.columns):
        participant_points_rows.append(out[["pID", "sID", "sesPoints"]].drop_duplicates())

    out = downcast_df(out)

    out.to_csv(CSV_PATH, mode="a", header=not header_written, index=False)
    header_written = True
    processed += 1
    print("Rows written:", out.shape)

    del out
    gc.collect()

print("\n✅ sessions processed:", processed)
print("⚠️ missing XDF sessions:", missing_xdf)
print("✅ CSV saved:", CSV_PATH)

# ----------------------------
# Optional second pass: add pPoints + reorder + save final files
# ----------------------------
master = pd.read_csv(CSV_PATH, low_memory=False)

if len(participant_points_rows):
    tmp = pd.concat(participant_points_rows, ignore_index=True).drop_duplicates()
    totals = tmp.groupby("pID")["sesPoints"].sum()
    master["pPoints"] = master["pID"].map(totals)

schema_cols = [
    "pID","risk_group","cond","pPoints","sID","ses_num",
    "sesPoints","ses_mTheta","ses_SD_mThetaB",
    "ses_avgP300_All","ses_avgP300_Fz","ses_avgP300_FCz","ses_avgP300_Cz","ses_avgP300_Pz",
    "smPmp","SD_smPmp","smADJ_pmp","SD_smADJ_pmp",
    "exp_rate","exp_N","bonus_N","collect_N","collect_rate",
    "pretask_EOB_avgTheta","posttask_EOB_avgTheta",
    "pretask_ECB_avgTheta","posttask_ECB_avgTheta",
    "pretask_CB_avgTheta","posttask_CB_avgTheta",
    "tID","trial_mTheta","trial_SD_mThetaB",
    "pop","collect","nf_high_fraction","nf_green_success","t_earned","t_loss",
    "collectL_trial","collectL_ready",
    "pmp_count",
    "pre200_pop_Theta","post200_pop_Theta","pre200_collect_Theta","post200_collect_Theta","pre200_pmp_Theta",
    "risk_after_explosion","risk_after_collect","pump_speed_change",
    "pump_number","pmpID","pmpL",
    "mpmpL","SD_mpmpL","medpmpL","SD_medpmpL",
    "theta_before_pump","theta_before_pump_Fp1","theta_before_pump_Fp2","frontal_asymmetry",
]

for c in schema_cols:
    if c not in master.columns:
        master[c] = np.nan

survey_cols = [c for c in master.columns if str(c).lower().startswith("survey_")]
base_order = [c for c in schema_cols if c in master.columns and c != "pID"]
remaining = [c for c in master.columns if c not in (["pID"] + survey_cols + base_order)]

ordered = ["pID"] + survey_cols + base_order + remaining
master = master[ordered]

master = downcast_df(master)
master.to_csv(CSV_PATH, index=False)

print("Survey columns found:", len(survey_cols))
print("Unique risk_group values:", sorted(master["risk_group"].dropna().astype(str).unique().tolist()) if "risk_group" in master.columns else [])
print("Final master shape:", master.shape)
print("✅ Final CSV saved:", CSV_PATH)

if WRITE_XLSX_AT_END:
    master.to_excel(XLSX_PATH, index=False)
    print("✅ XLSX saved:", XLSX_PATH)

master.head(10)

Loaded manifest: /content/manifest.xlsx
Manifest rows: (49, 2)
Found BEH files: 230
Found XDF files: 227

---
BEH: sub-P001_ses-S001_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P001_ses-S001_task-Default_run-001_eeg.xdf
Rows written: (536, 250)

---
BEH: sub-P001_ses-S002_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P001_ses-S002_task-Default_run-001_eeg.xdf
Rows written: (691, 250)

---
BEH: sub-P001_ses-S003_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P001_ses-S003_task-Default_run-001_eeg.xdf
Rows written: (688, 250)

---
BEH: sub-P001_ses-S004_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: MISSING (BEH-only)
Rows written: (562, 250)

---
BEH: sub-P001_ses-S005_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P001_ses-S005_task-Default_run-001_eeg.xdf
Rows written: (678, 250)

---
BEH: sub-P003_ses-S001_task-BART_cond-nf_sham_run-001_beh.xlsx
XDF: sub-P003_ses-S001_task-Default_run-001_eeg.xdf
Rows written: (305, 250)

---
BEH: sub-P003_ses-S002_task-BART_cond-nf_sham_ru

Rows written: (786, 250)

---
BEH: sub-P005_ses-S001_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P005_ses-S001_task-Default_run-001_eeg.xdf
Rows written: (633, 250)

---
BEH: sub-P005_ses-S002_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P005_ses-S002_task-Default_run-001_eeg.xdf
Rows written: (580, 250)

---
BEH: sub-P005_ses-S003_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P005_ses-S003_task-Default_run-001_eeg.xdf
Rows written: (618, 250)

---
BEH: sub-P005_ses-S004_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P005_ses-S004_task-Default_run-001_eeg.xdf
Rows written: (621, 250)

---
BEH: sub-P005_ses-S005_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P005_ses-S005_task-Default_run-001_eeg.xdf
Rows written: (574, 250)

---
BEH: sub-P006_ses-S001_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P006_ses-S001_task-Default_run-001_eeg.xdf
Rows written: (507, 250)

---
BEH: sub-P006_ses-S002_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P006_ses-S002_task-Default_r

ERROR:pyxdf.pyxdf:found likely XDF file corruption (unpack requires a buffer of 8 bytes), scanning forward to next boundary chunk.


Rows written: (719, 250)

---
BEH: sub-P016_ses-S005_task-BART_cond-nf_real_run-001_beh.xlsx
XDF: sub-P016_ses-S005_task-Default_run-001_eeg.xdf


ERROR:pyxdf.pyxdf:found likely XDF file corruption (unpack requires a buffer of 8 bytes), scanning forward to next boundary chunk.


Rows written: (782, 250)

---
BEH: sub-P017_ses-S001_task-BART_cond-nf_sham_run-001_beh.xlsx
XDF: sub-P017_ses-S001_task-Default_run-001_eeg.xdf


... (13 outputs left out)

Rows written: (868, 250)

---
BEH: sub-P036_ses-S005_task-BART_cond-nf_sham_run-001_beh.xlsx
XDF: sub-P036_ses-S005_task-Default_run-001_eeg.xdf


ERROR:pyxdf.pyxdf:found likely XDF file corruption (unpack requires a buffer of 8 bytes), scanning forward to next boundary chunk.


Rows written: (761, 250)

---
BEH: sub-P037_ses-S001_task-BART_cond-nf_sham_run-001_beh.xlsx
XDF: sub-P037_ses-S001_task-Default_run-001_eeg.xdf
Rows written: (584, 250)

---
BEH: sub-P037_ses-S002_task-BART_cond-nf_sham_run-001_beh.xlsx
XDF: sub-P037_ses-S002_task-Default_run-001_eeg.xdf
Rows written: (730, 250)

---
BEH: sub-P037_ses-S003_task-BART_cond-nf_sham_run-001_beh.xlsx
XDF: sub-P037_ses-S003_task-Default_run-001_eeg.xdf
Rows written: (618, 250)

---
BEH: sub-P037_ses-S004_task-BART_cond-nf_sham_run-001_beh.xlsx
XDF: sub-P037_ses-S004_task-Default_run-001_eeg.xdf
Rows written: (638, 250)

---
BEH: sub-P037_ses-S005_task-BART_cond-nf_sham_run-001_beh.xlsx
XDF: sub-P037_ses-S005_task-Default_run-001_eeg.xdf
Rows written: (559, 250)

---
BEH: sub-P038_ses-S001_task-BART_cond-nf_sham_run-001_beh.xlsx
XDF: sub-P038_ses-S001_task-Default_run-001_eeg.xdf
Rows written: (549, 250)

---
BEH: sub-P038_ses-S002_task-BART_cond-nf_sham_run-001_beh.xlsx
XDF: sub-P038_ses-S002_task-Default_r

,pID,survey_r1_sub,survey_r1_ses,survey_r1_run,survey_r1_task,survey_r1_condition,survey_r1_qid,survey_r1_section,survey_r1_question,survey_r1_response_1to5,...,pmpID,pmpL,mpmpL,SD_mpmpL,medpmpL,SD_medpmpL,theta_before_pump,theta_before_pump_Fp1,theta_before_pump_Fp2,frontal_asymmetry
0,P001,P001,S005,1,BART,nf_real,S1_Q1,Stress,"Overall, how stressful did you find the tasks ...",1,...,P001_S001_T1_P1,0.0234,0.270921,0.08398,0.237424,0.09864,4.527153e-11,2.317803e-11,3.801778e-11,0.478573
1,P001,P001,S005,1,BART,nf_real,S1_Q1,Stress,"Overall, how stressful did you find the tasks ...",1,...,P001_S001_T1_P1,0.0234,0.270921,0.08398,0.237424,0.09864,2.709053e-11,3.875172e-11,4.269305e-11,0.094536
2,P001,P001,S005,1,BART,nf_real,S1_Q1,Stress,"Overall, how stressful did you find the tasks ...",1,...,P001_S001_T1_P2,1.3350,0.270921,0.08398,0.237424,0.09864,4.678747e-11,2.006348e-11,5.439087e-11,0.966874
3,P001,P001,S005,1,BART,nf_real,S1_Q1,Stress,"Overall, how stressful did you find the tasks ...",1,...,P001_S001_T1_P2,1.3350,0.270921,0.08398,0.237424,0.09864,6.140072e-11,9.258616e-11,1.432737e-10,0.432829
4,P001,P001,S005,1,BART,nf_real,S1_Q1,Stress,"Overall, how stressful did you find the tasks ...",1,...,P001_S001_T1_P3,0.3838,0.270921,0.08398,0.237424,0.09864,2.324804e-11,1.047637e-11,1.462358e-11,0.308491
5,P001,P001,S005,1,BART,nf_real,S1_Q1,Stress,"Overall, how stressful did you find the tasks ...",1,...,P001_S001_T1_P3,0.3838,0.270921,0.08398,0.237424,0.09864,4.378090e-11,4.157908e-11,4.646828e-11,0.108699
6,P001,P001,S005,1,BART,nf_real,S1_Q1,Stress,"Overall, how stressful did you find the tasks ...",1,...,P001_S001_T1_P4,0.3838,0.270921,0.08398,0.237424,0.09864,6.958430e-11,1.502773e-10,1.646301e-10,0.090642
7,P001,P001,S005,1,BART,nf_real,S1_Q1,Stress,"Overall, how stressful did you find the tasks ...",1,...,P001_S001_T1_P4,0.3838,0.270921,0.08398,0.237424,0.09864,1.505727e-11,1.371368e-11,6.212361e-12,-0.712981
8,P001,P001,S005,1,BART,nf_real,S1_Q1,Stress,"Overall, how stressful did you find the tasks ...",1,...,P001_S001_T1_P5,0.4171,0.270921,0.08398,0.237424,0.09864,4.014219e-11,4.098364e-11,5.847307e-11,0.348244
9,P001,P001,S005,1,BART,nf_real,S1_Q1,Stress,"Overall, how stressful did you find the tasks ...",1,...,P001_S001_T1_P5,0.4171,0.270921,0.08398,0.237424,0.09864,3.838750e-11,2.838470e-11,3.432853e-11,0.184217
